# Кейс 4.2 — Детекція змін на вирівняних аерознімках (Change Detection) · v7

**Вхід:** пара геометрично вирівняних знімків «До» / «Після» (цільовий кадр 1024×1024).
**Вихід:** маска змін + **тип кожної зміни** + список об'єктів (клас, площа, рамка, впевненість) + візуалізація.

| Клас | Що означає | Звідки беремо розмітку |
|---|---|---|
| 1 · **зруйнований об'єкт** | об'єкт був «до», зруйнований/знесений «після» | xBD Global (`major-damage`, `destroyed`) + FOTBCD-Instances `DEMOLISHED` |
| 2 · **нова вирва** | свіжа вирва від вибуху | синтетика (рендер карти висот) |
| 3 · **свіжа фортифікація** | траншеї, ходи сполучення, бліндажі, протитанкові рови, «зуби дракона» | синтетика |
| 4 · нова забудова *(інша зміна)* | нова будівля — не цільовий тип, але виділяється окремо, щоб не плутати з цільовими | FOTBCD-Instances `NEW` |

**Модель:** BIT-CD (ResNet-18 + bitemporal image transformer) з **двома головами**:
- *голова змін* (є/немає зміни) — стартує з офіційних ваг BIT-CD, навчених на LEVIR-CD;
- *голова типу* — нова, **спрямована**: бачить ознаки «до» та «після» окремо (`[f₁, f₂, f₂−f₁]`), тому відрізняє знесення від будівництва та появу вирви від її зникнення.

**Протокол чесної оцінки:** офіційні спліти FOTBCD (різні департаменти Франції); xBD — просторовий спліт за блоками координат (~2 км); синтетика будується лише на основах зі *свого* спліту; повний аудит (хеші, майже-дублікати між сплітами, походження синтетики); поріг — **тільки на validation**; test — **один раз**; метрики — **окремо по кожному джерелу**, синтетика окремим рядком і ще окремо «Synthetic-Shift» (відкладені палітри/параметри генератора).

> Синтетика — це **дослідницький проксі** для вирв і фортифікацій, а не доказ роботи на реальних об'єктах. Висновки щодо реальних знімків робимо лише за FOTBCD / xBD / xBD-Ukraine.

## 0. Як запускати

**Найпростіше:** `Runtime → Change runtime type → T4 GPU`, перевірте шляхи в комірці 1.2 «Конфігурація» і `Runtime → Run all`.
Перша хвилина — **самотест GPU-частини** (розділ 2): якщо він пройшов, далі все працює без вашої участі.
Орієнтовно на T4: підготовка даних ≈ 30–45 хв при першому запуску (xBD ≈ 8,4 ГБ + синтетика; далі — кеш на Drive), навчання ≈ 40–70 хв, оцінка ≈ 5 хв. Точний темп друкується після першої епохи.

**Що покласти на Google Drive** (`MyDrive/EPS_2026/`, змінюється в 1.2):
- `FOTBCD-Instances.zip` (або розпакована тека `FOTBCD-Instances/`) з офіційного Google Drive FOTBCD. Якщо його немає — ноутбук **сам** перейде на FOTBCD-Binary (стрімінг з Hugging Face; тоді тип «знесено/нове» для FOTBCD невідомий, і на FOTBCD навчається лише голова «є/немає зміни»).
- xBD Global завантажується автоматично з Hugging Face (`DrNerd/xBD-Training-Data`). Якщо tar уже є на Drive — вкажіть `XBD_TAR_ON_DRIVE`.

**Економія GPU-квоти (необов'язково):** запустіть розділи 1–5 на CPU-рантаймі з `RUN_TRAIN=False` — підготовлені дані збережуться на Drive (`CACHE_DATA_TO_DRIVE=True`), а на GPU-рантаймі відновляться за кілька хвилин.

**Обрив сесії:** `last.pt` пишеться на Drive щоепохи; повторний `Run all` продовжить навчання з місця зупинки (усі кроки підготовки ідемпотентні). Test виконується один раз (`final_test.json`); повтор — лише з `FORCE_RETEST=True`, і це треба чесно зазначити у звіті.

**Готові ваги без повторного навчання:** `RUN_TRAIN=False`, `DOWNLOAD_SHARED_WEIGHTS=True`. Якщо в `RUN_DIR` немає `best.pt`, розділ 10 завантажить готовий checkpoint команди й відсутній `threshold.json`. Підготовка даних, аудит, validation / test усе одно потребують даних; відбиток даних має збігатися з checkpoint. Для швидкого інференсу без датасетів використовуйте `EPS_ChangeDetection_Demo_Gradio_UA.ipynb`.

**Свої знімки для демо:** розділ 13 — `USER_PRE` / `USER_POST` (і `USER_GSD`, якщо відомий).

| Розділ | Що робить | GPU |
|---|---|---|
| 1 | Налаштування: залежності, конфігурація, утиліти | — |
| 2 | Модель BIT-CD (2 голови), втрати, інференс, **самотест** | бажано |
| 3 | Дані: FOTBCD, xBD Global, xBD-Ukraine (лише test) | — |
| 4 | Синтетика: нові вирви та свіжі фортифікації | — |
| 5 | Аудит даних + галереї | — |
| 6 | Банк тренувальних тайлів | — |
| 7 | Метрики | — |
| 8 | Zero-shot baseline (LEVIR) | так |
| 9 | Навчання + графіки | так |
| 10 | Вибір порога на validation | так |
| 11 | Одноразовий TEST + візуалізації | так |
| 12 | Швидкодія 1024×1024 (вимога ≤ 2 с) | так |
| 13 | Демо: `detect_changes(pre, post)` | так |
| 14 | Підсумковий звіт і обмеження | — |

## 1. Налаштування

In [ ]:
# 1.1 Залежності. PyTorch / torchvision / OpenCV / pandas у Colab уже є — не перевстановлюємо.
# `datasets` ставиться автоматично пізніше, ЛИШЕ якщо знадобиться FOTBCD-Binary зі стрімінгом.
%pip -q install "huggingface_hub>=0.23" "shapely>=2" pycocotools

In [ ]:
# ==============================================================================
# 1.2 КОНФІГУРАЦІЯ — усі налаштування в одному місці
# ==============================================================================
import os, sys, json, time, math, random, shutil, hashlib, zipfile, tarfile, re, glob, warnings, subprocess
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
cv2.setNumThreads(0)   # без внутрішнього пулу потоків OpenCV: інакше дедлоки у fork-процесах (синтетика, DataLoader)
SEED = 42

# ---------------------------------------------------------------- що запускати
RUN_PREPARE    = True    # підготовка даних (FOTBCD, xBD, синтетика); можна на CPU
RUN_AUDIT      = True    # аудит; критичні проблеми БЛОКУЮТЬ навчання
RUN_BASELINE   = True    # zero-shot BIT-CD (LEVIR) на validation — довідкова точка відліку
RUN_TRAIN      = True    # навчання (потрібен GPU); відновлюється з last.pt
RUN_FINAL_TEST = True    # одноразовий test із порогом, обраним на validation
FORCE_RETEST   = False   # повторний test лише свідомо (зазначити у звіті!)
CACHE_DATA_TO_DRIVE = True   # зберегти підготовлені дані на Drive (zip без стиснення), якщо вистачає місця
DOWNLOAD_SHARED_WEIGHTS = True  # публічні ваги команди, якщо локальних файлів немає

# Публічні файли команди; SHA-256 фіксує перевірену версію кожного артефакту.
SHARED_FILES = {
    'best.pt': ('1jtceZDX4cNY9udOLS_6zsN8ptSNVQozj',
                'aabcc6c299fe5e2b75aa452ba9aa4397c639f3aeedc9bffc4351c97f589d938a'),
    'threshold.json': ('1RAjId11DcGvu1hq0K4JN4hJhDT32R1Bs',
                       'c98e9f2610c6afb6c7653222520809a16881c87249b0a605e46bdd0e003dfe64'),
    'BIT_LEVIR_best_ckpt.pt': ('1Qoh-0_Ub3xOdwRphfT_I4lXksm9nEZMK',
                             'c159ba76143447f58c9f367ce8126a0014f2e4ba218cdb97cca173952c38cb3b'),
}


# ---------------------------------------------------------------- шляхи
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/EPS_2026')
    WORK = Path('/content/eps_cd_v7')
else:
    DRIVE_ROOT = Path(os.environ.get('EPS_DRIVE_ROOT', './EPS_2026'))
    WORK = Path(os.environ.get('EPS_WORK', './eps_cd_v7_work'))
RUN_NAME = 'bitcd_semantic_v7'
DATA = WORK / 'data'                       # підготовлені дані: PNG + manifest.csv
RAW = WORK / 'raw'                         # сирі архіви/розпаковка
RUN_DIR = DRIVE_ROOT / 'runs' / RUN_NAME   # checkpoint-и, історія, звіти — на Drive
DATA_CACHE_ZIP = DRIVE_ROOT / 'cache' / 'eps_cd_v7_data.zip'
for d in (DATA, RAW, RUN_DIR, RUN_DIR / 'figures'):
    d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------- FOTBCD
FOTBCD_MODE = 'auto'          # 'instances' | 'binary' | 'auto' (= instances, якщо архів знайдено)
FOT_INSTANCES_CANDIDATES = [DRIVE_ROOT / 'FOTBCD-Instances.zip', DRIVE_ROOT / 'FOTBCD-Instances',
                            Path('/content/FOTBCD-Instances'), RAW / 'FOTBCD-Instances']
FOT_INST_LIMITS = {'train': None, 'val': 400, 'test': 800}       # None = усі пари спліту
FOT_BIN_HF_ID = 'retgenai/FOTBCD-Binary'
FOT_BIN_REVISION = 'e87438d8f8eaba3ace6dee7150716302dec51b1d'      # зафіксована ревізія
FOT_BIN_LIMITS = {'train': 2000, 'val': 300, 'test': 600}
FOT_GSD = 0.20                                                      # м/піксель (BD ORTHO)

# ---------------------------------------------------------------- xBD Global
XBD_DIR_CANDIDATES = [RAW / 'xbd_global', DRIVE_ROOT / 'xbd_global', Path('/content/change_detection_data/xbd_global')]
XBD_TAR_ON_DRIVE = DRIVE_ROOT / 'train_images_labels_targets.tar'  # якщо tar уже є на Drive
XBD_HF_REPO, XBD_HF_FILE = 'DrNerd/xBD-Training-Data', 'train_images_labels_targets.tar'
XBD_ALLOW_HF_DOWNLOAD = True      # ~8.4 ГБ; витягуємо лише images + labels
XBD_DELETE_TAR_AFTER = True       # звільнити диск після розпаковки
XBD_SPLIT_MODE = 'spatial'        # 'spatial' — блоки координат усередині кожної події; 'event' — цілі події
XBD_BLOCK_DEG = 0.02              # ~2 км
XBD_SPLIT_RATIOS = {'train': 0.70, 'val': 0.15, 'test': 0.15}
XBD_EVENT_SPLIT = {'val': ['hurricane-matthew'], 'test': ['santa-rosa-wildfire', 'palu-tsunami']}  # лише для 'event'
XBD_POS_LIMITS = {'train': 650, 'val': 90, 'test': 200}           # пари з руйнуваннями
XBD_NEG_FRACTION = 0.25           # + частка пар без руйнувань (є будівлі, немає major/destroyed)
XBD_MIN_POS_PX = 400              # мінімум пікселів «зруйновано» для позитивної пари
XBD_MINOR_AS = 'ignore'           # 'ignore' | 'background' — як трактувати minor-damage / un-classified
XBD_DEFAULT_GSD = 0.5

# ---------------------------------------------------------------- xBD-Ukraine (ЛИШЕ оцінка, OOD)
USE_XBD_UKRAINE_EVAL = True
XBD_UKR_HF_REPO = 'sda-kr/xbd-ukraine'
XBD_UKR_GSD = 0.3                 # у JSON немає GSD — припущення (впливає лише на площі в м²)

# ---------------------------------------------------------------- синтетика
SYN_COUNTS = {'train': 2000, 'val': 300, 'test': 400, 'test_shift': 400}
SYN_XBD_FRACTION = 0.20           # частка синтетики на xBD-основах (супутниковий домен, кадри 1024)
SYN_P_NEG_BASE = 0.7              # частка основ без реальних змін (решта — змішані сцени)
SYN_WORKERS = 2                   # процеси генерації (у Colab 2 CPU) — ≈12–15 хв на весь набір, далі кеш

# ---------------------------------------------------------------- класи
LBL_BG, LBL_DESTROYED, LBL_CRATER, LBL_FORT, LBL_NEWBLD = 0, 1, 2, 3, 4
LBL_UNKNOWN, LBL_IGNORE = 254, 255            # зміна невідомого типу (FOTBCD-Binary) / ігнор
CLASS_NAMES = {1: "зруйнований об'єкт", 2: 'нова вирва', 3: 'свіжа фортифікація', 4: 'нова забудова (інша зміна)'}
CLASS_SHORT = {1: 'destroyed', 2: 'crater', 3: 'fortification', 4: 'new_building'}
CLASS_COLORS = {1: (230, 25, 75), 2: (255, 140, 0), 3: (255, 225, 25), 4: (0, 130, 200)}
N_TYPES = 4                                   # вихід голови типу: класи 1..4 → індекси 0..3
TARGET_CLASSES = [1, 2, 3]

# ---------------------------------------------------------------- навчання
CROP = 256
BATCH = 16
EPOCHS = 30
ITERS_PER_EPOCH = 500
LR = 2e-4
LR_TYPE_HEAD_MULT = 5.0
WEIGHT_DECAY = 1e-4
WARMUP_ITERS = 400
SOURCE_WEIGHTS = {'FOTBCD': 0.35, 'xBD': 0.35, 'SYN': 0.30}   # частки джерел у кожному батчі
P_POS_CROP = 0.6                  # імовірність кропу з центром на зміні
CHANGE_POS_WEIGHT = 2.0
LAMBDA_TYPE = 1.0
EARLY_STOP_PATIENCE = 10
AMP = True
NUM_WORKERS = 2
BANK_CHIP = 512                   # розмір фрагмента в банку (memmap)

# ---------------------------------------------------------------- інференс / оцінка
TILE = 256
STRIDE = 192                      # перекриття вікон для фінальної оцінки та демо
VAL_STRIDE_DURING_TRAIN = 256     # швидша валідація під час навчання
VAL_MAX_PER_SUBSET_DURING_TRAIN = 200   # підвибірка val для вибору епохи (поріг — на ПОВНОМУ val)
INFER_BATCH = 32
TTA = False
MIN_OBJ_AREA_M2 = {1: 6.0, 2: 0.4, 3: 0.6, 4: 6.0}   # мінімальна площа об'єкта у виводі
OBJ_COVER = 0.3                   # об'єкт «знайдено», якщо ≥30% його пікселів покриті прогнозом того ж класу
TARGET_F1, TARGET_LATENCY_S = 0.75, 2.0

random.seed(SEED); np.random.seed(SEED)
print('Colab:', IN_COLAB, '| WORK:', WORK, '| RUN_DIR:', RUN_DIR)

In [ ]:
# ==============================================================================
# 1.3 Утиліти: читання/запис, оверлеї, хеші, кеш на Drive
# ==============================================================================
PNG_FAST = [cv2.IMWRITE_PNG_COMPRESSION, 1]


def read_rgb(path):
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(f'Не вдалося прочитати зображення: {path}')
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def write_rgb(path, rgb):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    if not cv2.imwrite(str(path), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), PNG_FAST):
        raise IOError(f'Не вдалося записати {path}')


def read_gray(path):
    m = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if m is None:
        raise FileNotFoundError(f'Не вдалося прочитати маску: {path}')
    if m.ndim == 3:
        m = m[..., 0]
    return m


def write_gray(path, m):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    if not cv2.imwrite(str(path), np.ascontiguousarray(m.astype(np.uint8)), PNG_FAST):
        raise IOError(f'Не вдалося записати {path}')


def file_sha1(path, chunk=1 << 20):
    h = hashlib.sha1()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def dhash64(img_rgb):
    """Перцептивний хеш (64 біти) для пошуку майже-дублікатів між сплітами."""
    g = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    s = cv2.resize(g, (9, 8), interpolation=cv2.INTER_AREA).astype(np.int16)
    bits = (s[:, 1:] > s[:, :-1]).ravel()
    return np.packbits(bits).view(np.uint64)[0]


def window_origins(length, win, stride):
    if length <= win:
        return [0]
    pos = list(range(0, length - win + 1, stride))
    if pos[-1] + win < length:
        pos.append(length - win)
    return pos


def label_overlay(rgb, label, alpha=0.5, show_unknown=True):
    out = rgb.astype(np.float32).copy()
    for c, col in CLASS_COLORS.items():
        m = label == c
        out[m] = out[m] * (1 - alpha) + np.array(col, np.float32) * alpha
    if show_unknown:
        m = label == LBL_UNKNOWN
        out[m] = out[m] * (1 - alpha) + np.array((255, 0, 255), np.float32) * alpha
    m = label == LBL_IGNORE
    out[m] = out[m] * 0.6 + 128 * 0.4
    return out.clip(0, 255).astype(np.uint8)


def legend_handles(include_unknown=False, include_ignore=False):
    from matplotlib.patches import Patch
    hs = [Patch(color=np.array(CLASS_COLORS[c]) / 255, label=f'{c}: {CLASS_NAMES[c]}') for c in CLASS_COLORS]
    if include_unknown:
        hs.append(Patch(color=(1, 0, 1), label='зміна невідомого типу (Binary)'))
    if include_ignore:
        hs.append(Patch(color=(0.5, 0.5, 0.5), label='ігнор (minor-damage тощо)'))
    return hs


def save_fig(fig, name):
    p = RUN_DIR / 'figures' / f'{name}.png'
    fig.savefig(p, dpi=110, bbox_inches='tight')
    return p


def free_gb(path):
    try:
        return shutil.disk_usage(path).free / 1e9
    except Exception:
        return float('nan')


def zip_folder_stored(folder, zip_path, exclude_suffixes=()):
    """Zip без стиснення (PNG і так стиснуті) — швидко; атомарний запис."""
    folder, zip_path = Path(folder), Path(zip_path)
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = zip_path.with_suffix('.partial')
    files = [p for p in folder.rglob('*') if p.is_file() and not p.name.endswith(tuple(exclude_suffixes))]
    with zipfile.ZipFile(tmp, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as z:
        for p in tqdm(files, desc=f'zip → {zip_path.name}', leave=False):
            z.write(p, p.relative_to(folder))
    os.replace(tmp, zip_path)
    return zip_path


def safe_unzip(zip_path, dest):
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        for m in z.infolist():
            p = Path(m.filename)
            if p.is_absolute() or '..' in p.parts:
                raise ValueError(f'Небезпечний шлях у zip: {m.filename}')
        z.extractall(dest)


NUM_COLS = ['gsd', 'h', 'w', 'sun_az', 'sun_el', 'chg_px', 'n_destroyed_px', 'n_new_px']
TXT_COLS = ['uid', 'source', 'subset', 'split', 'pre', 'post', 'label', 'aux', 'fp', 'group', 'base_uid', 'event']


def normalize_manifest(df):
    """Єдині типи колонок (після CSV порожні рядки/NaN поводяться однаково)."""
    df = df.copy()
    for c in NUM_COLS:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    for c in TXT_COLS:
        if c in df:
            df[c] = df[c].fillna('').astype(str)
    return df


class Timer:
    def __init__(self, name): self.name = name
    def __enter__(self): self.t = time.perf_counter(); return self
    def __exit__(self, *a): print(f'⏱ {self.name}: {time.perf_counter() - self.t:.1f} с')


print('Утиліти готові. Вільно на локальному диску:', round(free_gb(WORK), 1), 'ГБ')

## 2. Модель BIT-CD з двома головами (+ самотест GPU за ≈30 с)

Модель стоїть **до** підготовки даних навмисно: самотест одразу перевіряє всю GPU-частину (ваги, прямий/зворотний прохід з AMP, функцію втрат, інференс 512/1024), щоб не чекати 30 хв підготовки даних і тільки тоді дізнатися про проблему середовища.

**BIT-CD** (Chen et al., 2021): спільний ResNet-18 (до layer3, stride 8) для обох дат → по 4 семантичні токени на знімок → transformer-encoder над 8 токенами → transformer-decoder (8 шарів) повертає токени у просторові карти → різниця `|f₁ − f₂|` → класифікатор. Конфігурація = офіційна `base_transformer_pos_s4_dd8_dedim8`; ваги LEVIR-CD завантажуються з перевіркою, що бракує **лише** ваг нової голови.

**Друга голова (тип зміни)** працює на `[f₁, f₂, f₂ − f₁]` (96 каналів, 1/4 роздільності): на відміну від симетричної `|f₁ − f₂|`, вона знає *напрям* зміни — без цього знесення та нове будівництво нерозрізненні.

**Функція втрат:** зміна — `CE (вага позитиву 2) + Dice`; тип — `CE` з вагами класів лише на пікселях зміни відомого типу (`254` — тип невідомий — та `255` ігноруються).

In [ ]:
# ==============================================================================
# 2.1 BIT-CD з двома головами + завантаження офіційних ваг LEVIR-CD
# ==============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
print('PyTorch', torch.__version__, '| torchvision', torchvision.__version__, '| пристрій:', DEVICE,
      '|', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU')


# ---- блоки трансформера (імена параметрів = офіційний BIT_CD, щоб ваги завантажились строго)
class Residual(nn.Module):
    def __init__(self, fn):
        super().__init__(); self.fn = fn

    def forward(self, x, **kw):
        return self.fn(x, **kw) + x


class Residual2(nn.Module):
    def __init__(self, fn):
        super().__init__(); self.fn = fn

    def forward(self, x, x2, **kw):
        return self.fn(x, x2, **kw) + x


class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__(); self.norm = nn.LayerNorm(dim); self.fn = fn

    def forward(self, x, **kw):
        return self.fn(self.norm(x), **kw)


class PreNorm2(nn.Module):
    def __init__(self, dim, fn):
        super().__init__(); self.norm = nn.LayerNorm(dim); self.fn = fn

    def forward(self, x, x2, **kw):
        return self.fn(self.norm(x), self.norm(x2), **kw)


class FeedForward(nn.Module):
    def __init__(self, dim, hidden, dropout=0.):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, dim), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


def _split_heads(t, h):            # 'b n (h d) -> b h n d'
    b, n, _ = t.shape
    return t.reshape(b, n, h, -1).transpose(1, 2)


def _merge_heads(t):               # 'b h n d -> b n (h d)'
    b, h, n, d = t.shape
    return t.transpose(1, 2).reshape(b, n, h * d)


class Attention(nn.Module):
    def __init__(self, dim, heads=8, dim_head=64, dropout=0.):
        super().__init__()
        inner = dim_head * heads
        self.heads, self.scale = heads, dim ** -0.5          # як в оригіналі BIT: масштаб від dim
        self.to_qkv = nn.Linear(dim, inner * 3, bias=False)
        self.to_out = nn.Sequential(nn.Linear(inner, dim), nn.Dropout(dropout))

    def forward(self, x, mask=None):
        q, k, v = (_split_heads(t, self.heads) for t in self.to_qkv(x).chunk(3, dim=-1))
        attn = (torch.einsum('bhid,bhjd->bhij', q, k) * self.scale).softmax(dim=-1)
        return self.to_out(_merge_heads(torch.einsum('bhij,bhjd->bhid', attn, v)))


class CrossAttention(nn.Module):
    def __init__(self, dim, heads=8, dim_head=64, dropout=0., softmax=True):
        super().__init__()
        inner = dim_head * heads
        self.heads, self.scale, self.softmax = heads, dim ** -0.5, softmax
        self.to_q = nn.Linear(dim, inner, bias=False)
        self.to_k = nn.Linear(dim, inner, bias=False)
        self.to_v = nn.Linear(dim, inner, bias=False)
        self.to_out = nn.Sequential(nn.Linear(inner, dim), nn.Dropout(dropout))

    def forward(self, x, m, mask=None):
        q, k, v = _split_heads(self.to_q(x), self.heads), _split_heads(self.to_k(m), self.heads), _split_heads(self.to_v(m), self.heads)
        dots = torch.einsum('bhid,bhjd->bhij', q, k) * self.scale
        attn = dots.softmax(dim=-1) if self.softmax else dots
        return self.to_out(_merge_heads(torch.einsum('bhij,bhjd->bhid', attn, v)))


class Transformer(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout=0.):
        super().__init__()
        self.layers = nn.ModuleList([nn.ModuleList([
            Residual(PreNorm(dim, Attention(dim, heads, dim_head, dropout))),
            Residual(PreNorm(dim, FeedForward(dim, mlp_dim, dropout)))]) for _ in range(depth)])

    def forward(self, x):
        for attn, ff in self.layers:
            x = ff(attn(x))
        return x


class TransformerDecoder(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout=0., softmax=True):
        super().__init__()
        self.layers = nn.ModuleList([nn.ModuleList([
            Residual2(PreNorm2(dim, CrossAttention(dim, heads, dim_head, dropout, softmax))),
            Residual(PreNorm(dim, FeedForward(dim, mlp_dim, dropout)))]) for _ in range(depth)])

    def forward(self, x, m):
        for attn, ff in self.layers:
            x = ff(attn(x, m))
        return x


def two_layer_conv(cin, cout, k=3):
    return nn.Sequential(nn.Conv2d(cin, cin, k, padding=k // 2, bias=False), nn.BatchNorm2d(cin), nn.ReLU(),
                         nn.Conv2d(cin, cout, k, padding=k // 2))


class BITSemantic(nn.Module):
    """BIT-CD base_transformer_pos_s4_dd8_dedim8 + голова типу зміни."""

    def __init__(self, n_types=N_TYPES, token_len=4, dec_depth=8, dec_dim_head=8):
        super().__init__()
        self.resnet = torchvision.models.resnet18(weights=None)
        # офіційний BIT: replace_stride_with_dilation=[False, True, True]; у BasicBlock дилатація ігнорується,
        # тож еквівалент — прибрати stride у layer3/layer4
        for layer in (self.resnet.layer3, self.resnet.layer4):
            layer[0].conv1.stride = (1, 1)
            layer[0].downsample[0].stride = (1, 1)
        self.upsamplex2 = nn.Upsample(scale_factor=2)
        self.upsamplex4 = nn.Upsample(scale_factor=4, mode='bilinear')
        self.conv_pred = nn.Conv2d(256, 32, kernel_size=3, padding=1)
        self.classifier = two_layer_conv(32, 2)                    # голова «є/немає зміни» (LEVIR)
        self.token_len = token_len
        self.conv_a = nn.Conv2d(32, token_len, kernel_size=1, padding=0, bias=False)
        self.pos_embedding = nn.Parameter(torch.randn(1, token_len * 2, 32))
        self.transformer = Transformer(dim=32, depth=1, heads=8, dim_head=64, mlp_dim=64)
        self.transformer_decoder = TransformerDecoder(dim=32, depth=dec_depth, heads=8, dim_head=dec_dim_head, mlp_dim=64)
        self.type_head = nn.Sequential(                            # нова спрямована голова типу
            nn.Conv2d(96, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, n_types, 1))

    def forward_single(self, x):
        r = self.resnet
        x = r.maxpool(r.relu(r.bn1(r.conv1(x))))
        x = r.layer3(r.layer2(r.layer1(x)))                      # 1/8, 256 каналів
        return self.conv_pred(self.upsamplex2(x))                  # 1/4, 32 канали

    def tokens(self, x):
        b, c, h, w = x.shape
        att = torch.softmax(self.conv_a(x).view(b, self.token_len, -1), dim=-1)
        return torch.einsum('bln,bcn->blc', att, x.view(b, c, -1))

    def decode(self, x, m):
        b, c, h, w = x.shape
        y = self.transformer_decoder(x.flatten(2).transpose(1, 2), m)
        return y.transpose(1, 2).reshape(b, c, h, w)

    def forward(self, x1, x2):
        f1, f2 = self.forward_single(x1), self.forward_single(x2)
        t = self.transformer(torch.cat([self.tokens(f1), self.tokens(f2)], dim=1) + self.pos_embedding)
        t1, t2 = t.chunk(2, dim=1)
        f1, f2 = self.decode(f1, t1), self.decode(f2, t2)
        change = self.classifier(self.upsamplex4(torch.abs(f1 - f2)))
        typ = self.type_head(torch.cat([f1, f2, f2 - f1], dim=1))
        typ = F.interpolate(typ, size=change.shape[-2:], mode='bilinear', align_corners=False)
        return change, typ


def _download_shared_file(target, file_id, expected_sha256):
    """Кешований публічний файл Drive: SHA-256 перевіряється до заміни цільового файлу."""
    import hashlib

    target = Path(target)

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    if target.is_file() and digest(target) == expected_sha256:
        print('Перевірений кеш:', target.name)
        return target

    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'gdown>=5.2,<6'], check=True)
        import gdown

    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_name(target.name + '.download')
    try:
        print('Завантаження:', target.name)
        result = gdown.download(id=file_id, output=str(temporary), quiet=False, use_cookies=False)
        if result is None or not temporary.is_file():
            raise RuntimeError('Google Drive не повернув файл.')
        if digest(temporary) != expected_sha256:
            raise ValueError('SHA-256 не збігається з перевіреною версією файлу.')
        os.replace(temporary, target)
    except Exception as exc:
        raise RuntimeError(
            f'Не вдалося отримати {target.name}: {exc} '
            'Перевірте доступ за посиланням, ліміт завантажень Google Drive '
            'або використайте власні файли за інструкцією README.'
        ) from exc
    finally:
        temporary.unlink(missing_ok=True)
    return target


def get_bit_levir_weights():
    cached = DRIVE_ROOT / 'weights' / 'BIT_LEVIR_best_ckpt.pt'
    if cached.exists() and cached.stat().st_size > 5e6:
        return cached
    if DOWNLOAD_SHARED_WEIGHTS:
        return _download_shared_file(cached, *SHARED_FILES['BIT_LEVIR_best_ckpt.pt'])
    repo = WORK / 'BIT_CD'
    if not (repo / 'checkpoints' / 'BIT_LEVIR' / 'best_ckpt.pt').exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/justchenhao/BIT_CD.git', str(repo)], check=True)
    src = repo / 'checkpoints' / 'BIT_LEVIR' / 'best_ckpt.pt'
    if src.stat().st_size < 5e6:
        raise RuntimeError('BIT checkpoint схожий на Git-LFS pointer, а не на ваги')
    cached.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, cached)
    return cached


def build_model(pretrained=True):
    model = BITSemantic()
    if pretrained:
        # офіційний чекпойнт BIT_CD (довірене джерело; legacy-формат → weights_only=False)
        ck = torch.load(get_bit_levir_weights(), map_location='cpu', weights_only=False)
        sd = ck.get('model_G_state_dict', ck)
        sd = {k.replace('module.', '').replace('net_G.', ''): v for k, v in sd.items()}
        missing, unexpected = model.load_state_dict(sd, strict=False)
        assert not unexpected, f'Зайві ключі: {unexpected[:5]}'
        assert all(k.startswith('type_head.') for k in missing), f'Бракує ключів: {[k for k in missing if not k.startswith("type_head.")][:5]}'
        print(f'LEVIR-ваги BIT-CD завантажено: {len(sd)} тензорів; нові (голова типу): {len(missing)}')
    for p in list(model.resnet.layer4.parameters()) + list(model.resnet.fc.parameters()):
        p.requires_grad_(False)                                    # не використовуються (resnet_stages_num=4)
    return model.to(DEVICE)


print('BIT-CD визначено.')

In [ ]:
# ==============================================================================
# 2.2 Функція втрат і підготовка входу
# ==============================================================================
def to_input(x_u8):
    """uint8 B×3×H×W → нормалізація BIT (mean=std=0.5)."""
    return x_u8.to(DEVICE, non_blocking=True).float().div_(255.0).sub_(0.5).div_(0.5)


def type_class_weights(meta):
    """Ваги класів голови типу ~ (медіана / частота)^0.5 за кількістю тайлів з класом."""
    pres = meta['pres'][:, :N_TYPES].sum(0).astype(float)
    w = np.ones(N_TYPES)
    ok = pres > 0
    if ok.any():
        w[ok] = np.clip((np.median(pres[ok]) / pres[ok]) ** 0.5, 0.5, 4.0)
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)


def compute_loss(change_logits, type_logits, lab, type_w):
    """Без синхронізацій GPU: зважені середні рахуємо як сума / сума ваг."""
    lab = lab.to(DEVICE, non_blocking=True).long()
    valid = lab != LBL_IGNORE
    chg = ((lab >= 1) & (lab <= 4)) | (lab == LBL_UNKNOWN)
    tgt = chg.long().masked_fill(~valid, 255)
    cl = change_logits.float()
    cw = torch.tensor([1.0, CHANGE_POS_WEIGHT], device=DEVICE)
    ce_sum = F.cross_entropy(cl, tgt, weight=cw, ignore_index=255, reduction='sum')
    ce = ce_sum / (cw[chg.long()] * valid).sum().clamp(min=1e-6)
    p1 = torch.softmax(cl, dim=1)[:, 1]
    g, vm = chg.float(), valid.float()
    dice = 1 - (2 * (p1 * g * vm).sum() + 1) / ((p1 * vm).sum() + (g * vm).sum() + 1)
    typed = (lab >= 1) & (lab <= 4)
    tt = (lab - 1).masked_fill(~typed, 255)
    tce_sum = F.cross_entropy(type_logits.float(), tt, weight=type_w, ignore_index=255, reduction='sum')
    tce = tce_sum / (type_w[tt.clamp(max=N_TYPES - 1)] * typed).sum().clamp(min=1e-6)
    total = ce + dice + LAMBDA_TYPE * tce
    return total, {'ce': ce.detach(), 'dice': dice.detach(), 'type_ce': tce.detach()}   # без синхронізації GPU


print('Функцію втрат визначено.')

In [ ]:
# ==============================================================================
# 2.3 Інференс кадру довільного розміру: ковзне вікно + зважене злиття (+ TTA за бажанням)
# ==============================================================================
_WIN_CACHE = {}


def _blend_window(tile):
    if tile not in _WIN_CACHE:
        h = 0.1 + 0.9 * torch.sin(math.pi * (torch.arange(tile, dtype=torch.float32) + 0.5) / tile)
        _WIN_CACHE[tile] = (h[:, None] * h[None, :]).to(DEVICE)
    return _WIN_CACHE[tile]


def _forward_probs(model, a, b, tta):
    """Ймовірності зміни (B×H×W) і типів (B×T×H×W) з опційним TTA (віддзеркалення)."""
    flips = [[]] + ([[3], [2], [2, 3]] if tta else [])
    pc_sum, pt_sum = 0, 0
    for f in flips:
        aa, bb = (a.flip(f), b.flip(f)) if f else (a, b)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(AMP and DEVICE.type == 'cuda')):
            ch, ty = model(aa, bb)
        pc, pt = torch.softmax(ch.float(), 1)[:, 1], torch.softmax(ty.float(), 1)
        if f:
            pc, pt = pc.flip([d - 1 for d in f]), pt.flip(f)
        pc_sum, pt_sum = pc_sum + pc, pt_sum + pt
    return pc_sum / len(flips), pt_sum / len(flips)


@torch.inference_mode()
def predict_full(model, pre, post, stride=STRIDE, tta=TTA, batch=INFER_BATCH, return_type_prob=False):
    """pre/post: H×W×3 uint8 (numpy) → prob [H,W] float32, typ [H,W] uint8 (індекс 0..3 ⇒ клас 1..4)."""
    model.eval()
    H, W = pre.shape[:2]
    ph, pw = max(0, TILE - H), max(0, TILE - W)
    if ph or pw:
        pre = cv2.copyMakeBorder(pre, 0, ph, 0, pw, cv2.BORDER_REFLECT_101)
        post = cv2.copyMakeBorder(post, 0, ph, 0, pw, cv2.BORDER_REFLECT_101)
    Hp, Wp = pre.shape[:2]
    a = torch.from_numpy(np.ascontiguousarray(pre)).to(DEVICE).permute(2, 0, 1).float().div_(255).sub_(0.5).div_(0.5)
    b = torch.from_numpy(np.ascontiguousarray(post)).to(DEVICE).permute(2, 0, 1).float().div_(255).sub_(0.5).div_(0.5)
    coords = [(y, x) for y in window_origins(Hp, TILE, stride) for x in window_origins(Wp, TILE, stride)]
    win = _blend_window(TILE)
    prob = torch.zeros(Hp, Wp, device=DEVICE)
    tsum = torch.zeros(N_TYPES, Hp, Wp, device=DEVICE)
    wsum = torch.zeros(Hp, Wp, device=DEVICE)
    for i in range(0, len(coords), batch):
        cs = coords[i:i + batch]
        pa = torch.stack([a[:, y:y + TILE, x:x + TILE] for y, x in cs])
        pb = torch.stack([b[:, y:y + TILE, x:x + TILE] for y, x in cs])
        pc, pt = _forward_probs(model, pa, pb, tta)
        for j, (y, x) in enumerate(cs):
            prob[y:y + TILE, x:x + TILE] += pc[j] * win
            tsum[:, y:y + TILE, x:x + TILE] += pt[j] * win
            wsum[y:y + TILE, x:x + TILE] += win
    prob = (prob / wsum)[:H, :W]
    tprob = (tsum / wsum)[:, :H, :W]
    typ = tprob.argmax(0).to(torch.uint8)
    out = (prob.float().cpu().numpy(), typ.cpu().numpy())
    if return_type_prob:
        out = out + (tprob.max(0).values.float().cpu().numpy(),)
    return out


def cuda_sync():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()


def save_checkpoint(obj, path):
    tmp = Path(str(path) + '.tmp')
    torch.save(obj, tmp)
    os.replace(tmp, path)


def load_trained_model(path):
    ck = torch.load(path, map_location='cpu', weights_only=False)
    m = build_model(pretrained=False)
    m.load_state_dict(ck['model'], strict=True)
    m.eval()
    return m, ck


print('Інференс визначено.')

In [ ]:
# ==============================================================================
# 2.4 Самотест GPU-частини (≈30 с): ваги LEVIR, крок навчання з AMP, втрати на всіх типах міток, інференс
# ==============================================================================
def gpu_selftest():
    t0 = time.perf_counter()
    m = build_model(pretrained=True)
    n_par = sum(p.numel() for p in m.parameters() if p.requires_grad)
    m.train()
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=1e-5)
    use_amp = AMP and DEVICE.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp) if DEVICE.type == 'cuda' else None
    g = torch.Generator().manual_seed(SEED)
    pre = torch.randint(0, 256, (4, 3, CROP, CROP), dtype=torch.uint8, generator=g)
    post = torch.randint(0, 256, (4, 3, CROP, CROP), dtype=torch.uint8, generator=g)
    lab = torch.zeros(4, CROP, CROP, dtype=torch.uint8)
    lab[0, 40:90, 40:90] = LBL_DESTROYED; lab[0, 120:130, :] = LBL_FORT
    lab[1, 60:80, 60:80] = LBL_CRATER; lab[1, :, 200:210] = LBL_NEWBLD
    lab[2, 100:150, 100:150] = LBL_UNKNOWN; lab[2, :8, :8] = LBL_IGNORE      # 3-й зразок: лише «невідомий тип»
    tw = torch.ones(N_TYPES, device=DEVICE)
    losses = []
    for _ in range(3):
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
            ch, ty = m(to_input(pre), to_input(post))
        assert ch.shape == (4, 2, CROP, CROP) and ty.shape == (4, N_TYPES, CROP, CROP), (ch.shape, ty.shape)
        loss, parts = compute_loss(ch, ty, lab, tw)
        opt.zero_grad(set_to_none=True)
        if scaler is not None:
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in m.parameters() if p.requires_grad], 5.0)
            scaler.step(opt); scaler.update()
        else:
            loss.backward(); opt.step()
        losses.append(float(loss))
        assert np.isfinite(losses[-1]), 'loss = NaN/Inf'
    # батч, де немає жодного пікселя зміни відомого типу — втрата має лишатися скінченною
    l2, _ = compute_loss(ch.detach(), ty.detach(), torch.zeros_like(lab), tw)
    assert torch.isfinite(l2), 'втрата без змін — NaN'
    rng = np.random.default_rng(SEED)
    sizes = ((512, 512), (1024, 1024), (300, 700)) if DEVICE.type == 'cuda' else ((300, 700),)   # на CPU — коротко
    for (h, w) in sizes:
        a = rng.integers(0, 255, (h, w, 3), dtype=np.uint8); b = rng.integers(0, 255, (h, w, 3), dtype=np.uint8)
        prob, typ = predict_full(m, a, b)
        assert prob.shape == (h, w) and typ.shape == (h, w) and np.isfinite(prob).all() and typ.max() < N_TYPES
    prob_t, typ_t = predict_full(m, a, b, tta=True)
    assert prob_t.shape == prob.shape
    msg = ''
    if DEVICE.type == 'cuda':
        a = rng.integers(0, 255, (1024, 1024, 3), dtype=np.uint8)
        predict_full(m, a, a); cuda_sync()
        t1 = time.perf_counter(); predict_full(m, a, a); cuda_sync()
        msg = f' | інференс 1024×1024: {time.perf_counter() - t1:.3f} с'
    print(f'✅ Самотест пройдено за {time.perf_counter() - t0:.1f} с | параметрів, що навчаються: {n_par / 1e6:.2f} M | '
          f'втрати: {[round(x, 3) for x in losses]}{msg} | {DEVICE}')
    if DEVICE.type != 'cuda':
        print('⚠ GPU не підключено: підготовка даних працюватиме, але навчання — ні (Runtime → T4 GPU).')
    del m, opt
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()


gpu_selftest()

## 3. Дані

**Єдине кодування розмітки** (одна PNG-маска на пару): `0` — без змін · `1` — зруйнований об'єкт · `2` — нова вирва · `3` — свіжа фортифікація · `4` — нова забудова · `254` — зміна невідомого типу (FOTBCD-Binary) · `255` — ігнор (неоднозначні пікселі, напр. xBD `minor-damage`).

Кожне джерело готується ідемпотентно: якщо його `*_manifest.csv` уже існує з тією ж конфігурацією — крок пропускається. Якщо на Drive є кеш підготовлених даних — він відновлюється першим.

**Що виправлено відносно v3** (за аудитом попередніх ноутбуків):
- FOTBCD: у v3 `image_before` використовувався двічі (pre = post) — тут беремо `image_after`/`after` і **зупиняємось**, якщо pre і post ідентичні при позитивній масці;
- FOTBCD-train у v3 був порожній (0 кадрів) — тут використовується повний офіційний train;
- xBD: у v3 масштаб рахувався з поля `gsd` (мультиспектральний GSD) — тут для розмірів об'єктів синтетики беремо `pan_resolution`, а знімки **не перемасштабовуються**;
- xBD Global у v3 був лише оцінкою на 30 парах — тут він повноцінне джерело train/val/test з просторовим сплітом;
- синтетика без мозаїк 2×2 (див. розділ 4).

In [ ]:
# ==============================================================================
# 3.1 Кеш на Drive (якщо є) + FOTBCD (Instances → типи змін; Binary → лише «є зміна»)
# ==============================================================================
if RUN_PREPARE and not (DATA / 'manifest.csv').exists() and DATA_CACHE_ZIP.exists():
    with Timer('Відновлення кешу даних з Drive'):
        safe_unzip(DATA_CACHE_ZIP, DATA)


def part_ready(name, cfg):
    """Частину даних можна пропустити, якщо manifest + config збігаються і файли на місці."""
    man, conf = DATA / f'{name}_manifest.csv', DATA / f'{name}_config.json'
    if not (man.exists() and conf.exists()):
        return None
    if json.loads(conf.read_text()) != json.loads(json.dumps(cfg, default=str)):
        print(f'[{name}] конфігурація змінилась → перебудова')
        return None
    try:
        df = normalize_manifest(pd.read_csv(man))
    except Exception:                                  # порожній/пошкоджений manifest з невдалого запуску
        print(f'[{name}] manifest порожній або пошкоджений → перебудова')
        return None
    if not len(df) or 'split' not in df:
        print(f'[{name}] manifest порожній → перебудова')
        return None
    cols = [c for c in ('pre', 'post', 'label') if c in df]
    missing = sum(not Path(p).exists() for c in cols for p in df[c].head(50))
    if missing:
        print(f'[{name}] {missing} файлів з перших 50 записів відсутні → перебудова')
        return None
    return df


def part_save(name, cfg, df):
    df = normalize_manifest(df)
    df.to_csv(DATA / f'{name}_manifest.csv', index=False)
    (DATA / f'{name}_config.json').write_text(json.dumps(cfg, default=str, indent=1))
    return df


def find_instances_root(d):
    for j in sorted(Path(d).rglob('train.json')):
        if j.parent.name == 'annotations' and (j.parent.parent / 'images').is_dir():
            return j.parent.parent
    return None


def locate_fotbcd_instances():
    for c in FOT_INSTANCES_CANDIDATES:
        c = Path(c)
        if c.is_file() and c.suffix.lower() == '.zip':
            dest = RAW / 'FOTBCD-Instances'
            root = find_instances_root(dest) if dest.exists() else None
            if root is None:
                with Timer(f'Розпаковка {c.name}'):
                    safe_unzip(c, dest)
                root = find_instances_root(dest)
            if root is not None:
                return root
        elif c.is_dir():
            root = find_instances_root(c)
            if root is not None:
                return root
    return None


def _coco_polys_to_mask(segm, h, w):
    m = np.zeros((h, w), np.uint8)
    if isinstance(segm, list):
        for poly in segm:
            pts = np.asarray(poly, np.float64).reshape(-1, 2)
            if len(pts) >= 3:
                cv2.fillPoly(m, [np.round(pts).astype(np.int32)], 1)
    elif isinstance(segm, dict):                      # RLE
        from pycocotools import mask as mask_utils
        rle = mask_utils.frPyObjects(segm, h, w) if isinstance(segm.get('counts'), list) else segm
        m = mask_utils.decode(rle).astype(np.uint8)
        if m.ndim == 3:
            m = m.max(2)
    return m.astype(bool)


IMG_EXT = ('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp', '.webp')
_TAG_RE = re.compile(r'(^|[_\-.])(before|after|pre|post|t1|t2|a|b|img1|img2|image1|image2)(?=$|[_\-.])')


def _name_keys(name):
    """Ключі для зіставлення імені файлу: повний stem → stem без тегів before/after → числові групи."""
    base = Path(str(name).replace('\\', '/')).name.lower()
    stem = base.split('.')[0]
    keys = [stem]
    clean = _TAG_RE.sub(r'\1', stem).strip('_-.')
    if clean and clean != stem:
        keys.append(clean)
    keys += ['#' + str(int(d)) for d in reversed(re.findall(r'\d+', stem))]
    return keys


def _index_images(folder):
    """Рекурсивний індекс зображень (враховує підтеки, напр. за департаментами)."""
    idx = defaultdict(list)
    files = [p for p in Path(folder).rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXT]
    for p in files:
        for k in dict.fromkeys(_name_keys(p.name)):
            idx[k].append(p)
    return idx, files


def _path_fields(im, words):
    """Рядкові поля COCO-запису, що схожі на шлях до зображення і містять одне зі слів (before/after…)."""
    out = []
    for k, v in im.items():
        if isinstance(v, str) and v.lower().endswith(IMG_EXT) and any(w in k.lower() for w in words):
            out.append(v)
    return out


def _resolve_direct(rel, bases):
    for b in bases:
        p = Path(b) / rel
        if p.is_file():
            return p
    return None


def _match_pair(im, idx_b, idx_a, bases):
    """Повертає (pre, post, спосіб) або (None, None, None)."""
    pb, pa = _path_fields(im, ('before', 'pre', 't1')), _path_fields(im, ('after', 'post', 't2'))
    if pb and pa:                                         # явні шляхи у COCO-записі
        p1, p2 = _resolve_direct(pb[0], bases), _resolve_direct(pa[0], bases)
        if p1 is not None and p2 is not None:
            return p1, p2, 'explicit_path'
    names = [im.get('file_name', '')] + pb + pa + [str(im.get(k, '')) for k in ('name', 'image_name', 'tile_id', 'patch_id')]
    keys = []
    for n in names:
        if n:
            keys += _name_keys(n)
    keys.append('#' + str(int(im['id'])) if str(im.get('id', '')).isdigit() else '')
    for k in dict.fromkeys(k for k in keys if k):
        lb, la = idx_b.get(k, []), idx_a.get(k, [])
        if len(lb) == 1 and len(la) == 1:
            return lb[0], la[0], ('number' if k.startswith('#') else 'stem')
    return None, None, None


def _naming_diagnostics(split, coco, files_b, files_a, img_dir):
    print(f'\n[{split}] ДІАГНОСТИКА іменування (скопіюйте це, якщо зіставлення не вдалося):')
    for im in coco['images'][:3]:
        print('   COCO image:', {k: (v if not isinstance(v, (list, dict)) else '…') for k, v in im.items()})
    print(f'   before/: {len(files_b)} файлів, напр.:', [str(p.relative_to(img_dir)) for p in files_b[:4]])
    print(f'   after/ : {len(files_a)} файлів, напр.:', [str(p.relative_to(img_dir)) for p in files_a[:4]])


def prepare_fotbcd_instances(root):
    root = Path(root)
    rows, stats = [], Counter()
    split_dirs = {'train': ['train'], 'val': ['val', 'validation'], 'test': ['test']}
    for split, names in split_dirs.items():
        ann = next((root / 'annotations' / f'{n}.json' for n in names if (root / 'annotations' / f'{n}.json').exists()), None)
        img_dir = next((root / 'images' / n for n in names if (root / 'images' / n).is_dir()), None)
        if ann is None or img_dir is None:
            raise FileNotFoundError(f'FOTBCD-Instances: немає анотацій/зображень для {split} у {root}')
        coco = json.loads(ann.read_text())
        cats = {c['id']: str(c['name']).upper() for c in coco['categories']}
        print(f'[{split}] категорії COCO:', cats)
        cid = {v: k for k, v in cats.items()}
        if not {'DEMOLISHED', 'NEW'} <= set(cid):
            raise ValueError(f'Очікувались категорії DEMOLISHED/NEW, отримано {cats}')
        by_img = defaultdict(list)
        for a in coco['annotations']:
            by_img[a['image_id']].append(a)
        dir_b = next((img_dir / n for n in ('before', 'Before', 'A', 'pre', 't1', 'im1') if (img_dir / n).is_dir()), None)
        dir_a = next((img_dir / n for n in ('after', 'After', 'B', 'post', 't2', 'im2') if (img_dir / n).is_dir()), None)
        if dir_b is None or dir_a is None:
            print('Вміст', img_dir, ':', sorted(p.name for p in img_dir.iterdir())[:20])
            raise FileNotFoundError(f'FOTBCD-Instances {split}: не знайдено теки before/after у {img_dir}')
        idx_b, files_b = _index_images(dir_b)
        idx_a, files_a = _index_images(dir_a)
        bases = [img_dir, root / 'images', root, dir_b.parent]
        images = sorted(coco['images'], key=lambda im: im['id'])
        n_total = len(images)
        # спочатку зіставляємо ВСІ записи спліту — щоб помилка іменування проявилась одразу і явно
        matched = {}
        how = Counter()
        for im in images:
            pre_p, post_p, m = _match_pair(im, idx_b, idx_a, bases)
            if pre_p is not None:
                matched[im['id']] = (pre_p, post_p); how[m] += 1
        print(f'[{split}] зіставлено пар: {len(matched)}/{n_total} (спосіб: {dict(how)})')
        if len(matched) < n_total:
            _naming_diagnostics(split, coco, files_b, files_a, img_dir)
        if len(matched) < 0.5 * n_total:
            raise RuntimeError(f'FOTBCD-Instances {split}: зіставлено лише {len(matched)}/{n_total} пар. '
                               'Надішліть діагностику вище, або тимчасово встановіть FOTBCD_MODE = "binary".')
        images = [im for im in images if im['id'] in matched]
        lim = FOT_INST_LIMITS.get(split)
        if lim and len(images) > lim:
            r = random.Random(SEED * 100 + {'train': 1, 'val': 2, 'test': 3}[split])   # детерміновано між сесіями
            images = sorted(r.sample(images, lim), key=lambda im: im['id'])
        out_dir = DATA / 'FOTBCD' / split
        for im in tqdm(images, desc=f'FOTBCD-Instances {split}'):
            pre, post = matched[im['id']]
            stem = re.sub(r'[^0-9A-Za-z_-]+', '_', Path(str(im.get('file_name') or im['id'])).stem)
            h, w = int(im.get('height', 512)), int(im.get('width', 512))
            dem = np.zeros((h, w), bool); new = np.zeros((h, w), bool); fp = np.zeros((h, w), bool)
            for a in by_img.get(im['id'], []):
                m = _coco_polys_to_mask(a.get('segmentation', []), h, w)
                name = cats.get(a['category_id'], '')
                fp |= m
                if name == 'DEMOLISHED':
                    dem |= m
                elif name == 'NEW':
                    new |= m
            lab = np.zeros((h, w), np.uint8)
            lab[dem] = LBL_DESTROYED
            lab[new] = LBL_NEWBLD
            both = dem & new
            lab[both] = LBL_IGNORE
            stats['overlap_new_demolished_px'] += int(both.sum())
            uid = f'fot_{split}_{im["id"]}_{stem}'[:80]
            lp, fpp = out_dir / f'{uid}_label.png', out_dir / f'{uid}_fp.png'
            write_gray(lp, lab); write_gray(fpp, fp.astype(np.uint8) * 255)
            group = next((str(im[k]) for k in ('dep', 'department', 'departement', 'dept') if k in im), split)
            rows.append(dict(uid=uid, source='FOTBCD', subset='FOTBCD-Instances', split=split, pre=str(pre), post=str(post),
                             label=str(lp), aux='', fp=str(fpp), gsd=FOT_GSD, group=group, base_uid='', h=h, w=w,
                             sun_az=np.nan, sun_el=np.nan,
                             chg_px=int(((lab >= 1) & (lab <= 4)).sum()),
                             n_destroyed_px=int(dem.sum()), n_new_px=int(new.sum())))
    print('Статистика FOTBCD-Instances:', dict(stats))
    if not rows:
        raise RuntimeError('FOTBCD-Instances: не підготовлено жодної пари.')
    return pd.DataFrame(rows)


def prepare_fotbcd_binary():
    try:
        from datasets import load_dataset
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'datasets>=3,<5'], check=True)
        from datasets import load_dataset
    rows = []
    for split_hf, split in (('train', 'train'), ('validation', 'val'), ('test', 'test')):
        lim = FOT_BIN_LIMITS[split]
        ds = load_dataset(FOT_BIN_HF_ID, split=split_hf, streaming=True, revision=FOT_BIN_REVISION)
        need = {'image_id', 'image_before', 'image_after', 'label', 'dep'}
        if not need <= set(ds.features):
            raise ValueError(f'Неочікувані поля FOTBCD-Binary: {ds.features}')
        out_dir = DATA / 'FOTBCD' / split
        for i, ex in enumerate(tqdm(ds.shuffle(seed=SEED, buffer_size=256).take(lim), total=lim, desc=f'FOTBCD-Binary {split}')):
            a = np.asarray(ex['image_before'].convert('RGB'))
            b = np.asarray(ex['image_after'].convert('RGB'))       # саме AFTER (помилка v3)
            m = np.asarray(ex['label'].convert('L')) > 127
            if m.any() and np.array_equal(a, b):
                raise RuntimeError(f'ЗУПИНЕНО: pre == post при позитивній масці ({split}/{ex["image_id"]})')
            uid = f'fotb_{split}_{i:05d}'
            pp, qp, lp = out_dir / f'{uid}_pre.png', out_dir / f'{uid}_post.png', out_dir / f'{uid}_label.png'
            write_rgb(pp, a); write_rgb(qp, b)
            lab = np.where(m, LBL_UNKNOWN, LBL_BG).astype(np.uint8)
            write_gray(lp, lab)
            rows.append(dict(uid=uid, source='FOTBCD', subset='FOTBCD-Binary', split=split, pre=str(pp), post=str(qp),
                             label=str(lp), aux='', fp='', gsd=FOT_GSD, group=str(ex['dep']), base_uid='',
                             h=a.shape[0], w=a.shape[1], sun_az=np.nan, sun_el=np.nan, chg_px=int(m.sum()),
                             n_destroyed_px=0, n_new_px=0, hf_image_id=str(ex['image_id'])))
    return pd.DataFrame(rows)


fot_df = None
if RUN_PREPARE or (DATA / 'FOTBCD_manifest.csv').exists():
    inst_root = locate_fotbcd_instances() if FOTBCD_MODE in ('auto', 'instances') else None
    if FOTBCD_MODE == 'instances' and inst_root is None:
        raise FileNotFoundError('FOTBCD_MODE="instances", але архів не знайдено: ' + str(FOT_INSTANCES_CANDIDATES))
    mode = 'instances' if inst_root is not None else 'binary'
    cfg = dict(mode=mode, root=str(inst_root), limits=FOT_INST_LIMITS if mode == 'instances' else FOT_BIN_LIMITS,
               rev=FOT_BIN_REVISION, seed=SEED, v=2)
    fot_df = part_ready('FOTBCD', cfg)
    if fot_df is None and RUN_PREPARE:
        with Timer(f'FOTBCD ({mode})'):
            fot_df = prepare_fotbcd_instances(inst_root) if mode == 'instances' else prepare_fotbcd_binary()
        fot_df = part_save('FOTBCD', cfg, fot_df)
    FOTBCD_ACTIVE_MODE = mode
    print(f'FOTBCD режим: {mode.upper()}', '(типи NEW/DEMOLISHED доступні)' if mode == 'instances'
          else '(лише бінарна зміна; тип невідомий → голова типу на FOTBCD не навчається)')
    if fot_df is not None and len(fot_df):
        display(fot_df.groupby('split').agg(pairs=('uid', 'count'), groups=('group', 'nunique'),
                                            with_change=('chg_px', lambda x: int((x > 0).sum()))))

In [ ]:
# ==============================================================================
# 3.2 xBD Global: «зруйнований об'єкт» = major-damage + destroyed (будівлі після стихійних лих)
# ==============================================================================
from shapely import wkt as shapely_wkt
from PIL import Image

DAMAGE_POS = {'major-damage', 'destroyed'}
DAMAGE_AMBIG = {'minor-damage', 'un-classified'}


def _safe_member_path(name):
    p = Path(name)
    if p.is_absolute() or '..' in p.parts:
        raise ValueError(f'Небезпечний шлях у tar: {name}')
    return p


def extract_xbd_tar(tar_path, dest):
    """Потокове розпакування лише images/ та labels/ (targets/ пропускаємо — економія диска)."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    n = 0
    with tarfile.open(tar_path, 'r|*') as tf:
        for m in tqdm(tf, desc='xBD tar', unit='file'):
            if not m.isfile():
                continue
            parts = _safe_member_path(m.name).parts
            if not (('images' in parts and m.name.endswith('.png')) or ('labels' in parts and m.name.endswith('.json'))):
                continue
            out = dest / _safe_member_path(m.name)
            out.parent.mkdir(parents=True, exist_ok=True)
            with tf.extractfile(m) as src, open(out, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            n += 1
    (dest / '.extracted').write_text(str(n))
    return n


def locate_xbd_root():
    for c in XBD_DIR_CANDIDATES:
        if Path(c).is_dir() and next(Path(c).rglob('*_post_disaster.json'), None) is not None:
            return Path(c)
    dest = RAW / 'xbd_global'
    if XBD_TAR_ON_DRIVE.exists():
        tar_path, downloaded = XBD_TAR_ON_DRIVE, False
    elif XBD_ALLOW_HF_DOWNLOAD:
        from huggingface_hub import hf_hub_download
        print('Завантаження xBD Global з Hugging Face (~8.4 ГБ)…')
        with Timer('xBD download'):
            tar_path = Path(hf_hub_download(repo_id=XBD_HF_REPO, repo_type='dataset', filename=XBD_HF_FILE,
                                            local_dir=str(RAW / 'xbd_tar')))
        downloaded = True
    else:
        return None
    with Timer('xBD extract'):
        n = extract_xbd_tar(tar_path, dest)
    print('Витягнуто файлів:', n)
    if downloaded and XBD_DELETE_TAR_AFTER:
        Path(tar_path).unlink(missing_ok=True)
    return dest


def parse_xbd_label(json_path):
    """Повертає метадані та полігони (у пікселях) для одного *_post_disaster.json."""
    d = json.loads(Path(json_path).read_text())
    meta = d.get('metadata', {}) or {}
    polys = []
    for f in d.get('features', {}).get('xy', []):
        sub = str(f.get('properties', {}).get('subtype', 'no-damage')).lower()
        try:
            g = shapely_wkt.loads(f['wkt'])
        except Exception:
            continue
        for gg in getattr(g, 'geoms', [g]):
            if gg.geom_type == 'Polygon' and not gg.is_empty:
                polys.append((sub, np.asarray(gg.exterior.coords, np.float64)[:, :2], float(gg.area)))
    lon, lat = [], []
    for f in d.get('features', {}).get('lng_lat', []):
        try:
            c = shapely_wkt.loads(f['wkt']).centroid
            lon.append(c.x); lat.append(c.y)
        except Exception:
            pass
    res = meta.get('pan_resolution') or meta.get('gsd') or XBD_DEFAULT_GSD
    return dict(meta=meta, polys=polys, lon=float(np.mean(lon)) if lon else np.nan,
                lat=float(np.mean(lat)) if lat else np.nan,
                gsd=float(res) if res else XBD_DEFAULT_GSD,
                sun_az=float(meta['sun_azimuth']) if meta.get('sun_azimuth') is not None else np.nan,
                sun_el=float(meta['sun_elevation']) if meta.get('sun_elevation') is not None else np.nan,
                w=int(meta.get('width', 1024) or 1024), h=int(meta.get('height', 1024) or 1024))


def rasterize_xbd(info, h, w):
    lab = np.zeros((h, w), np.uint8)
    fp = np.zeros((h, w), np.uint8)
    for sub, pts, _ in info['polys']:
        poly = [np.round(pts).astype(np.int32)]
        cv2.fillPoly(fp, poly, 255)
        if sub in DAMAGE_AMBIG and XBD_MINOR_AS == 'ignore':
            cv2.fillPoly(lab, poly, LBL_IGNORE)
    for sub, pts, _ in info['polys']:            # «зруйновано» має пріоритет
        if sub in DAMAGE_POS:
            cv2.fillPoly(lab, [np.round(pts).astype(np.int32)], LBL_DESTROYED)
    return lab, fp


def index_xbd_pairs(root):
    pngs = {p.stem: p for p in Path(root).rglob('*.png') if 'target' not in p.stem}
    rows = []
    for jp in tqdm(sorted(Path(root).rglob('*_post_disaster.json')), desc='xBD labels'):
        stem = jp.stem
        post = pngs.get(stem); pre = pngs.get(stem.replace('_post_disaster', '_pre_disaster'))
        if post is None or pre is None:
            continue
        info = parse_xbd_label(jp)
        pos_area = sum(a for s, _, a in info['polys'] if s in DAMAGE_POS)
        rows.append(dict(stem=stem, pre=str(pre), post=str(post), json=str(jp),
                         event=str(info['meta'].get('disaster') or stem.split('_')[0]),
                         n_bld=len(info['polys']), pos_area=pos_area,
                         n_minor=sum(s in DAMAGE_AMBIG for s, _, _ in info['polys']),
                         lon=info['lon'], lat=info['lat'], gsd=info['gsd'], sun_az=info['sun_az'], sun_el=info['sun_el'],
                         h=info['h'], w=info['w']))
    return pd.DataFrame(rows)


def assign_xbd_splits(idx):
    idx = idx.copy()
    if XBD_SPLIT_MODE == 'event':
        ev2s = {e: s for s, evs in XBD_EVENT_SPLIT.items() for e in evs}
        idx['split'] = idx['event'].map(lambda e: ev2s.get(e, 'train'))
        idx['block'] = idx['event']
        return idx
    has_xy = np.isfinite(idx['lon']) & np.isfinite(idx['lat'])
    print(f'xBD: координати є у {int(has_xy.sum())}/{len(idx)} пар')
    blk = pd.Series(index=idx.index, dtype=object)
    blk[has_xy] = (idx.loc[has_xy, 'event'] + ':' +
                   np.floor(idx.loc[has_xy, 'lon'] / XBD_BLOCK_DEG).astype(int).astype(str) + ':' +
                   np.floor(idx.loc[has_xy, 'lat'] / XBD_BLOCK_DEG).astype(int).astype(str))
    # запасний варіант без координат: послідовні ID однієї події → псевдо-блоки по 25 тайлів
    num = idx['stem'].str.extract(r'_(\d+)_post', expand=False).fillna('0').astype(int)
    blk[~has_xy] = idx.loc[~has_xy, 'event'] + ':id' + (num[~has_xy] // 25).astype(str)
    idx['block'] = blk
    rng = random.Random(SEED)
    split_of = {}
    for ev, g in idx.groupby('event'):
        blocks = g.groupby('block').size()
        order = list(blocks.index); rng.shuffle(order)
        total = len(g); have = {s: 0 for s in XBD_SPLIT_RATIOS}
        for b in order:                                   # жадібно: у спліт з найбільшим дефіцитом
            s = max(XBD_SPLIT_RATIOS, key=lambda k: XBD_SPLIT_RATIOS[k] * total - have[k])
            split_of[b] = s; have[s] += int(blocks[b])
    idx['split'] = idx['block'].map(split_of)
    return idx


def _round_robin_by_event(df, n, seed):
    groups = [g.sample(frac=1, random_state=seed) for _, g in df.groupby('event')]
    out, i = [], 0
    while len(out) < n and any(i < len(g) for g in groups):
        for g in groups:
            if i < len(g) and len(out) < n:
                out.append(g.iloc[i])
        i += 1
    return pd.DataFrame(out)


def prepare_xbd(root):
    idx = index_xbd_pairs(root)
    print('Пар xBD з мітками:', len(idx), '| подій:', idx['event'].nunique())
    idx = idx[idx['n_bld'] > 0]
    idx = assign_xbd_splits(idx)
    # площа в px² від shapely ≈ кількість пікселів полігона
    pos_all = idx[idx['pos_area'] >= XBD_MIN_POS_PX]
    neg_all = idx[(idx['pos_area'] == 0)]
    sel = []
    for split, n_pos in XBD_POS_LIMITS.items():
        p = _round_robin_by_event(pos_all[pos_all['split'] == split], n_pos, SEED)
        n_neg = int(round(len(p) * XBD_NEG_FRACTION / (1 - XBD_NEG_FRACTION)))
        q = _round_robin_by_event(neg_all[neg_all['split'] == split], n_neg, SEED + 1)
        sel += [p, q]
    sel = pd.concat(sel, ignore_index=True)
    rows = []
    for r in tqdm(sel.itertuples(), total=len(sel), desc='xBD → data'):
        info = parse_xbd_label(r.json)
        with Image.open(r.post) as im:                      # лише заголовок, без декодування
            w, h = im.size
        lab, fp = rasterize_xbd(info, h, w)
        out = DATA / 'xBD' / r.split
        uid = f'xbd_{r.stem.replace("_post_disaster", "")}'
        pp, qp = out / f'{uid}_pre.png', out / f'{uid}_post.png'
        out.mkdir(parents=True, exist_ok=True)
        shutil.copy2(r.pre, pp); shutil.copy2(r.post, qp)
        lp, fpp = out / f'{uid}_label.png', out / f'{uid}_fp.png'
        write_gray(lp, lab); write_gray(fpp, fp)
        rows.append(dict(uid=uid, source='xBD', subset='xBD-Global', split=r.split, pre=str(pp), post=str(qp),
                         label=str(lp), aux='', fp=str(fpp), gsd=r.gsd, group=r.block, base_uid='', h=h, w=w,
                         sun_az=r.sun_az, sun_el=r.sun_el, chg_px=int((lab == LBL_DESTROYED).sum()),
                         n_destroyed_px=int((lab == LBL_DESTROYED).sum()), n_new_px=0, event=r.event))
    return pd.DataFrame(rows)


xbd_df = None
xbd_cfg = dict(split=XBD_SPLIT_MODE, block=XBD_BLOCK_DEG, ratios=XBD_SPLIT_RATIOS, ev=XBD_EVENT_SPLIT,
               pos=XBD_POS_LIMITS, neg=XBD_NEG_FRACTION, minpx=XBD_MIN_POS_PX, minor=XBD_MINOR_AS, seed=SEED)
xbd_df = part_ready('xBD', xbd_cfg)
if xbd_df is None and RUN_PREPARE:
    xbd_root = locate_xbd_root()
    if xbd_root is None:
        print('⚠ xBD Global не знайдено і завантаження вимкнено — продовжуємо без xBD.')
    else:
        with Timer('xBD підготовка'):
            xbd_df = prepare_xbd(xbd_root)
        xbd_df = part_save('xBD', xbd_cfg, xbd_df)
if xbd_df is not None and len(xbd_df):
    print(f'xBD Global, спліт = {XBD_SPLIT_MODE}; minor-damage → {XBD_MINOR_AS}')
    display(xbd_df.groupby(['split']).agg(pairs=('uid', 'count'), events=('event', 'nunique'), blocks=('group', 'nunique'),
                                          with_destroyed=('chg_px', lambda x: int((x > 0).sum()))))
    display(pd.crosstab(xbd_df['event'], xbd_df['split']))

In [ ]:
# ==============================================================================
# 3.3 xBD-Ukraine — ЛИШЕ для оцінки (out-of-distribution): реальні руйнування в українських містах
# ==============================================================================
# Модель на цих знімках НЕ навчається. Це найближчий до цільового домену реальний тест для класу
# «зруйнований об'єкт». Розмічені лише будівлі: вирви/траншеї тут не розмічені, тому для цього
# джерела рахуємо F1 саме класу 1 та бінарні метрики з відповідною оговоркою.


def prepare_xbd_ukr():
    from huggingface_hub import snapshot_download
    root = RAW / 'xbd_ukraine'
    need_download = (not root.exists()) or next(root.rglob('*.json'), None) is None
    if need_download:
        with Timer('xBD-Ukraine download'):
            snapshot_download(repo_id=XBD_UKR_HF_REPO, repo_type='dataset', local_dir=str(root))
    pngs = {p.stem: p for p in root.rglob('*.png') if not any(k in p.stem.lower() for k in ('target', 'mask'))}
    jsons = {p.stem: p for p in root.rglob('*.json')}
    rows, skipped = [], Counter()
    for stem, post in sorted(pngs.items()):
        if 'post' not in stem:
            continue
        cands = [stem.replace('_post_disaster', '_pre_disaster'), stem.replace('_post_', '_pre_'), stem.replace('post', 'pre')]
        pre = next((pngs[c] for c in cands if c in pngs and c != stem), None)
        js = jsons.get(stem)
        if pre is None or js is None:
            skipped['no_pair_or_json'] += 1
            continue
        info = parse_xbd_label(js)
        if not info['polys']:
            skipped['no_buildings'] += 1
            continue
        a, b = read_rgb(pre), read_rgb(post)
        if a.shape != b.shape:
            b = cv2.resize(b, (a.shape[1], a.shape[0]), interpolation=cv2.INTER_LINEAR)
            skipped['resized_post'] += 1
        h, w = a.shape[:2]
        lab, fp = rasterize_xbd(info, h, w)
        uid = f'ukr_{stem}'
        out = DATA / 'xBD_UKR' / 'test'
        pp, qp, lp, fpp = (out / f'{uid}_{k}.png' for k in ('pre', 'post', 'label', 'fp'))
        write_rgb(pp, a); write_rgb(qp, b); write_gray(lp, lab); write_gray(fpp, fp)
        rows.append(dict(uid=uid, source='xBD_UKR', subset='xBD-Ukraine', split='test', pre=str(pp), post=str(qp),
                         label=str(lp), aux='', fp=str(fpp), gsd=XBD_UKR_GSD, group=stem.split('_')[0], base_uid='',
                         h=h, w=w, sun_az=np.nan, sun_el=np.nan, chg_px=int((lab == LBL_DESTROYED).sum()),
                         n_destroyed_px=int((lab == LBL_DESTROYED).sum()), n_new_px=0, event=stem.split('_')[0]))
    print('xBD-Ukraine: пар', len(rows), '| пропущено:', dict(skipped))
    return pd.DataFrame(rows)


ukr_df = None
if USE_XBD_UKRAINE_EVAL:
    ukr_cfg = dict(repo=XBD_UKR_HF_REPO, minor=XBD_MINOR_AS)
    ukr_df = part_ready('xBD_UKR', ukr_cfg)
    if ukr_df is None and RUN_PREPARE:
        try:
            ukr_df = prepare_xbd_ukr()
            ukr_df = part_save('xBD_UKR', ukr_cfg, ukr_df)
        except Exception as e:                       # необов'язкове джерело не повинно ламати пайплайн
            print('⚠ xBD-Ukraine недоступний:', repr(e)[:300])
            ukr_df = None
    if ukr_df is not None and len(ukr_df):
        print('xBD-Ukraine (test-only):', len(ukr_df), 'пар; міста/групи:', ukr_df['group'].value_counts().to_dict())

## 4. Синтетика: нові вирви та свіжі фортифікації

Публічної розмітки вирв і фортифікацій «до/після» немає, тому ці два класи навчаємо на синтетиці. Генератор переписано з урахуванням критики v3:

| Проблема v3 | Що зроблено |
|---|---|
| мозаїка 2×2 з видимими швами | лише **цілі** реальні пари: FOTBCD 512×512 (0,2 м/px) та xBD 1024×1024 (≈0,3–0,6 м/px) |
| зміни «намальовані», виділяються | рендер **карти висот**: чаша, нерівний вал, рваний викид ґрунту з «променями», освітлення від сонця й **відкинуті тіні**; колір ґрунту під яскравість сцени, приглушена насиченість; шум/розмиття під знімок. На xBD-основах — **реальний азимут і висота сонця** з метаданих |
| неправдоподібне розміщення | карта «природного ґрунту» (рослинність/ґрунт за кольором) — траншеї лише на полях/луках/лісосмугах, вирви переважно там же, на дахах — практично ніколи; розміри **в метрах** через GSD |
| мітка там, де зміну майже не видно | **мітка за видимістю**: піксель розмічається лише там, де рендер реально змінив знімок |
| модель може вчити «сліди генератора» | **«старі» вирви/траншеї на ОБОХ знімках з міткою 0** — те саме походження, але без зміни. Окремо міряємо частку хибних спрацювань на них (*shortcut-тест*) |
| train і test з одного генератора | окремий **Synthetic-Shift**: відкладена палітра ґрунтів (крейда, червонозем, кіптява, лес), низьке сонце, ширший діапазон розмірів — у навчанні їх немає |

Фортифікації: стрілецькі траншеї (зигзаг, «зубчасті» з траверсами), хвилясті ходи сполучення, сегментовані лінії, протитанкові рови з бруствером, бліндажі/позиції біля траншей, «зуби дракона». Вирви: суміш калібрів (діаметр ≈1–16 м), скупчення (обстріли) та поодинокі влучання.

Синтетика будується **лише на основах зі свого спліту** (train-основи → synthetic-train тощо); походження кожного зразка записано в `base_uid` і перевіряється аудитом.

In [ ]:
# ==============================================================================
# 4.1 Синтетичний генератор: нові вирви та свіжі фортифікації
# ==============================================================================
# Принципи (відповідь на критику v3):
#   * без мозаїк: беремо ЦІЛІ реальні пари «до/після» (різні дати, реальні сезонні відмінності);
#   * об'єкти рендеряться через карту висот: чаша + вал + рваний викид ґрунту, освітлення
#     від сонця (hillshade) і ВІДКИНУТІ тіні; колір ґрунту адаптується до яскравості сцени;
#     шум і різкість підганяються під знімок;
#   * розміри задаються в МЕТРАХ і переводяться в пікселі через GSD знімка;
#   * «старі» об'єкти малюються на ОБОХ знімках з міткою 0 (без змін) — контроль того,
#     що модель вчить саме ЗМІНУ, а не артефакти генератора;
#   * окремий режим "shift" (відкладені палітри/параметри) для чесного тесту переносу.
# (numpy, cv2 та константи міток LBL_* уже визначені в конфігурації)

# Палітри свіжого ґрунту (RGB, приблизно для середньої яскравості сцени ~115)
SOIL_PALETTES = {
    # 'A' — для train / val / test-ID
    'A': {'dark_brown': (88, 72, 58), 'brown': (122, 100, 80), 'grey_brown': (118, 108, 98),
          'tan': (158, 136, 108), 'ochre': (140, 118, 82)},
    # 'B' — ВІДКЛАДЕНА палітра лише для test-shift (у навчанні не зустрічається)
    'B': {'chalk': (196, 190, 176), 'reddish': (148, 98, 74), 'scorched': (62, 58, 55),
          'loess_yellow': (178, 158, 112)},
}

SYN_REGIMES = {
    # діапазони для різних режимів генерації
    'id':    dict(palette='A', sun_elev=(28, 62), crater_diam_mix=((1.2, 2.5, .35), (2.5, 6.0, .50), (6.0, 13.0, .15)),
                  trench_styles=('zigzag', 'crenellated', 'wavy', 'segmented'), p_dragon=0.10, p_ditch=0.12),
    'shift': dict(palette='B', sun_elev=(18, 30), crater_diam_mix=((1.0, 2.0, .25), (2.0, 7.0, .45), (7.0, 16.0, .30)),
                  trench_styles=('zigzag', 'crenellated', 'wavy', 'segmented'), p_dragon=0.18, p_ditch=0.18),
}


# ------------------------------------------------------------------ шуми / утиліти
def _smoothstep(e0, e1, x):
    t = np.clip((x - e0) / (e1 - e0 + 1e-9), 0, 1)
    return t * t * (3 - 2 * t)


def fractal_noise(h, w, rng, sigmas=(1.0, 2.5, 6.0, 14.0), weights=(0.15, 0.3, 0.35, 0.2)):
    """Гладкий багатомасштабний шум, нормований до std≈1."""
    out = np.zeros((h, w), np.float32)
    for s, wt in zip(sigmas, weights):
        n = rng.standard_normal((h, w)).astype(np.float32)
        n = cv2.GaussianBlur(n, (0, 0), s)
        out += wt * n / (n.std() + 1e-6)
    return out / (out.std() + 1e-6)


def scene_stats(img):
    g = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)
    hp = g - cv2.GaussianBlur(g, (0, 0), 1.2)
    noise = 1.4826 * np.median(np.abs(hp - np.median(hp)))
    return {'lum': float(np.median(g)), 'noise': float(np.clip(noise, 0.8, 6.0))}


def ground_maps(img, exclusion=None):
    """Повертає дві карти ймовірностей розміщення:
    natural — природний ґрунт/рослинність (поля, луки, лісосмуги, оголений ґрунт): для траншей і вирв;
    any     — природний ґрунт + слабка вага для сірих покриттів (дороги/майданчики): лише для вирв.
    Дахи (дуже світлі однорідні та сірі/сині поверхні) майже виключаються.
    exclusion — маска, де розміщувати НЕ можна (будівлі, вже розмічені зміни)."""
    f = img.astype(np.float32) / 255.0
    r, g, b = f[..., 0], f[..., 1], f[..., 2]
    exg = (2 * g - r - b) / (r + g + b + 1e-3)
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV).astype(np.float32)
    hue, sat, val = hsv[..., 0] * 2, hsv[..., 1] / 255, hsv[..., 2] / 255
    veg = _smoothstep(0.0, 0.08, exg)
    soil = ((hue >= 17) & (hue <= 55) & (sat >= 0.10) & (sat <= 0.60) & (val > 0.22) & (val < 0.92)).astype(np.float32)
    natural = np.maximum(veg, soil)
    natural = cv2.GaussianBlur(natural, (0, 0), 3)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    mag = cv2.GaussianBlur(np.sqrt(gx * gx + gy * gy), (0, 0), 5)
    calm = np.clip(1.0 - mag / (np.percentile(mag, 97) + 1e-6), 0, 1)
    lum = cv2.GaussianBlur(gray, (0, 0), 3)
    not_extreme = np.clip((lum - 30) / 30, 0, 1) * np.clip((215 - lum) / 35, 0, 1)
    nat = natural * calm ** 1.5 * not_extreme                 # траншеї: відкритий ґрунт/луки, не під кронами
    manmade_flat = (1 - natural) * calm * np.clip((lum - 40) / 30, 0, 1) * np.clip((165 - lum) / 30, 0, 1)
    anyp = natural * (0.25 + 0.75 * calm) * not_extreme + 0.10 * manmade_flat   # вирви: будь-де, переважно на ґрунті
    if exclusion is not None and exclusion.any():
        ex = cv2.dilate(exclusion.astype(np.uint8), np.ones((15, 15), np.uint8)) > 0
        nat[ex] = 0
        anyp[ex] = 0
    return nat.astype(np.float32), anyp.astype(np.float32)


def sample_point(prob, rng, margin=8):
    h, w = prob.shape
    p = prob[margin:h - margin, margin:w - margin].ravel().astype(np.float64)
    s = p.sum()
    if s <= 1e-6:
        return None
    i = rng.choice(p.size, p=p / s)
    y, x = divmod(i, w - 2 * margin)
    return x + margin, y + margin


# ------------------------------------------------------------------ рендер карти висот
class Canvas:
    """Накопичує висоти/покриття ґрунтом для ОДНОГО знімка, потім рендерить за один прохід.
    Так тіні від сусідніх об'єктів (вал вирви, бруствер траншеї) взаємодіють коректно."""

    def __init__(self, img, gsd, sun_az, sun_el, rng, palette_rgb, lum):
        self.img = img
        self.h, self.w = img.shape[:2]
        self.gsd = gsd
        self.rng = rng
        self.height = np.zeros((self.h, self.w), np.float32)    # метри
        self.alpha = np.zeros((self.h, self.w), np.float32)     # покриття свіжим ґрунтом
        self.color = np.zeros((self.h, self.w, 3), np.float32)  # колір ґрунту (передмножений на alpha)
        self.dark = np.zeros((self.h, self.w), np.float32)      # затемнення (вологий ґрунт/кіптява)
        self.sun_az, self.sun_el = np.deg2rad(sun_az), np.deg2rad(sun_el)
        k = np.clip((lum / 115.0) ** 0.7, 0.6, 1.5)
        self.soil = np.array(palette_rgb, np.float32) * k * rng.uniform(0.92, 1.08)
        self.relief = rng.uniform(0.45, 0.8)                  # сила 3D-освітлення (надир: слабка)
        self.shadow_k = rng.uniform(0.40, 0.60)               # глибина тіні (розсіяне світло)

    def add(self, y0, y1, x0, x1, height, alpha, soil_rgb, dark=None):
        sl = (slice(y0, y1), slice(x0, x1))
        self.height[sl] += height
        a_old = self.alpha[sl]
        a_new = np.maximum(a_old, alpha)
        # колір змішуємо пропорційно приросту покриття; soil_rgb — (3,) або карта (h, w, 3)
        gain = np.clip(a_new - a_old, 0, 1)[..., None]
        self.color[sl] = self.color[sl] * (1 - gain) + np.asarray(soil_rgb, np.float32) * gain
        self.alpha[sl] = a_new
        if dark is not None:
            self.dark[sl] = np.maximum(self.dark[sl], dark)

    def render(self, noise_sigma, blur_sigma):
        """Рендер лише в межах активної області (швидко)."""
        act = (np.abs(self.height) > 1e-4) | (self.alpha > 1e-3) | (self.dark > 1e-3)
        if not act.any():
            return self.img.copy()
        ys, xs = np.where(act)
        pad = 90
        y0, y1 = max(0, ys.min() - pad), min(self.h, ys.max() + pad + 1)
        x0, x1 = max(0, xs.min() - pad), min(self.w, xs.max() + pad + 1)
        sub = _SubCanvas(self, y0, y1, x0, x1)
        res = self.img.copy()
        res[y0:y1, x0:x1] = sub._render(noise_sigma, blur_sigma)
        return res

    def _render(self, noise_sigma, blur_sigma):
        img = self.img.astype(np.float32)
        hpx = self.height / self.gsd                      # висота в «пікселях» для нахилів
        hrel = hpx * self.relief                           # нахили для дифузного освітлення (приглушені)
        gy, gx = np.gradient(hrel)
        nx, ny, nz = -gx, -gy, np.ones_like(gx)
        nrm = np.sqrt(nx * nx + ny * ny + 1)
        lx = np.cos(self.sun_el) * np.sin(self.sun_az)
        ly = -np.cos(self.sun_el) * np.cos(self.sun_az)
        lz = np.sin(self.sun_el)
        shade = (nx * lx + ny * ly + nz * lz) / nrm / lz
        shade = np.clip(shade, 0.45, 1.3)
        # відкинуті тіні: «марш» уздовж напрямку на сонце
        shadow = np.zeros_like(hpx, dtype=bool)
        active = np.abs(hpx) > 1e-3
        if active.any():
            dx, dy = np.sin(self.sun_az), -np.cos(self.sun_az)
            tan_el = np.tan(self.sun_el)
            max_steps = int(min(80, np.ceil((hpx.max() - hpx.min()) / max(tan_el, 0.05)) + 1))
            for s in range(1, max_steps + 1):
                ox, oy = int(round(dx * s)), int(round(dy * s))
                shifted = np.full_like(hpx, -1e9)
                ys0, ys1 = max(0, -oy), min(self.h, self.h - oy)
                xs0, xs1 = max(0, -ox), min(self.w, self.w - ox)
                shifted[ys0:ys1, xs0:xs1] = hpx[ys0 + oy:ys1 + oy, xs0 + ox:xs1 + ox]
                shadow |= shifted > hpx + s * tan_el + 0.05
        shadow_f = cv2.GaussianBlur(shadow.astype(np.float32), (0, 0), 0.7)
        # композиція: ґрунт поверх фону (частково зберігаємо текстуру фону)
        a = self.alpha[..., None]
        base_lum = img.mean(2, keepdims=True)
        soil = self.color * (1 + 0.10 * self.tex[..., None])
        # тонка модуляція текстурою фону: ґрунт не «пластиковий»
        soil = soil * (0.85 + 0.15 * base_lum / (base_lum.mean() + 1e-6))
        out = img * (1 - 0.92 * a) + soil * 0.92 * a
        out = out * (1 - 0.45 * self.dark[..., None])
        out = out * shade[..., None]
        out = out * (1 - self.shadow_k * shadow_f[..., None])
        # маска впливу: де змінилась картинка
        infl = np.clip(np.maximum(self.alpha, np.maximum(np.abs(shade - 1) * 3, shadow_f)), 0, 1)
        infl = np.maximum(infl, self.dark)
        infl = cv2.GaussianBlur(infl, (0, 0), 1.0)
        if blur_sigma > 0:
            out = cv2.GaussianBlur(out, (0, 0), blur_sigma)
        noise = self.rng.standard_normal(out.shape).astype(np.float32) * noise_sigma
        out = out + noise * infl[..., None]
        res = img * (1 - infl[..., None]) + out * infl[..., None]
        return np.clip(res, 0, 255).astype(np.uint8)


class _SubCanvas(Canvas):
    """Вирізка полотна (ті самі параметри сонця/ґрунту) для швидкого рендеру."""

    def __init__(self, parent, y0, y1, x0, x1):  # noqa: D401 — без виклику Canvas.__init__
        sl = (slice(y0, y1), slice(x0, x1))
        self.img = parent.img[sl]
        self.h, self.w = self.img.shape[:2]
        self.gsd, self.rng = parent.gsd, parent.rng
        self.height, self.alpha = parent.height[sl], parent.alpha[sl]
        self.color, self.dark = parent.color[sl], parent.dark[sl]
        self.tex = fractal_noise(self.h, self.w, parent.rng, sigmas=(0.7, 1.6, 4.0), weights=(0.4, 0.4, 0.2))
        self.sun_az, self.sun_el = parent.sun_az, parent.sun_el
        self.relief, self.shadow_k, self.soil = parent.relief, parent.shadow_k, parent.soil


# ------------------------------------------------------------------ вирва
def crater_geometry(canvas, cx, cy, diam_m, rng, soil_rgb, old=False):
    """Свіжа вирва у надирному знімку: темна волога/затінена чаша, низький нерівний вал,
    світліший рваний викид ґрунту з «променями» та окремими грудками на відстані до ~4 радіусів."""
    gsd = canvas.gsd
    r = diam_m / 2 / gsd                                     # радіус у пікселях
    ext = rng.uniform(2.2, 3.6)                              # протяжність суцільного викиду (в радіусах)
    R = int(np.ceil(r * (ext + 1.0) / (0.78 * 0.8) + 3))     # з запасом: еліпс і збурення контуру
    x0, x1 = max(0, int(cx) - R), min(canvas.w, int(cx) + R + 1)
    y0, y1 = max(0, int(cy) - R), min(canvas.h, int(cy) + R + 1)
    if x1 - x0 < 3 or y1 - y0 < 3:
        return None
    yy, xx = np.mgrid[y0:y1, x0:x1].astype(np.float32)
    dx, dy = xx - cx, yy - cy
    th, ecc = rng.uniform(0, np.pi), rng.uniform(0.78, 1.0)
    u = dx * np.cos(th) + dy * np.sin(th)
    v = (-dx * np.sin(th) + dy * np.cos(th)) / ecc
    rho = np.sqrt(u * u + v * v) / max(r, 0.5)
    phi = np.arctan2(v, u)
    hh, ww = rho.shape
    pert = np.zeros_like(rho)
    for k in range(2, 9):                                    # нерівний контур чаші
        pert += rng.normal(0, 0.06 / k ** 0.5) * np.cos(k * phi + rng.uniform(0, 2 * np.pi))
    pert = np.clip(pert, -0.2, 0.2) * min(1.0, r / 10.0)     # малі вирви — майже круглі
    edge_noise = fractal_noise(hh, ww, rng, sigmas=(max(1.2, r / 6), max(1.5, r / 3)), weights=(0.5, 0.5))
    rho_p = rho * (1 + pert) * (1 + 0.04 * edge_noise)
    # висоти в метрах: неглибока чаша (d/D≈0.12–0.22), низький нерівний вал
    depth = diam_m * rng.uniform(0.12, 0.22)
    rim_h = depth * rng.uniform(0.12, 0.30) * (1 + 0.2 * edge_noise)
    bowl = -depth * np.clip(1 - rho_p ** 2, 0, 1) ** 0.9
    rim = rim_h * np.exp(-((rho_p - 1.0) / 0.25) ** 2)
    height = bowl + rim
    # викид: промені + великі плями, спадає з відстанню
    n_rays = int(rng.integers(6, 16))
    rays = np.zeros_like(phi)
    for _ in range(n_rays):
        c, wdt = rng.uniform(-np.pi, np.pi), rng.uniform(0.10, 0.40)
        dphi = (phi - c + np.pi) % (2 * np.pi) - np.pi
        rays = np.maximum(rays, np.exp(-(dphi / wdt) ** 2) * rng.uniform(0.4, 1.0))
    patch = fractal_noise(hh, ww, rng, sigmas=(max(0.8, r / 6), max(1.2, r / 2.5), max(1.5, r)), weights=(0.2, 0.45, 0.35))
    fall = np.clip(1 - (rho_p - 0.9) / (ext - 0.9), 0, 1) ** 1.4
    ej = _smoothstep(0.0, 0.5, fall * (1.2 + 0.8 * rays) + 0.4 * patch - 0.5) * (rho_p > 0.8)
    clods = (fractal_noise(hh, ww, rng, sigmas=(max(0.6, r / 8),), weights=(1.0,)) > 2.5) & (rho_p > 1.1) & (rho_p < ext)
    ej = np.maximum(ej, clods.astype(np.float32) * 0.7 * (rays > 0.25))
    pit = _smoothstep(0.95, 0.70, rho_p)                     # темна яма
    a_bowl = _smoothstep(1.12, 0.85, rho_p)
    alpha = np.maximum(a_bowl, ej * rng.uniform(0.7, 0.95))
    # кольори: світліший викид (підґрунтя) + темна волога/затінена яма
    ej_col = np.asarray(soil_rgb, np.float32) * rng.uniform(1.0, 1.25)
    pit_col = np.asarray(soil_rgb, np.float32) * rng.uniform(0.45, 0.68)
    color = ej_col[None, None, :] * (1 - pit[..., None]) + pit_col[None, None, :] * pit[..., None]
    dark = pit * rng.uniform(0.08, 0.25)
    if rng.random() < 0.3:                                    # кіптява навколо
        dark = np.maximum(dark, _smoothstep(0.2, 0.9, ej) * rng.uniform(0.15, 0.35))
    if old:                                                   # «стара» вирва: заросла, просіла, низький контраст
        height *= rng.uniform(0.3, 0.6)
        alpha = a_bowl * rng.uniform(0.25, 0.5)
        dark *= rng.uniform(0.2, 0.5)
    # мітка: чаша + вал + ЩІЛЬНИЙ близький викид (окремі грудки не розмічаються)
    ej_dense = cv2.GaussianBlur(np.where(clods, 0, ej).astype(np.float32), (0, 0), 1.2)
    label = (rho_p <= 1.15) | ((ej_dense > 0.6) & (rho_p < 1.8))
    return dict(box=(y0, y1, x0, x1), height=height.astype(np.float32), alpha=alpha.astype(np.float32),
                dark=dark.astype(np.float32), label=label, color=color.astype(np.float32))


# ------------------------------------------------------------------ траншеї/фортифікації
def _path_zigzag(p0, ang, n_seg, seg_m, gsd, rng):
    pts = [np.array(p0, np.float32)]
    sgn = 1
    for _ in range(n_seg):
        a = ang + sgn * np.deg2rad(rng.uniform(25, 55))
        L = rng.uniform(*seg_m) / gsd
        pts.append(pts[-1] + L * np.array([np.cos(a), np.sin(a)], np.float32))
        sgn *= -1
    return np.array(pts)


def _path_crenellated(p0, ang, n_seg, seg_m, gsd, rng):
    pts = [np.array(p0, np.float32)]
    fwd = np.array([np.cos(ang), np.sin(ang)], np.float32)
    side = np.array([-fwd[1], fwd[0]], np.float32)
    s = 1
    for i in range(n_seg):
        L = rng.uniform(*seg_m) / gsd
        pts.append(pts[-1] + fwd * L)
        if i < n_seg - 1:
            pts.append(pts[-1] + side * s * rng.uniform(1.5, 3.0) / gsd)
            s *= -1
    return np.array(pts)


def _path_wavy(p0, ang, length_m, gsd, rng):
    step = max(2.0, 1.5 / gsd)
    n = int(length_m / gsd / step)
    pts = [np.array(p0, np.float32)]
    a, da = ang, 0.0
    for _ in range(max(2, n)):
        da = 0.85 * da + rng.normal(0, 0.06)
        a += da
        pts.append(pts[-1] + step * np.array([np.cos(a), np.sin(a)], np.float32))
    return np.array(pts)


def _path_segmented(p0, ang, n_seg, seg_m, gsd, rng):
    pts = [np.array(p0, np.float32)]
    a = ang
    for _ in range(n_seg):
        a += np.deg2rad(rng.normal(0, 12))
        L = rng.uniform(*seg_m) / gsd
        pts.append(pts[-1] + L * np.array([np.cos(a), np.sin(a)], np.float32))
    return np.array(pts)


def _dist_to_polyline(shape, pts_local, thickness=1):
    canvas = np.zeros(shape, np.uint8)
    cv2.polylines(canvas, [np.round(pts_local).astype(np.int32).reshape(-1, 1, 2)], False, 255, thickness, cv2.LINE_8)
    return cv2.distanceTransform(255 - canvas, cv2.DIST_L2, 5).astype(np.float32)


def _offset_polyline(pts, off):
    out = []
    for i in range(len(pts)):
        a = pts[max(0, i - 1)]
        b = pts[min(len(pts) - 1, i + 1)]
        d = b - a
        n = np.array([-d[1], d[0]], np.float32) / (np.linalg.norm(d) + 1e-6)
        out.append(pts[i] + n * off)
    return np.array(out)


def trench_geometry(canvas, pts, rng, kind='trench', old=False):
    gsd = canvas.gsd
    if kind == 'ditch':                                       # протитанковий рів
        w_m, d_m = rng.uniform(3.0, 5.5), rng.uniform(1.8, 2.8)
        sp_w_m, sp_h_m = rng.uniform(3.0, 6.0), rng.uniform(0.8, 1.4)
        both = False
    else:                                                     # стрілецька/ходова траншея
        w_m, d_m = rng.uniform(0.8, 1.5), rng.uniform(1.1, 1.8)
        sp_w_m, sp_h_m = rng.uniform(1.0, 2.6), rng.uniform(0.25, 0.6)
        both = rng.random() < 0.55
    w, sp_w = max(1.2, w_m / gsd), max(1.5, sp_w_m / gsd)
    margin = int(np.ceil(w / 2 + sp_w * 2 + 6))
    x0 = max(0, int(np.floor(pts[:, 0].min())) - margin); x1 = min(canvas.w, int(np.ceil(pts[:, 0].max())) + margin + 1)
    y0 = max(0, int(np.floor(pts[:, 1].min())) - margin); y1 = min(canvas.h, int(np.ceil(pts[:, 1].max())) + margin + 1)
    if x1 - x0 < 4 or y1 - y0 < 4:
        return None
    loc = pts - np.array([x0, y0], np.float32)
    shape = (y1 - y0, x1 - x0)
    d = _dist_to_polyline(shape, loc)
    hh, ww = shape
    rough = fractal_noise(hh, ww, rng, sigmas=(0.8, 2.0, 5.0), weights=(0.3, 0.4, 0.3))
    wiggle = 1 + 0.12 * rough
    trench = _smoothstep(w / 2 * wiggle + 0.8, w / 2 * wiggle - 0.8, d)
    height = -d_m * trench
    side = 1 if rng.random() < 0.5 else -1
    d_front = _dist_to_polyline(shape, _offset_polyline(loc, side * (w / 2 + sp_w / 2 + 0.5)))
    berm = _smoothstep(sp_w / 2 * (1 + 0.25 * rough) + 1.0, 0, d_front) * (1 - trench)
    if both:
        d_back = _dist_to_polyline(shape, _offset_polyline(loc, -side * (w / 2 + sp_w / 3 + 0.5)))
        berm = np.maximum(berm, 0.6 * _smoothstep(sp_w / 3 * (1 + 0.25 * rough) + 1.0, 0, d_back) * (1 - trench))
    height += sp_h_m * berm * (1 + 0.2 * rough)
    patch = _smoothstep(-0.8, 0.5, fractal_noise(hh, ww, rng, sigmas=(0.8, 2.0, 6.0), weights=(0.35, 0.4, 0.25)) + 0.6)
    alpha = np.maximum(trench, berm * patch * rng.uniform(0.7, 1.0))
    dark = trench * rng.uniform(0.25, 0.5)
    if old:
        height *= rng.uniform(0.4, 0.7)
        alpha *= rng.uniform(0.25, 0.5)
        dark *= 0.4
    label = (d <= w / 2 + 1.0) | (cv2.GaussianBlur((berm * patch).astype(np.float32), (0, 0), 1.0) > 0.45)
    return dict(box=(y0, y1, x0, x1), height=height.astype(np.float32), alpha=alpha.astype(np.float32),
                dark=dark.astype(np.float32), label=label)


def dugout_geometry(canvas, cx, cy, rng, old=False):
    gsd = canvas.gsd
    side_m, d_m = rng.uniform(2.0, 4.0), rng.uniform(1.2, 2.0)
    s = side_m / gsd
    sp = rng.uniform(1.0, 2.0) / gsd
    R = int(np.ceil(s + 2 * sp + 4))
    x0, x1 = max(0, int(cx) - R), min(canvas.w, int(cx) + R + 1)
    y0, y1 = max(0, int(cy) - R), min(canvas.h, int(cy) + R + 1)
    if x1 - x0 < 4 or y1 - y0 < 4:
        return None
    yy, xx = np.mgrid[y0:y1, x0:x1].astype(np.float32)
    th = rng.uniform(0, np.pi / 2)
    u = (xx - cx) * np.cos(th) + (yy - cy) * np.sin(th)
    v = -(xx - cx) * np.sin(th) + (yy - cy) * np.cos(th)
    dbox = np.maximum(np.abs(u), np.abs(v))                   # «квадратна» відстань
    pit = _smoothstep(s / 2 + 0.8, s / 2 - 0.8, dbox)
    ring = _smoothstep(s / 2 + sp + 1, s / 2 + sp * 0.3, dbox) * (1 - pit)
    hh, ww = dbox.shape
    patch = _smoothstep(-0.8, 0.5, fractal_noise(hh, ww, rng, sigmas=(0.8, 2.0), weights=(0.5, 0.5)) + 0.6)
    height = -d_m * pit + rng.uniform(0.3, 0.6) * ring
    alpha = np.maximum(pit, ring * patch)
    dark = pit * rng.uniform(0.3, 0.6)
    if old:
        height *= 0.5; alpha *= 0.4; dark *= 0.4
    label = (dbox <= s / 2 + 1) | ((ring * patch) > 0.5)
    return dict(box=(y0, y1, x0, x1), height=height.astype(np.float32), alpha=alpha.astype(np.float32),
                dark=dark.astype(np.float32), label=label)


def dragon_teeth_geometry(canvas, p0, ang, rng):
    """«Зуби дракона» — ряди бетонних пірамід (світлі, з тінями)."""
    gsd = canvas.gsd
    size_m, h_m = rng.uniform(1.0, 1.5), rng.uniform(0.9, 1.3)
    spacing = rng.uniform(1.9, 2.6) / gsd
    rows, n = int(rng.integers(2, 5)), int(rng.integers(10, 35))
    fwd = np.array([np.cos(ang), np.sin(ang)], np.float32)
    side = np.array([-fwd[1], fwd[0]], np.float32)
    centers = [np.asarray(p0, np.float32) + fwd * spacing * (i + 0.5 * (r % 2)) + side * spacing * r * 0.9
               for r in range(rows) for i in range(n)]
    centers = np.array(centers)
    s = size_m / gsd
    margin = int(np.ceil(s + 4))
    x0 = max(0, int(centers[:, 0].min()) - margin); x1 = min(canvas.w, int(centers[:, 0].max()) + margin + 1)
    y0 = max(0, int(centers[:, 1].min()) - margin); y1 = min(canvas.h, int(centers[:, 1].max()) + margin + 1)
    if x1 - x0 < 4 or y1 - y0 < 4:
        return None
    yy, xx = np.mgrid[y0:y1, x0:x1].astype(np.float32)
    height = np.zeros(yy.shape, np.float32)
    for c in centers:
        u = np.abs(xx - c[0]); v = np.abs(yy - c[1])
        pyr = np.clip(1 - np.maximum(u, v) / (s / 2 + 1e-6), 0, 1)
        height = np.maximum(height, pyr * h_m)
    body = (height > 0.02).astype(np.float32)
    alpha = body
    label = cv2.dilate(body.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0
    return dict(box=(y0, y1, x0, x1), height=height, alpha=alpha, dark=np.zeros_like(alpha), label=label,
                color=np.array([175, 172, 165], np.float32))


# ------------------------------------------------------------------ сцена
def _place(canvas, geom, soil, label_map=None, lbl=None, aux=None, aux_val=None, protect=None):
    if geom is None:
        return 0
    y0, y1, x0, x1 = geom['box']
    canvas.add(y0, y1, x0, x1, geom['height'], geom['alpha'], geom.get('color', soil), geom['dark'])
    lab = geom['label']
    if label_map is not None:
        region = label_map[y0:y1, x0:x1]
        ok = lab.copy()
        if protect is not None:
            ok &= ~protect[y0:y1, x0:x1]
        region[ok] = lbl
        return int(ok.sum())
    if aux is not None:
        aux[y0:y1, x0:x1][lab] = aux_val
    return 0


def _trench_path(canvas, openness, rng, regime, gsd):
    for _ in range(12):
        p0 = sample_point(openness, rng, margin=10)
        if p0 is None:
            return None, None
        ang = rng.uniform(0, 2 * np.pi)
        r = rng.random()
        if r < 0.12 * (regime['p_ditch'] / 0.12):
            kind = 'ditch'
            pts = _path_segmented(p0, ang, int(rng.integers(2, 5)), (25, 60), gsd, rng)
        else:
            kind = 'trench'
            style = rng.choice(regime['trench_styles'])
            if style == 'zigzag':
                pts = _path_zigzag(p0, ang, int(rng.integers(4, 14)), (3, 8), gsd, rng)
            elif style == 'crenellated':
                pts = _path_crenellated(p0, ang, int(rng.integers(3, 9)), (4, 9), gsd, rng)
            elif style == 'wavy':
                pts = _path_wavy(p0, ang, rng.uniform(25, 110), gsd, rng)
            else:
                pts = _path_segmented(p0, ang, int(rng.integers(2, 7)), (6, 20), gsd, rng)
        pts[:, 0] = np.clip(pts[:, 0], 2, canvas.w - 3)
        pts[:, 1] = np.clip(pts[:, 1], 2, canvas.h - 3)
        # перевірка: траса має пролягати переважно відкритою місцевістю
        dense = np.concatenate([np.linspace(pts[i], pts[i + 1], max(2, int(np.linalg.norm(pts[i + 1] - pts[i]) // 2)))
                                for i in range(len(pts) - 1)])
        xi = np.clip(dense[:, 0].astype(int), 0, canvas.w - 1); yi = np.clip(dense[:, 1].astype(int), 0, canvas.h - 1)
        if (openness[yi, xi] > 0.12).mean() >= 0.85 and np.ptp(pts[:, 0]) + np.ptp(pts[:, 1]) > 8:
            return pts, kind
    return None, None


def _visible_label(label, before, after, classes=(LBL_CRATER, LBL_FORT), thr=9.0, min_px=6):
    """Мітка лише там, де зміна реально видима: |after-before| у RGB > thr (з невеликим допуском).
    Об'єкти, які після рендеру майже не видно, повністю знімаються з розмітки."""
    diff = np.sqrt(((after.astype(np.float32) - before.astype(np.float32)) ** 2).sum(2))
    vis = cv2.GaussianBlur(diff, (0, 0), 1.0) > thr
    vis = cv2.dilate(vis.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0
    out = label.copy()
    for c in classes:
        m = label == c
        if not m.any():
            continue
        keep = m & vis
        keep = cv2.morphologyEx(keep.astype(np.uint8), cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8)).astype(bool) & m
        n, comp, stats, _ = cv2.connectedComponentsWithStats(keep.astype(np.uint8), connectivity=8)
        small = np.isin(comp, np.where(stats[:, cv2.CC_STAT_AREA] < min_px)[0])
        keep &= ~small
        out[m & ~keep] = LBL_BG
    return out


def synthesize_pair(pre, post, gsd, rng, regime='id', base_label=None, exclusion=None, sun=None,
                    p_crater=0.75, p_fort=0.6, p_old=0.45, p_empty=0.12):
    """Повертає (pre_out, post_out, label, aux, info).
    label: 0 фон / 2 нова вирва / 3 свіжа фортифікація (+ збережені мітки бази: 1/4/254/255)
    aux:   1 «стара» вирва, 2 «стара» фортифікація (є на ОБОХ знімках, мітка 0) — для shortcut-тесту."""
    R = SYN_REGIMES[regime]
    palette = SOIL_PALETTES[R['palette']]
    h, w = post.shape[:2]
    label = np.zeros((h, w), np.uint8) if base_label is None else base_label.copy()
    protect = label != 0                                        # не перезаписуємо мітки бази
    aux = np.zeros((h, w), np.uint8)
    excl = protect.copy() if exclusion is None else (protect | exclusion)
    st_post, st_pre = scene_stats(post), scene_stats(pre)
    nat_post, any_post = ground_maps(post, excl)
    az = rng.uniform(0, 360)
    el_post = rng.uniform(*R['sun_elev']); el_pre = rng.uniform(*R['sun_elev'])
    if sun is not None and regime == 'id':          # реальне сонце знімка «після» (метадані xBD)
        az, el_post = float(sun[0]) % 360, float(np.clip(sun[1], 15, 75))
    az_pre = (az + rng.normal(0, 25)) % 360
    pal_name = rng.choice(list(palette))
    cv_post = Canvas(post, gsd, az, el_post, rng, palette[pal_name], st_post['lum'])
    cv_pre = Canvas(pre, gsd, az_pre, el_pre, rng, palette[pal_name], st_pre['lum'])
    info = {'n_crater': 0, 'n_fort': 0, 'n_old': 0, 'palette': str(pal_name), 'regime': regime}

    def soil_color(cv):
        name = rng.choice(list(palette))
        ref = np.array(palette[pal_name], np.float32)
        scale = cv.soil / (ref + 1e-6)
        col = np.array(palette[name], np.float32) * scale * rng.uniform(0.92, 1.08, 3)
        grey = col.mean()
        return grey + (col - grey) * rng.uniform(0.55, 0.9)        # приглушена насиченість

    def crater_diam():
        mix = R['crater_diam_mix']
        k = rng.choice(len(mix), p=np.array([m[2] for m in mix]) / sum(m[2] for m in mix))
        lo, hi, _ = mix[k]
        return float(np.exp(rng.uniform(np.log(lo), np.log(hi))))

    # --- «старі» об'єкти на ОБОХ знімках (мітка 0, aux) — однакова геометрія, різний рендер
    if rng.random() < p_old:
        seed_geo = int(rng.integers(1 << 30))
        for _ in range(int(rng.integers(1, 6))):
            p = sample_point(any_post, rng)
            if p is None:
                break
            d = crater_diam()
            sc = soil_color(cv_post)
            g_post = crater_geometry(cv_post, p[0], p[1], d, np.random.default_rng(seed_geo), sc, old=True)
            g_pre = crater_geometry(cv_pre, p[0], p[1], d, np.random.default_rng(seed_geo), sc, old=True)
            seed_geo += 1
            _place(cv_post, g_post, sc, aux=aux, aux_val=1)
            _place(cv_pre, g_pre, sc)
            info['n_old'] += 1
        if rng.random() < 0.4:
            pts, kind = _trench_path(cv_post, nat_post, rng, R, gsd)
            if pts is not None:
                g_post = trench_geometry(cv_post, pts, np.random.default_rng(seed_geo), kind, old=True)
                g_pre = trench_geometry(cv_pre, pts, np.random.default_rng(seed_geo), kind, old=True)
                sc = soil_color(cv_post)
                _place(cv_post, g_post, sc, aux=aux, aux_val=2)
                _place(cv_pre, g_pre, sc)
                info['n_old'] += 1
    old_mask = aux > 0
    free = ~cv2.dilate(old_mask.astype(np.uint8), np.ones((9, 9), np.uint8)).astype(bool)
    nat_new, any_new = nat_post * free, any_post * free

    empty = rng.random() < p_empty
    if not empty:
        want_crater = rng.random() < p_crater
        want_fort = rng.random() < p_fort
        if not (want_crater or want_fort):
            want_crater = rng.random() < 0.5
            want_fort = not want_crater
        if want_crater:
            for _ in range(int(rng.integers(1, 4))):
                c = sample_point(any_new, rng)
                if c is None:
                    break
                spread = rng.uniform(4, 35) / gsd
                for _ in range(int(rng.integers(1, 20))):
                    if rng.random() < 0.15:                      # частина — будь-де (дороги, подвір'я)
                        q = (rng.uniform(10, w - 10), rng.uniform(10, h - 10))
                    else:
                        q = (c[0] + rng.normal(0, spread), c[1] + rng.normal(0, spread))
                    if not (4 <= q[0] < w - 4 and 4 <= q[1] < h - 4):
                        continue
                    # приймаємо точку з імовірністю ~ «природності» місця (дахи/виключені зони — майже ніколи)
                    if rng.random() > min(1.0, any_new[int(q[1]), int(q[0])] / 0.35):
                        continue
                    d = crater_diam()
                    if d / 2 / gsd < 1.2:                        # надто мала для розмітки
                        continue
                    sc = soil_color(cv_post)
                    g = crater_geometry(cv_post, q[0], q[1], d, rng, sc)
                    if _place(cv_post, g, sc, label_map=label, lbl=LBL_CRATER, protect=protect) > 0:
                        info['n_crater'] += 1
        if want_fort:
            for _ in range(int(rng.integers(1, 4))):
                pts, kind = _trench_path(cv_post, nat_new, rng, R, gsd)
                if pts is None:
                    break
                g = trench_geometry(cv_post, pts, rng, kind)
                if _place(cv_post, g, soil_color(cv_post), label_map=label, lbl=LBL_FORT, protect=protect) > 0:
                    info['n_fort'] += 1
                if kind == 'trench' and rng.random() < 0.5:     # бліндажі/позиції біля траншеї
                    for _ in range(int(rng.integers(1, 4))):
                        i = int(rng.integers(0, len(pts)))
                        off = rng.normal(0, 4 / gsd, 2)
                        g = dugout_geometry(cv_post, pts[i, 0] + off[0], pts[i, 1] + off[1], rng)
                        _place(cv_post, g, soil_color(cv_post), label_map=label, lbl=LBL_FORT, protect=protect)
            if rng.random() < R['p_dragon']:
                p = sample_point(nat_new, rng)
                if p is not None:
                    g = dragon_teeth_geometry(cv_post, p, rng.uniform(0, 2 * np.pi), rng)
                    if _place(cv_post, g, None, label_map=label, lbl=LBL_FORT, protect=protect) > 0:
                        info['n_fort'] += 1
    blur = rng.uniform(0.35, 0.8)
    post_out = cv_post.render(st_post['noise'], blur)
    pre_out = cv_pre.render(st_pre['noise'], blur) if info['n_old'] else pre.copy()
    label = _visible_label(label, post, post_out)
    info['px_crater'] = int((label == LBL_CRATER).sum())
    info['px_fort'] = int((label == LBL_FORT).sum())
    return pre_out, post_out, label, aux, info

In [ ]:
# ==============================================================================
# 4.2 Наочна перевірка генератора на реальних train-основах (до генерації всього набору)
# ==============================================================================
def _demo_bases(n=3):
    parts = [d for d in (fot_df, xbd_df) if d is not None and len(d)]
    if not parts:
        return pd.DataFrame()
    real = pd.concat(parts, ignore_index=True)
    real = real[(real['split'] == 'train') & (real['chg_px'] == 0)]
    out = []
    for src in ('FOTBCD', 'xBD'):
        sub = real[real['source'] == src]
        if len(sub):
            out.append(sub.sample(min(len(sub), n if src == 'FOTBCD' else 1), random_state=SEED))
    return pd.concat(out) if out else pd.DataFrame()


def _crop_around(mask, size):
    ys, xs = np.where(mask)
    if len(ys) == 0:
        return 0, 0
    i = len(ys) // 2
    cy, cx = int(ys[i]), int(xs[i])
    return max(0, min(mask.shape[0] - size, cy - size // 2)), max(0, min(mask.shape[1] - size, cx - size // 2))


demo = _demo_bases()
if len(demo):
    fig, axes = plt.subplots(len(demo), 5, figsize=(22, 4.6 * len(demo)), squeeze=False)
    for i, r in enumerate(demo.itertuples()):
        pre, post = read_rgb(r.pre), read_rgb(r.post)
        lab0 = read_gray(r.label)
        fp = read_gray(r.fp) > 0 if r.fp else None
        sun = (r.sun_az, r.sun_el) if np.isfinite(r.sun_az) and np.isfinite(r.sun_el) else None
        a, b, lab, aux, info = synthesize_pair(pre, post, float(r.gsd), np.random.default_rng(1000 + i), 'id', lab0, fp, sun,
                                               p_old=0.9, p_empty=0.0)
        sz = 160 if r.source == 'FOTBCD' else 256
        y0, x0 = _crop_around((lab == LBL_CRATER) | (lab == LBL_FORT), sz)
        panels = [(a, 'До (з «старими» об\'єктами)'), (b, 'Після (синтетика)'), (label_overlay(b, lab), 'Мітка + aux (блакитне = «старе», мітка 0)'),
                  (b[y0:y0 + sz, x0:x0 + sz], f'Збільшено «після» ({sz}px)'), (a[y0:y0 + sz, x0:x0 + sz], 'Збільшено «до»')]
        ov = panels[2][0].copy()
        ov[aux > 0] = (ov[aux > 0] * 0.5 + np.array([0, 220, 255]) * 0.5).astype(np.uint8)
        panels[2] = (ov, panels[2][1])
        for j, (im, t) in enumerate(panels):
            axes[i, j].imshow(im, interpolation='nearest' if j >= 3 else None)
            axes[i, j].set_title(t if j else f'{r.source} | GSD {float(r.gsd):.2f} м | {t}', fontsize=9)
            axes[i, j].axis('off')
        axes[i, 4].text(2, sz - 6, f"вирв: {info['n_crater']}, фортиф.: {info['n_fort']}, старих: {info['n_old']}, палітра: {info['palette']}",
                        color='w', fontsize=8, backgroundcolor='k')
    fig.legend(handles=legend_handles(), loc='lower center', ncol=4, fontsize=9)
    fig.suptitle('Генератор синтетики на реальних train-основах', fontsize=13)
    plt.tight_layout(rect=(0, 0.03, 1, 0.97)); save_fig(fig, 'synthetic_generator_demo'); plt.show()
else:
    print('Немає реальних основ для демонстрації (FOTBCD/xBD ще не підготовлені, розділ 3).')

In [ ]:
# ==============================================================================
# 4.3 Генерація синтетичних наборів: train / val / test (ID) / test_shift
# ==============================================================================
import multiprocessing as mp

SYN_SPLIT_CODE = {'train': 1, 'val': 2, 'test': 3, 'test_shift': 4}


def _syn_worker(job):
    rng = np.random.default_rng([SEED, SYN_SPLIT_CODE[job['split_key']], job['k']])
    pre, post = read_rgb(job['pre']), read_rgb(job['post'])
    lab0 = read_gray(job['label'])
    excl = (read_gray(job['fp']) > 0) if job['fp'] else None
    sun = (job['sun_az'], job['sun_el']) if np.isfinite(job['sun_az']) and np.isfinite(job['sun_el']) else None
    a, b, lab, aux, info = synthesize_pair(pre, post, float(job['gsd']), rng, job['regime'], lab0, excl, sun)
    out = DATA / 'SYN' / job['split_key']
    uid = job['uid']
    qp, lp = out / f'{uid}_post.png', out / f'{uid}_label.png'
    write_rgb(qp, b); write_gray(lp, lab)
    if info['n_old']:
        pp = out / f'{uid}_pre.png'; write_rgb(pp, a)
    else:
        pp = job['pre']                                   # «до» не змінювалось — посилання на основу
    ap = ''
    if aux.any():
        ap = str(out / f'{uid}_aux.png'); write_gray(ap, aux)
    return dict(uid=uid, source='SYN', subset='SYN-SHIFT' if job['regime'] == 'shift' else 'SYN-ID', split=job['split'],
                pre=str(pp), post=str(qp), label=str(lp), aux=ap, fp=job['fp'], gsd=job['gsd'], group=job['base_group'],
                base_uid=job['base_uid'], h=lab.shape[0], w=lab.shape[1], sun_az=job['sun_az'], sun_el=job['sun_el'],
                chg_px=int(((lab >= 1) & (lab <= 4)).sum()), n_destroyed_px=int((lab == LBL_DESTROYED).sum()),
                n_new_px=int((lab == LBL_NEWBLD).sum()), base_source=job['base_source'],
                n_crater=info['n_crater'], n_fort=info['n_fort'], n_old=info['n_old'], palette=info['palette'],
                px_crater=info['px_crater'], px_fort=info['px_fort'])


def build_syn_jobs(real):
    jobs = []
    for split_key, n in SYN_COUNTS.items():
        split = 'test' if split_key == 'test_shift' else split_key
        regime = 'shift' if split_key == 'test_shift' else 'id'
        fot = real[(real['source'] == 'FOTBCD') & (real['split'] == split)]
        xbd = real[(real['source'] == 'xBD') & (real['split'] == split)]
        n_x = int(round(n * SYN_XBD_FRACTION)) if len(xbd) else 0
        k = 0
        for src_df, cnt in ((fot, n - n_x), (xbd, n_x)):
            if cnt <= 0 or not len(src_df):
                continue
            rs = np.random.default_rng([SEED, SYN_SPLIT_CODE[split_key], 7 if src_df is xbd else 3])
            neg = src_df[src_df['chg_px'] == 0].sample(frac=1, random_state=int(rs.integers(1 << 30)))
            pos = src_df[src_df['chg_px'] > 0].sample(frac=1, random_state=int(rs.integers(1 << 30)))
            i_neg = i_pos = 0
            for _ in range(cnt):
                use_neg = (rs.random() < SYN_P_NEG_BASE and len(neg)) or not len(pos)
                if use_neg:
                    b = neg.iloc[i_neg % len(neg)]; i_neg += 1
                else:
                    b = pos.iloc[i_pos % len(pos)]; i_pos += 1
                jobs.append(dict(uid=f'syn_{split_key}_{k:05d}', k=k, split_key=split_key, split=split, regime=regime,
                                 pre=b['pre'], post=b['post'], label=b['label'], fp=b['fp'], gsd=float(b['gsd']),
                                 sun_az=float(b['sun_az']), sun_el=float(b['sun_el']), base_uid=b['uid'],
                                 base_group=b['group'], base_source=b['source']))
                k += 1
    return jobs


syn_df = None
real_df = pd.concat([d for d in (fot_df, xbd_df) if d is not None and len(d)], ignore_index=True) \
    if any(d is not None and len(d) for d in (fot_df, xbd_df)) else pd.DataFrame()
syn_cfg = dict(counts=SYN_COUNTS, xbd_frac=SYN_XBD_FRACTION, p_neg=SYN_P_NEG_BASE, seed=SEED,
               bases=hashlib.sha1(','.join(sorted(real_df['uid'])).encode()).hexdigest() if len(real_df) else '',
               gen=hashlib.sha1(json.dumps([SOIL_PALETTES, SYN_REGIMES], default=str).encode()).hexdigest()[:12])
syn_df = part_ready('SYN', syn_cfg)
if syn_df is None and RUN_PREPARE and len(real_df):
    jobs = build_syn_jobs(real_df)
    shutil.rmtree(DATA / 'SYN', ignore_errors=True)
    print('Синтетичних зразків до генерації:', len(jobs), '| процесів:', SYN_WORKERS)
    with Timer('Генерація синтетики'):
        if SYN_WORKERS > 1:
            with mp.get_context('fork').Pool(SYN_WORKERS) as pool:
                rows = list(tqdm(pool.imap_unordered(_syn_worker, jobs, chunksize=4), total=len(jobs), desc='SYN'))
        else:
            rows = [_syn_worker(j) for j in tqdm(jobs, desc='SYN')]
    syn_df = pd.DataFrame(rows).sort_values('uid').reset_index(drop=True)
    syn_df = part_save('SYN', syn_cfg, syn_df)

if syn_df is not None and len(syn_df):
    display(syn_df.groupby(['subset', 'split', 'base_source']).agg(
        samples=('uid', 'count'), craters=('n_crater', 'sum'), fortifications=('n_fort', 'sum'),
        old_objects=('n_old', 'sum'), no_new_objects=('chg_px', lambda x: int((x == 0).sum()))))

In [ ]:
# ==============================================================================
# 4.4 Єдиний маніфест + кеш підготовлених даних на Drive
# ==============================================================================
parts = [d for d in (fot_df, xbd_df, ukr_df, syn_df) if d is not None and len(d)]
if not parts:
    raise RuntimeError('Немає жодного підготовленого джерела даних. Перевірте розділ 3.')
manifest = normalize_manifest(pd.concat(parts, ignore_index=True))
manifest.to_csv(DATA / 'manifest.csv', index=False)
MANIFEST_SHA = hashlib.sha1(manifest[['uid', 'subset', 'split', 'base_uid']].to_csv(index=False).encode()).hexdigest()[:16]
(DATA / 'manifest.sha1').write_text(MANIFEST_SHA)
SUBSETS_ORDER = [s for s in ('FOTBCD-Instances', 'FOTBCD-Binary', 'xBD-Global', 'xBD-Ukraine', 'SYN-ID', 'SYN-SHIFT')
                 if s in set(manifest['subset'])]
print('Маніфест:', len(manifest), 'пар | sha:', MANIFEST_SHA)
display(manifest.pivot_table(index='subset', columns='split', values='uid', aggfunc='count', fill_value=0)
        .reindex(SUBSETS_ORDER)[[c for c in ('train', 'val', 'test') if c in set(manifest['split'])]])

if CACHE_DATA_TO_DRIVE and RUN_PREPARE:
    side = DATA_CACHE_ZIP.with_suffix('.sha1')
    if DATA_CACHE_ZIP.exists() and side.exists() and side.read_text() == MANIFEST_SHA:
        print('Кеш на Drive актуальний:', DATA_CACHE_ZIP)
    else:
        size_gb = sum(p.stat().st_size for p in DATA.rglob('*') if p.is_file()) / 1e9
        fg = free_gb(DRIVE_ROOT)
        if np.isfinite(fg) and fg < size_gb * 1.15 + 1:
            print(f'⚠ На Drive вільно {fg:.1f} ГБ, потрібно ≈{size_gb:.1f} ГБ — кеш не зберігаємо.')
        else:
            with Timer(f'Кеш даних → Drive ({size_gb:.1f} ГБ)'):
                zip_folder_stored(DATA, DATA_CACHE_ZIP)
                side.write_text(MANIFEST_SHA)

## 5. Аудит даних

Перевіряємо **кожну** пару до навчання:

| Перевірка | Рівень |
|---|---|
| файли існують, розміри «до» = «після» = маска | CRITICAL |
| значення маски відповідають джерелу | CRITICAL |
| «до» і «після» побайтно однакові при наявності змін (помилка v3) | CRITICAL |
| однакова пара або однаковий знімок «до» в різних сплітах | CRITICAL |
| синтетика побудована на основі з іншого спліту | CRITICAL |
| спільні просторові блоки xBD у train і val/test | CRITICAL |
| спільні департаменти FOTBCD у train і val/test | WARNING (офіційний спліт їх розділяє) |
| майже-дублікати реальних сцен між сплітами (перцептивний хеш, ≤3 біти з 64) | WARNING + галерея для ручної перевірки |
| замалі val/test підвибірки (< 20 пар) | WARNING |

Відсутність точних дублікатів **не доводить** географічної незалежності — тому додатково розділяємо за департаментами/блоками координат і показуємо майже-дублікати.

In [ ]:
# ==============================================================================
# 5.1 Аудит даних (CPU). Критичні проблеми блокують навчання.
# ==============================================================================
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

ALLOWED_LABELS = {'FOTBCD-Instances': {0, 1, 4, 255}, 'FOTBCD-Binary': {0, 254},
                  'xBD-Global': {0, 1, 255}, 'xBD-Ukraine': {0, 1, 255},
                  'SYN-ID': {0, 1, 2, 3, 4, 254, 255}, 'SYN-SHIFT': {0, 1, 2, 3, 4, 254, 255}}


def _audit_one(r):
    out = dict(uid=r['uid'], problems=[])
    try:
        paths = {k: r[k] for k in ('pre', 'post', 'label')}
        missing = [k for k, p in paths.items() if not Path(p).exists()]
        if missing:
            out['problems'].append(('CRITICAL', 'MISSING_FILE', ','.join(missing)))
            return out
        sizes = {}
        for k in ('pre', 'post'):
            with Image.open(paths[k]) as im:
                sizes[k] = (im.size[1], im.size[0])
        lab = read_gray(paths['label'])
        sizes['label'] = lab.shape[:2]
        if len(set(sizes.values())) != 1:
            out['problems'].append(('CRITICAL', 'SHAPE_MISMATCH', str(sizes)))
        vals = set(np.unique(lab).tolist())
        bad = vals - ALLOWED_LABELS.get(r['subset'], set(range(256)))
        if bad:
            out['problems'].append(('CRITICAL', 'UNEXPECTED_LABEL_VALUES', str(sorted(bad))))
        out['sha_pre'], out['sha_post'] = file_sha1(paths['pre']), file_sha1(paths['post'])
        chg = (lab >= 1) & (lab <= 4) | (lab == LBL_UNKNOWN)
        if out['sha_pre'] == out['sha_post'] and chg.any():
            out['problems'].append(('CRITICAL', 'IDENTICAL_PRE_POST_WITH_CHANGE', ''))
        cnt = np.bincount(lab.ravel(), minlength=256)
        for c in (0, 1, 2, 3, 4, 254, 255):
            out[f'px_{c}'] = int(cnt[c])
        if r['source'] != 'SYN':                    # перцептивний хеш лише для реальних сцен (hex, без втрати точності)
            pre_img = read_rgb(paths['pre'])
            out['dhash'] = format(int(dhash64(pre_img)), '016x')
            out['pre_std'] = float(pre_img.std())
        if r['aux']:
            aux = read_gray(r['aux'])
            out['aux_px'] = int((aux > 0).sum())
    except Exception as e:
        out['problems'].append(('CRITICAL', 'READ_ERROR', repr(e)[:200]))
    return out


def popcount64(x):
    x = x.astype(np.uint64)
    return np.unpackbits(x.view(np.uint8).reshape(-1, 8), axis=1).sum(1)


def near_duplicates_across_splits(df, max_hamming=3):
    """Пари реальних сцен з РІЗНИХ сплітів із майже однаковим dHash (≤ max_hamming біт)."""
    if 'dhash' not in df:
        return pd.DataFrame()
    d = df[df['dhash'].fillna('').astype(str).str.len() == 16].copy()
    if not len(d):
        return pd.DataFrame()
    h = np.array([int(x, 16) for x in d['dhash']], dtype=np.uint64)
    sp = d['split'].to_numpy(); uid = d['uid'].to_numpy()
    out = []
    for i in range(len(d)):
        other = sp != sp[i]
        other[:i + 1] = False
        if not other.any():
            continue
        ham = popcount64(np.bitwise_xor(h[other], h[i]))
        for j in np.where(ham <= max_hamming)[0]:
            jj = np.where(other)[0][j]
            out.append(dict(uid_a=uid[i], split_a=sp[i], uid_b=uid[jj], split_b=sp[jj], hamming=int(ham[j])))
    return pd.DataFrame(out)


AUDIT_OK = True
audit_df = None
if RUN_AUDIT:
    with Timer('Аудит файлів'):
        with ThreadPoolExecutor(4) as ex:
            res = list(tqdm(ex.map(_audit_one, manifest.to_dict('records')), total=len(manifest), desc='audit'))
    probs = [(r['uid'],) + p for r in res for p in r['problems']]
    audit_df = manifest.merge(pd.DataFrame([{k: v for k, v in r.items() if k != 'problems'} for r in res]),
                              on='uid', how='left', suffixes=('', '_a'))
    issues = pd.DataFrame(probs, columns=['uid', 'severity', 'issue', 'detail'])

    # --- точні дублікати між сплітами (однакова пара / однаковий «до»)
    a = audit_df.dropna(subset=['sha_pre']).copy()
    # «порожні» кадри (напр. чорні тайли без даних) побайтно однакові — це не витік сцени → WARNING
    flat = set(a.loc[a.get('pre_std', pd.Series(np.nan, index=a.index)).fillna(99) < 3, 'sha_pre'])
    pair_key = a['sha_pre'] + ':' + a['sha_post']
    dup_pair = a.groupby(pair_key)['split'].nunique()
    for k in dup_pair[dup_pair > 1].index:
        sev = 'WARNING' if k.split(':')[0] in flat else 'CRITICAL'
        issues.loc[len(issues)] = ['*', sev, 'EXACT_PAIR_IN_SEVERAL_SPLITS', k[:20]]
    pre_split = a.groupby('sha_pre')['split'].nunique()
    for k in pre_split[pre_split > 1].index:
        sev = 'WARNING' if k in flat else 'CRITICAL'
        issues.loc[len(issues)] = ['*', sev, 'SAME_PRE_IMAGE_IN_SEVERAL_SPLITS', k[:20]]

    # --- походження синтетики: основа строго з того ж спліту
    syn = audit_df[audit_df['source'] == 'SYN']
    if len(syn):
        base_split = manifest.set_index('uid')['split']
        bad_lineage = syn[syn['base_uid'].map(base_split) != syn['split']]
        for u in bad_lineage['uid']:
            issues.loc[len(issues)] = [u, 'CRITICAL', 'SYNTHETIC_BASE_FROM_OTHER_SPLIT', '']

    # --- групи (департаменти FOTBCD / просторові блоки xBD) у train і val/test одночасно
    for src in ('FOTBCD', 'xBD'):
        s = audit_df[audit_df['source'] == src]
        if not len(s):
            continue
        tr = set(s.loc[s['split'] == 'train', 'group']); ev = set(s.loc[s['split'] != 'train', 'group'])
        inter = (tr & ev) - {'train', 'val', 'test', ''}
        if inter:
            sev = 'CRITICAL' if src == 'xBD' else 'WARNING'
            issues.loc[len(issues)] = ['*', sev, f'{src}_GROUP_OVERLAP_TRAIN_EVAL', str(sorted(inter))[:200]]
        print(f'{src}: груп у train {len(tr)}, у val/test {len(ev)}, спільних: {len(inter)}')

    # --- майже-дублікати реальних сцен між сплітами (перцептивний хеш)
    with Timer('Пошук майже-дублікатів (dHash)'):
        near = near_duplicates_across_splits(audit_df)
    if len(near):
        issues.loc[len(issues)] = ['*', 'WARNING', 'NEAR_DUPLICATE_SCENES_ACROSS_SPLITS', f'{len(near)} пар (див. галерею)']

    # --- зведення
    lab_cols = [f'px_{c}' for c in (0, 1, 2, 3, 4, 254, 255)]
    stats = audit_df.groupby(['subset', 'split'])[lab_cols].sum()
    tot = stats.sum(axis=1)
    frac = (stats.div(tot, axis=0) * 100).round(3)
    frac.columns = ['% фон', '% зруйн.', '% вирва', '% фортиф.', '% нова забуд.', '% невідомий тип', '% ігнор']
    pairs = audit_df.groupby(['subset', 'split']).agg(pairs=('uid', 'count'),
                                                      empty_label=('chg_px', lambda x: int((x == 0).sum())))
    print('\nСклад вибірок і частки пікселів класів:')
    display(pairs.join(frac))
    for (sub, sp), n in pairs['pairs'].items():
        if sp in ('val', 'test') and n < 20:
            issues.loc[len(issues)] = ['*', 'WARNING', 'SMALL_EVAL_SUBSET', f'{sub}/{sp}: {n} пар — метрики нестабільні']
    n_crit = int((issues['severity'] == 'CRITICAL').sum()); n_warn = int((issues['severity'] == 'WARNING').sum())
    print(f'\nПроблем: CRITICAL = {n_crit}, WARNING = {n_warn}')
    if len(issues):
        display(issues.groupby(['severity', 'issue']).size().rename('кількість').to_frame())
    AUDIT_OK = n_crit == 0
    (RUN_DIR / 'audit').mkdir(exist_ok=True)
    issues.to_csv(RUN_DIR / 'audit' / 'issues.csv', index=False)
    audit_df.drop(columns=['dhash'], errors='ignore').to_csv(RUN_DIR / 'audit' / 'audited_manifest.csv', index=False)
    near.to_csv(RUN_DIR / 'audit' / 'near_duplicates.csv', index=False)
    frac.to_csv(RUN_DIR / 'audit' / 'class_fractions.csv')
    print('✅ Аудит пройдено — навчання дозволено.' if AUDIT_OK else '❌ Є критичні проблеми — навчання ЗАБЛОКОВАНО (див. issues.csv).')

In [ ]:
# ==============================================================================
# 5.2 Галереї аудиту: приклади кожного джерела, частки класів, майже-дублікати
# ==============================================================================
def show_examples(df, n=2, title='', seed=SEED, name=None):
    subsets = [s for s in SUBSETS_ORDER if s in set(df['subset'])]
    rows = []
    for s in subsets:
        sub = df[df['subset'] == s]
        pos = sub[sub['chg_px'] > 0]
        pick = (pos if len(pos) >= n else sub).sample(min(n, len(sub)), random_state=seed)
        rows += list(pick.itertuples())
    if not rows:
        return
    fig, axes = plt.subplots(len(rows), 3, figsize=(14, 4.4 * len(rows)), squeeze=False)
    for i, r in enumerate(rows):
        pre, post, lab = read_rgb(r.pre), read_rgb(r.post), read_gray(r.label)
        axes[i, 0].imshow(pre); axes[i, 0].set_title(f'{r.subset} | {r.split} | До\n{r.uid}', fontsize=8)
        axes[i, 1].imshow(post); axes[i, 1].set_title('Після', fontsize=8)
        ov = label_overlay(post, lab)
        if getattr(r, 'aux', '') and str(r.aux):
            aux = read_gray(r.aux)
            ov[aux > 0] = (ov[aux > 0] * 0.55 + np.array([0, 220, 255]) * 0.45).astype(np.uint8)
        axes[i, 2].imshow(ov); axes[i, 2].set_title(f'Розмітка (змін: {int(r.chg_px)} px)', fontsize=8)
        for a in axes[i]:
            a.axis('off')
    fig.legend(handles=legend_handles(True, True), loc='lower center', ncol=3, fontsize=9)
    fig.suptitle(title, fontsize=13)
    plt.tight_layout(rect=(0, 0.03, 1, 0.98))
    if name:
        save_fig(fig, name)
    plt.show()


if RUN_AUDIT and audit_df is not None:
    show_examples(manifest[manifest['split'] == 'train'], n=2, title='Train: приклади кожного джерела', name='audit_train_examples')
    show_examples(manifest[manifest['split'] == 'test'], n=1, title='Test: приклади кожного джерела', name='audit_test_examples')

    # частки пікселів класів (лог-шкала)
    lab_cols = {1: 'px_1', 2: 'px_2', 3: 'px_3', 4: 'px_4'}
    agg = audit_df.groupby('subset')[list(lab_cols.values()) + ['px_0']].sum().reindex(SUBSETS_ORDER)
    fig, ax = plt.subplots(figsize=(11, 4))
    width = 0.2
    for k, (c, col) in enumerate(lab_cols.items()):
        ax.bar(np.arange(len(agg)) + (k - 1.5) * width, agg[col] / (agg.sum(axis=1) + 1) * 100, width,
               color=np.array(CLASS_COLORS[c]) / 255, label=CLASS_NAMES[c])
    ax.set_yscale('log'); ax.set_ylabel('% пікселів'); ax.set_xticks(range(len(agg))); ax.set_xticklabels(agg.index)
    ax.set_title('Частка пікселів класів змін за джерелами (усі спліти)'); ax.legend(fontsize=8); ax.grid(alpha=0.3, axis='y')
    save_fig(fig, 'audit_class_fractions'); plt.show()

    if len(near):
        print(f'Майже-дублікати між сплітами: {len(near)} пар. Перші приклади (перевірте вручну):')
        idx = manifest.set_index('uid')
        k = min(4, len(near))
        fig, axes = plt.subplots(k, 2, figsize=(9, 4.4 * k), squeeze=False)
        for i, r in enumerate(near.sort_values('hamming').head(k).itertuples()):
            for j, (u, sp) in enumerate(((r.uid_a, r.split_a), (r.uid_b, r.split_b))):
                axes[i, j].imshow(read_rgb(idx.loc[u, 'pre'])); axes[i, j].set_title(f'{sp} | {u} | ham={r.hamming}', fontsize=8)
                axes[i, j].axis('off')
        plt.tight_layout(); save_fig(fig, 'audit_near_duplicates'); plt.show()
    else:
        print('Майже-дублікатів реальних сцен між сплітами не знайдено.')

## 6. Тренувальні дані: банк тайлів 256×256

Усі train-пари один раз розрізаються на тайли й пишуться в один memmap-файл на локальному диску; кожен тайл — неперервні 458 КБ, тож 2 CPU Colab встигають годувати GPU (декодування PNG на льоту було б вузьким місцем). Тайли без змін з великих кадрів частково відкидаються (`P_KEEP_NEG_TILE`).

Батч збирається з джерел у пропорції `SOURCE_WEIGHTS`, 60% тайлів — зі змінами. Аугментації: повороти/віддзеркалення, zoom-in, **незалежна** фотометрія для «до» і «після» (різні умови зйомки), зсув «до» на ±2 px (неідеальне вирівнювання). Обмін «до»↔«після» **заборонений** — він змінює зміст класів (знесення ↔ будівництво).

In [ ]:
# ==============================================================================
# 6.1 Банк тренувальних тайлів (memmap) + аугментації + Dataset
# ==============================================================================
SRC_ID = {'FOTBCD': 0, 'xBD': 1, 'SYN': 2}
P_KEEP_NEG_TILE = {'FOTBCD': 0.6, 'xBD': 0.3, 'SYN': 0.4}   # частка тайлів без змін, що зберігаються


def _chg_mask(lab):
    return ((lab >= 1) & (lab <= 4)) | (lab == LBL_UNKNOWN)


def _tiles_of_row(r):
    """Розрізає пару на тайли CROP×CROP; повертає список (tile[CROP,CROP,7], has_change, class_presence)."""
    pre, post, lab = read_rgb(r['pre']), read_rgb(r['post']), read_gray(r['label'])
    h, w = lab.shape
    if h < CROP or w < CROP:
        ph, pw = max(0, CROP - h), max(0, CROP - w)
        pre = cv2.copyMakeBorder(pre, 0, ph, 0, pw, cv2.BORDER_REFLECT_101)
        post = cv2.copyMakeBorder(post, 0, ph, 0, pw, cv2.BORDER_REFLECT_101)
        lab = cv2.copyMakeBorder(lab, 0, ph, 0, pw, cv2.BORDER_CONSTANT, value=LBL_IGNORE)
        h, w = lab.shape
    rng = np.random.default_rng([SEED, int(hashlib.sha1(r['uid'].encode()).hexdigest()[:8], 16)])
    ys, xs = window_origins(h, CROP, CROP), window_origins(w, CROP, CROP)
    many = len(ys) * len(xs) > 1
    out = []
    for y in ys:
        for x in xs:
            l = lab[y:y + CROP, x:x + CROP]
            has = bool(_chg_mask(l).any())
            if many and not has and rng.random() > P_KEEP_NEG_TILE[r['source']]:
                continue
            if (l == LBL_IGNORE).mean() > 0.9:
                continue
            t = np.concatenate([pre[y:y + CROP, x:x + CROP], post[y:y + CROP, x:x + CROP], l[..., None]], axis=2)
            pres = np.array([(l == c).any() for c in (1, 2, 3, 4, LBL_UNKNOWN)], bool)
            out.append((t, has, pres))
    return out


def build_bank(train_rows, bank_path):
    """Один послідовний запис сирих тайлів (uint8) у файл; форма зберігається в .meta.npz."""
    meta_path = bank_path.with_suffix('.meta.npz')
    if bank_path.exists() and meta_path.exists():
        print('Банк уже існує:', bank_path)
        return
    rows = train_rows.to_dict('records')
    tmp = bank_path.with_suffix('.partial')
    meta = []
    with open(tmp, 'wb') as f, ThreadPoolExecutor(4) as ex:
        for r, tiles in tqdm(zip(rows, ex.map(_tiles_of_row, rows)), total=len(rows), desc='Банк тайлів'):
            for t, has, pres in tiles:
                f.write(np.ascontiguousarray(t, dtype=np.uint8).tobytes())
                meta.append((SRC_ID[r['source']], has, pres, r['uid']))
    os.replace(tmp, bank_path)
    n = len(meta)
    np.savez(meta_path, src=np.array([m[0] for m in meta], np.int8), has=np.array([m[1] for m in meta], bool),
             pres=np.stack([m[2] for m in meta]), uid=np.array([m[3] for m in meta]),
             shape=np.array([n, CROP, CROP, 7], np.int64))
    print(f'Банк: {n} тайлів, {bank_path.stat().st_size / 1e9:.1f} ГБ')


def open_bank(bank_path, meta):
    return np.memmap(bank_path, dtype=np.uint8, mode='r', shape=tuple(int(v) for v in meta['shape']))


def _photometric(img, rng):
    x = img.astype(np.float32)
    if rng.random() < 0.8:
        x = x * rng.uniform(0.85, 1.15) + rng.uniform(-15, 15)
        x = x * rng.uniform(0.94, 1.06, 3)
    if rng.random() < 0.3:
        g = rng.uniform(0.85, 1.18)
        x = 255.0 * (np.clip(x, 0, 255) / 255.0) ** g
    if rng.random() < 0.15:
        x = cv2.GaussianBlur(x, (0, 0), rng.uniform(0.5, 1.1))
    if rng.random() < 0.15:
        x = x + rng.normal(0, rng.uniform(2, 5), x.shape)
    return np.clip(x, 0, 255).astype(np.uint8)


def augment_tile(t, rng):
    """t: CROP×CROP×7 uint8 → (pre, post, label). Геометрія однакова для всіх каналів."""
    t = np.rot90(t, int(rng.integers(4)))
    if rng.random() < 0.5:
        t = t[:, ::-1]
    pre, post, lab = t[..., :3], t[..., 3:6], t[..., 6]
    if rng.random() < 0.3:                                   # zoom-in
        cs = int(CROP * rng.uniform(0.7, 0.95))
        y0, x0 = rng.integers(0, CROP - cs + 1, 2)
        sl = (slice(y0, y0 + cs), slice(x0, x0 + cs))
        pre = cv2.resize(np.ascontiguousarray(pre[sl]), (CROP, CROP), interpolation=cv2.INTER_LINEAR)
        post = cv2.resize(np.ascontiguousarray(post[sl]), (CROP, CROP), interpolation=cv2.INTER_LINEAR)
        lab = cv2.resize(np.ascontiguousarray(lab[sl]), (CROP, CROP), interpolation=cv2.INTER_NEAREST)
    if rng.random() < 0.3:                                   # неідеальне вирівнювання: зсув «до» на ±2 px
        dy, dx = rng.integers(-2, 3, 2)
        M = np.float32([[1, 0, dx], [0, 1, dy]])
        pre = cv2.warpAffine(np.ascontiguousarray(pre), M, (CROP, CROP), borderMode=cv2.BORDER_REFLECT_101)
    pre = _photometric(np.ascontiguousarray(pre), rng)       # незалежно для «до» і «після»
    post = _photometric(np.ascontiguousarray(post), rng)
    return pre, post, np.ascontiguousarray(lab)


class BankSampler:
    """Вибір тайлів: джерело ~ SOURCE_WEIGHTS, з імовірністю P_POS_CROP — тайл зі змінами."""

    def __init__(self, meta):
        self.src, self.has = meta['src'], meta['has']
        self.groups = {}
        for name, sid in SRC_ID.items():
            all_i = np.where(self.src == sid)[0]
            if len(all_i):
                self.groups[name] = (all_i, all_i[self.has[all_i]])
        w = np.array([SOURCE_WEIGHTS.get(k, 0) for k in self.groups], float)
        self.names, self.p = list(self.groups), w / w.sum()

    def draw(self, rng):
        name = self.names[rng.choice(len(self.names), p=self.p)]
        all_i, pos_i = self.groups[name]
        pool = pos_i if (len(pos_i) and rng.random() < P_POS_CROP) else all_i
        return int(pool[rng.integers(len(pool))])


class BankDataset(Dataset):
    """Кожен елемент — випадковий тайл з банку (джерело ~ SOURCE_WEIGHTS) з аугментаціями."""

    def __init__(self, bank_path, meta, length):
        self.bank_path, self.meta, self.length = bank_path, meta, length
        self.sampler = BankSampler(meta)
        self.arr, self.rng = None, None

    def __len__(self):
        return self.length

    def _lazy_init(self):                       # окремо в кожному процесі DataLoader
        self.arr = open_bank(self.bank_path, self.meta)
        self.rng = np.random.default_rng(torch.initial_seed() % (2 ** 32))

    def __getitem__(self, i):
        if self.arr is None:
            self._lazy_init()
        j = self.sampler.draw(self.rng)
        pre, post, lab = augment_tile(np.array(self.arr[j]), self.rng)
        return (torch.from_numpy(np.ascontiguousarray(pre.transpose(2, 0, 1))),
                torch.from_numpy(np.ascontiguousarray(post.transpose(2, 0, 1))),
                torch.from_numpy(lab))


from concurrent.futures import ThreadPoolExecutor
train_rows = manifest[manifest['split'] == 'train']
BANK_PATH = WORK / f'bank_{CROP}_{MANIFEST_SHA}.u8'
bank_meta = None
if RUN_TRAIN and len(train_rows):
    with Timer('Банк тайлів'):
        build_bank(train_rows, BANK_PATH)
    bank_meta = dict(np.load(BANK_PATH.with_suffix('.meta.npz')))
    tab = pd.DataFrame({'src': pd.Series(bank_meta['src']).map({v: k for k, v in SRC_ID.items()}), 'has': bank_meta['has']})
    print(tab.groupby('src')['has'].agg(tiles='count', with_change='sum'))
    pres = pd.DataFrame(bank_meta['pres'], columns=['destroyed', 'crater', 'fortification', 'new_building', 'unknown'])
    print('Тайлів з кожним класом:', pres.sum().to_dict())

## 7. Метрики та інференс

**Інференс кадру будь-якого розміру** (функція `predict_full`, розділ 2.3): ковзне вікно 256×256 (крок 192, зважене злиття перекриттів) — так само, як модель бачила дані під час навчання. Кадр 1024×1024 = 25 вікон, обробляються одним-двома батчами на GPU.

**Піксельні метрики** рахуються через гістограми ймовірностей (100 бінів) окремо для кожного джерела → за один прохід отримуємо F1 / Precision / Recall / IoU **для будь-якого порога**, PR-криві та матрицю помилок.
- *Бінарні* (є/немає зміни): позитив — будь-яка зміна (класи 1–4 і `254`).
- *Покласові*: піксель класу `c` вгадано, якщо ймовірність зміни ≥ порога **і** голова типу каже `c`. Пікселі `254` (тип невідомий) у покласових метриках не беруть участі; `255` — ніде.

**Об'єктні метрики** (для «скільки вирв/траншей/будівель знайдено»): компоненти зв'язності; GT-об'єкт знайдено, якщо ≥ 30% його пікселів покрито прогнозом того ж класу; прогнозований об'єкт правильний, якщо ≥ 30% його пікселів лежить на GT того ж класу. Дрібні прогнози (< `MIN_OBJ_AREA_M2`) відкидаються.

**Shortcut-тест синтетики:** частка пікселів «старих» об'єктів (є на обох знімках, мітка 0), які модель помилково назвала зміною, і частка таких об'єктів із хибним спрацюванням.

**Скор валідації** (вибір epoch і порога): середнє бінарних F1 по джерелах val (FOTBCD, xBD, SYN-ID) та F1 трьох цільових класів (зруйновано, вирва, фортифікація).

In [ ]:
# ==============================================================================
# 7.1 Метрики (numpy): гістограми ймовірностей, покласові, об'єктні, shortcut-тест
# ==============================================================================
NB = 100
GT_LUT = np.full(256, -1, np.int64)
for _c in range(5):
    GT_LUT[_c] = _c
GT_LUT[LBL_UNKNOWN] = 5                     # зміна невідомого типу
THR_GRID = np.round(np.arange(0.05, 0.951, 0.01), 2)


def prf(tp, fp, fn):
    tp, fp, fn = float(tp), float(fp), float(fn)
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0
    iou = tp / (tp + fp + fn) if (tp + fp + fn) else 0.0
    return dict(P=p, R=r, F1=f, IoU=iou)


class EvalAcc:
    """Накопичувач: гістограма [gt(0..5), тип(0..3), бін ймовірності] для кожного джерела."""

    def __init__(self):
        self.h = defaultdict(lambda: np.zeros((6, N_TYPES, NB), np.int64))
        self.aux = defaultdict(lambda: np.zeros(NB, np.int64))
        self.images = []                       # пер-кадрові записи (заповнюються в test)

    def add(self, subset, prob, typ, lab, aux=None):
        g = GT_LUT[lab]
        v = g >= 0
        b = np.minimum((prob * NB).astype(np.int64), NB - 1)
        idx = (g[v] * N_TYPES + typ[v].astype(np.int64)) * NB + b[v]
        self.h[subset] += np.bincount(idx, minlength=6 * N_TYPES * NB).reshape(6, N_TYPES, NB)
        if aux is not None:
            m = (aux > 0) & (lab == 0)
            self.aux[subset] += np.bincount(b[m], minlength=NB)

    def pooled(self, subsets):
        out = np.zeros((6, N_TYPES, NB), np.int64)
        for s in subsets:
            if s in self.h:
                out += self.h[s]
        return out


def _tb(t):
    return int(round(t * NB))


def binary_metrics(H, t):
    tb = _tb(t)
    pos, neg = H[1:].sum((0, 1)), H[0].sum(0)
    m = prf(pos[tb:].sum(), neg[tb:].sum(), pos[:tb].sum())
    m['support_px'] = int(pos.sum())
    return m


def class_metrics(H, t, c):
    tb, ti = _tb(t), c - 1
    tp = H[c, ti, tb:].sum()
    fn = H[c].sum() - tp
    fp = sum(H[g, ti, tb:].sum() for g in range(5) if g != c)      # g=5 (невідомий тип) — виключено
    m = prf(tp, fp, fn)
    m['support_px'] = int(H[c].sum())
    return m


def confusion(H, t):
    """Рядки — GT (0..4, 5=невідомий тип), стовпці — прогноз (0 = без змін, 1..4)."""
    tb = _tb(t)
    cm = np.zeros((6, 5), np.int64)
    for g in range(6):
        cm[g, 0] = H[g, :, :tb].sum()
        for ti in range(N_TYPES):
            cm[g, ti + 1] = H[g, ti, tb:].sum()
    return cm


VAL_BIN_SUBSETS = ('FOTBCD-Instances', 'FOTBCD-Binary', 'xBD-Global', 'SYN-ID')


def val_score(acc, t):
    comps, parts = [], {}
    for s in VAL_BIN_SUBSETS:
        if s in acc.h and acc.h[s][1:].sum() > 0:
            f = binary_metrics(acc.h[s], t)['F1']; comps.append(f); parts[f'bin_{s}'] = f
    pooled = acc.pooled(VAL_BIN_SUBSETS)
    for c in TARGET_CLASSES:
        if pooled[c].sum() > 0:
            f = class_metrics(pooled, t, c)['F1']; comps.append(f); parts[f'cls_{CLASS_SHORT[c]}'] = f
    return (float(np.mean(comps)) if comps else 0.0), parts


def best_threshold(acc):
    scores = np.array([val_score(acc, t)[0] for t in THR_GRID])
    i = int(scores.argmax())
    return float(THR_GRID[i]), float(scores[i]), scores


def pred_class_map(prob, typ, thr):
    return np.where(prob >= thr, typ.astype(np.uint8) + 1, 0).astype(np.uint8)


def object_counts(pred_cls, lab, gsd):
    """Об'єктні TP/кількості для класів 1..4 (критерій покриття OBJ_COVER)."""
    out = {}
    ign = lab == LBL_IGNORE
    for c in (1, 2, 3, 4):
        G, P = lab == c, pred_cls == c
        if not G.any() and not P.any():
            continue
        tp_g = n_g = tp_p = n_p = 0
        if G.any():
            ng, cg = cv2.connectedComponents(G.astype(np.uint8), connectivity=8)
            area_g = np.bincount(cg.ravel(), minlength=ng)
            cov = np.bincount(cg[P], minlength=ng) / np.maximum(area_g, 1)
            n_g, tp_g = ng - 1, int((cov[1:] >= OBJ_COVER).sum())
        if P.any():
            npc, cp = cv2.connectedComponents(P.astype(np.uint8), connectivity=8)
            area_p = np.bincount(cp.ravel(), minlength=npc)
            keep = area_p >= MIN_OBJ_AREA_M2[c] / max(gsd, 1e-3) ** 2
            keep[0] = False
            hit = np.bincount(cp[G], minlength=npc) / np.maximum(area_p, 1)
            ig = np.bincount(cp[ign], minlength=npc) / np.maximum(area_p, 1)
            keep &= ig < 0.5
            n_p, tp_p = int(keep.sum()), int((hit[keep] >= OBJ_COVER).sum())
        out[c] = (tp_g, n_g, tp_p, n_p)
    return out


def distractor_hits(prob, aux, lab, thr):
    """Скільки «старих» об'єктів (є на обох знімках) модель хибно назвала зміною."""
    m = (aux > 0) & (lab == 0)
    if not m.any():
        return 0, 0
    n, comp = cv2.connectedComponents(m.astype(np.uint8), connectivity=8)
    area = np.bincount(comp.ravel(), minlength=n)
    fired = np.bincount(comp[prob >= thr], minlength=n) / np.maximum(area, 1)
    return int((fired[1:] >= OBJ_COVER).sum()), n - 1


def summarize_binary(acc, t, subsets=None):
    rows = []
    for s in (subsets or SUBSETS_ORDER):
        if s in acc.h and acc.h[s].sum():
            rows.append(dict(subset=s, **binary_metrics(acc.h[s], t)))
    real = [s for s in ('FOTBCD-Instances', 'FOTBCD-Binary', 'xBD-Global') if s in acc.h]
    if len(real) > 1:
        rows.append(dict(subset='REAL (FOTBCD+xBD)', **binary_metrics(acc.pooled(real), t)))
    return pd.DataFrame(rows)


def summarize_classes(acc, t, subsets=None):
    rows = []
    for s in (subsets or SUBSETS_ORDER):
        if s not in acc.h:
            continue
        for c in (1, 2, 3, 4):
            if acc.h[s][c].sum() > 0:
                rows.append(dict(subset=s, cls=c, name=CLASS_NAMES[c], **class_metrics(acc.h[s], t, c)))
    return pd.DataFrame(rows)


def load_row_arrays(r):
    pre, post, lab = read_rgb(r['pre']), read_rgb(r['post']), read_gray(r['label'])
    aux = read_gray(r['aux']) if r.get('aux') else None
    return pre, post, lab, aux


def iterate_predictions(model, rows, stride, tta=False, desc='eval', prefetch=4):
    """Генератор (row, pre, post, lab, aux, prob, typ) з фоновим читанням файлів."""
    rows = rows.to_dict('records') if isinstance(rows, pd.DataFrame) else list(rows)
    with ThreadPoolExecutor(2) as ex:
        futs = [ex.submit(load_row_arrays, r) for r in rows[:prefetch]]
        for i, r in enumerate(tqdm(rows, desc=desc, leave=False)):
            pre, post, lab, aux = futs[i].result()
            futs[i] = None
            if i + prefetch < len(rows):
                futs.append(ex.submit(load_row_arrays, rows[i + prefetch]))
            prob, typ = predict_full(model, pre, post, stride=stride, tta=tta)
            yield r, pre, post, lab, aux, prob, typ


def evaluate(model, rows, stride, tta=False, desc='eval'):
    acc = EvalAcc()
    for r, pre, post, lab, aux, prob, typ in iterate_predictions(model, rows, stride, tta, desc):
        acc.add(r['subset'], prob, typ, lab, aux)
    return acc


print('Метрики визначено.')

## 8. Zero-shot baseline

Точка відліку: офіційна BIT-CD з вагами LEVIR-CD **без донавчання** (лише бінарна голова «є/немає зміни»). Показує, скільки дає донавчання на наших даних.

In [ ]:
# ==============================================================================
# 8. Zero-shot baseline: BIT-CD з вагами LEVIR-CD без донавчання (лише бінарна голова, validation)
# ==============================================================================
val_rows = manifest[manifest['split'] == 'val'].reset_index(drop=True)
baseline_tbl = None
if RUN_BASELINE and len(val_rows):
    base_path = RUN_DIR / 'baseline_val.csv'
    if base_path.exists():
        baseline_tbl = pd.read_csv(base_path)
    else:
        bm = build_model(pretrained=True)
        with Timer('Baseline на validation'):
            acc_b = evaluate(bm, val_rows, stride=VAL_STRIDE_DURING_TRAIN, desc='baseline val')
        t05 = summarize_binary(acc_b, 0.5).assign(threshold=0.5)
        # найкращий поріг саме для baseline (теж на validation) — щоб порівняння було чесним до baseline
        best_t = max(THR_GRID, key=lambda t: np.mean([binary_metrics(acc_b.h[s], t)['F1'] for s in acc_b.h]))
        tb = summarize_binary(acc_b, best_t).assign(threshold=best_t)
        baseline_tbl = pd.concat([t05, tb], ignore_index=True)
        baseline_tbl.to_csv(base_path, index=False)
        del bm
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
    print('Zero-shot LEVIR BIT-CD, бінарні метрики на validation (тип змін ця модель не розрізняє):')
    display(baseline_tbl.round(4))
else:
    print('Baseline вимкнено.')

## 9. Навчання

- Оптимізатор AdamW, `lr = 2e-4` (голова типу ×5), warmup 400 ітерацій → косинусне згасання; AMP (fp16) на T4; градієнти обрізаються до норми 5.
- Епоха = `ITERS_PER_EPOCH × BATCH` тайлів з банку; після кожної — validation на фіксованій підвибірці (≤200 пар на джерело, крок вікна 256 — для швидкості), запис `history.csv`. Поріг потім обирається на **повному** validation.
- `best.pt` — за **скором валідації** при порозі 0,5 (поріг остаточно обирається в розділі 10); `last.pt` — для відновлення після обриву (модель, оптимізатор, scaler, крок, історія, відбиток даних).
- Ранній зупин, якщо скор не росте `EARLY_STOP_PATIENCE` епох. Test у навчанні **не читається**.
- Відбиток (`MANIFEST_SHA`) зберігається в checkpoint: відновити навчання на інших даних не вийде — це захист від тихого змішування експериментів.

In [ ]:
# ==============================================================================
# 9.1 Навчання з відновленням
# ==============================================================================
LAST_PT, BEST_PT, HIST_CSV = RUN_DIR / 'last.pt', RUN_DIR / 'best.pt', RUN_DIR / 'history.csv'
TOTAL_ITERS = EPOCHS * ITERS_PER_EPOCH


def lr_factor(step):
    if step < WARMUP_ITERS:
        return (step + 1) / WARMUP_ITERS
    q = (step - WARMUP_ITERS) / max(1, TOTAL_ITERS - WARMUP_ITERS)
    return 0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * min(q, 1.0)))


def run_training():
    if RUN_AUDIT and not AUDIT_OK:
        raise RuntimeError('Аудит виявив критичні проблеми — навчання заблоковано (див. розділ 5).')
    if DEVICE.type != 'cuda':
        raise RuntimeError('Навчання потребує GPU: Runtime → Change runtime type → T4 GPU.')
    model = build_model(pretrained=True)
    head = [p for n, p in model.named_parameters() if n.startswith('type_head.') and p.requires_grad]
    body = [p for n, p in model.named_parameters() if not n.startswith('type_head.') and p.requires_grad]
    opt = torch.optim.AdamW([{'params': body, 'lr': LR, 'base_lr': LR},
                             {'params': head, 'lr': LR * LR_TYPE_HEAD_MULT, 'base_lr': LR * LR_TYPE_HEAD_MULT}],
                            weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    history, start_epoch, step, best, bad_epochs = [], 1, 0, -1.0, 0
    if LAST_PT.exists():
        ck = torch.load(LAST_PT, map_location='cpu', weights_only=False)
        if ck.get('manifest_sha') != MANIFEST_SHA:
            raise RuntimeError(f'last.pt зроблено на інших даних ({ck.get("manifest_sha")} ≠ {MANIFEST_SHA}). '
                               f'Змініть RUN_NAME для нового експерименту.')
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); scaler.load_state_dict(ck['scaler'])
        history, start_epoch, step = ck['history'], ck['epoch'] + 1, ck['step']
        best, bad_epochs = ck['best'], ck['bad_epochs']
        print(f'Відновлено після епохи {ck["epoch"]} (best score {best:.4f})')
    if start_epoch > EPOCHS or bad_epochs >= EARLY_STOP_PATIENCE:
        print('Навчання вже завершено.'); return history
    type_w = type_class_weights(bank_meta)
    print('Ваги класів голови типу:', dict(zip([CLASS_SHORT[c] for c in (1, 2, 3, 4)], type_w.cpu().numpy().round(2))))
    ds = BankDataset(BANK_PATH, bank_meta, ITERS_PER_EPOCH * BATCH)
    dl = DataLoader(ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                    persistent_workers=False)
    val_all = manifest[manifest['split'] == 'val']
    val_rows = pd.concat([g.sample(min(len(g), VAL_MAX_PER_SUBSET_DURING_TRAIN), random_state=SEED)
                          for _, g in val_all.groupby('subset')]).sort_values('uid')
    print(f'Validation під час навчання: {len(val_rows)} з {len(val_all)} пар (≤{VAL_MAX_PER_SUBSET_DURING_TRAIN} на джерело)')
    for epoch in range(start_epoch, EPOCHS + 1):
        model.train()
        t0 = time.perf_counter()
        sums = {k: torch.zeros((), device=DEVICE) for k in ('loss', 'ce', 'dice', 'type_ce')}
        n_it = 0
        for pre, post, lab in tqdm(dl, desc=f'epoch {epoch}/{EPOCHS}', leave=False):
            for g in opt.param_groups:
                g['lr'] = g['base_lr'] * lr_factor(step)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP):
                ch, ty = model(to_input(pre), to_input(post))
            loss, parts = compute_loss(ch, ty, lab, type_w)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 5.0)
            scaler.step(opt); scaler.update()
            sums['loss'] += loss.detach()
            for k, v in parts.items():
                sums[k] += v
            n_it += 1; step += 1
        train_time = time.perf_counter() - t0
        acc = evaluate(model, val_rows, stride=VAL_STRIDE_DURING_TRAIN, desc=f'val {epoch}')
        score, sparts = val_score(acc, 0.5)
        rec = dict(epoch=epoch, step=step, lr=opt.param_groups[0]['lr'], score=score,
                   train_s=train_time, val_s=time.perf_counter() - t0 - train_time,
                   tiles_per_s=n_it * BATCH / max(train_time, 1e-6),
                   **{k: float(v) / max(n_it, 1) for k, v in sums.items()}, **sparts)
        history.append(rec)
        improved = score > best
        if improved:
            best, bad_epochs = score, 0
            save_checkpoint(dict(model=model.state_dict(), epoch=epoch, score=score, parts=sparts,
                                 manifest_sha=MANIFEST_SHA, config=dict(CROP=CROP, TILE=TILE, N_TYPES=N_TYPES)), BEST_PT)
        else:
            bad_epochs += 1
        save_checkpoint(dict(model=model.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(), epoch=epoch,
                             step=step, history=history, best=best, bad_epochs=bad_epochs, manifest_sha=MANIFEST_SHA), LAST_PT)
        pd.DataFrame(history).to_csv(HIST_CSV, index=False)
        msg = ' | '.join(f'{k.replace("bin_", "").replace("cls_", "")}={v:.3f}' for k, v in sparts.items())
        print(f'Епоха {epoch:02d} | loss {rec["loss"]:.4f} | val score {score:.4f}{" ★" if improved else ""} | {msg} | '
              f'{rec["tiles_per_s"]:.0f} тайлів/с, {train_time / 60:.1f}+{rec["val_s"] / 60:.1f} хв')
        if epoch == start_epoch:
            eta = (train_time + rec['val_s']) * (EPOCHS - epoch) / 60
            print(f'   ≈ залишилось {eta:.0f} хв (якщо задовго — зменшіть EPOCHS або ITERS_PER_EPOCH; навчання відновлюється)')
        if bad_epochs >= EARLY_STOP_PATIENCE:
            print(f'Ранній зупин: {EARLY_STOP_PATIENCE} епох без покращення.')
            break
    return history


if RUN_TRAIN:
    if bank_meta is None:
        raise RuntimeError('Банк тайлів не побудовано (розділ 6).')
    history = run_training()
    print('Найкращий val score:', max((h['score'] for h in history), default=float('nan')))
else:
    print('Навчання вимкнено (RUN_TRAIN=False).')

In [ ]:
# ==============================================================================
# 9.2 Графіки навчання
# ==============================================================================
HIST_CSV = RUN_DIR / 'history.csv'
if HIST_CSV.exists():
    hist = pd.read_csv(HIST_CSV)
    fig, ax = plt.subplots(2, 2, figsize=(15, 9))
    for k, lbl in (('loss', 'сумарна'), ('ce', 'CE зміни'), ('dice', 'Dice зміни'), ('type_ce', 'CE типу')):
        if k in hist:
            ax[0, 0].plot(hist['epoch'], hist[k], marker='o', ms=3, label=lbl)
    ax[0, 0].set_title('Втрати на train'); ax[0, 0].legend(); ax[0, 0].grid(alpha=0.3)
    for c in [c for c in hist.columns if c.startswith('bin_')]:
        ax[0, 1].plot(hist['epoch'], hist[c], marker='o', ms=3, label=c[4:])
    ax[0, 1].plot(hist['epoch'], hist['score'], 'k--', lw=2, label='скор валідації')
    ax[0, 1].axhline(TARGET_F1, color='r', ls=':', label=f'ціль F1={TARGET_F1}')
    ax[0, 1].set_title('Validation: бінарний F1 за джерелами (поріг 0,5)'); ax[0, 1].legend(fontsize=8); ax[0, 1].grid(alpha=0.3)
    for c in [c for c in hist.columns if c.startswith('cls_')]:
        cid = {v: k for k, v in CLASS_SHORT.items()}[c[4:]]
        ax[1, 0].plot(hist['epoch'], hist[c], marker='o', ms=3, color=np.array(CLASS_COLORS[cid]) / 255, label=CLASS_NAMES[cid])
    ax[1, 0].axhline(TARGET_F1, color='r', ls=':')
    ax[1, 0].set_title('Validation: F1 цільових класів (поріг 0,5)'); ax[1, 0].legend(fontsize=8); ax[1, 0].grid(alpha=0.3)
    ax[1, 1].plot(hist['epoch'], hist['lr'], marker='o', ms=3)
    ax[1, 1].set_title('Learning rate'); ax[1, 1].grid(alpha=0.3)
    ax2 = ax[1, 1].twinx(); ax2.plot(hist['epoch'], hist['tiles_per_s'], 'g--', alpha=0.6); ax2.set_ylabel('тайлів/с', color='g')
    for a in ax.ravel():
        a.set_xlabel('епоха')
    plt.tight_layout(); save_fig(fig, 'training_curves'); plt.show()
    best_row = hist.loc[hist['score'].idxmax()]
    print(f'Найкраща епоха: {int(best_row["epoch"])} | val score {best_row["score"]:.4f} | '
          f'час навчання сумарно {hist["train_s"].sum() / 60:.1f} хв + валідація {hist["val_s"].sum() / 60:.1f} хв')
    display(hist.round(4).tail(8))
else:
    print('Історії навчання ще немає.')

## 10. Вибір порога — тільки на validation

Найкращий checkpoint проганяємо по validation **тим самим інференсом, що й на test** (крок вікна 192, TTA згідно з конфігурацією). Поріг ймовірності зміни обираємо за максимумом скору валідації; він фіксується у `threshold.json` і далі не змінюється. Показуємо також F1(поріг) і PR-криві для кожного джерела.

In [ ]:
# ==============================================================================
# 10.1 Найкращий checkpoint → validation → поріг
# ==============================================================================
THR_JSON, VAL_HIST = RUN_DIR / 'threshold.json', RUN_DIR / 'val_hist.npz'
BEST_PT = RUN_DIR / 'best.pt'


def save_acc(acc, path):
    np.savez(path, **{f'h::{k}': v for k, v in acc.h.items()}, **{f'aux::{k}': v for k, v in acc.aux.items()})


def load_acc(path):
    acc, z = EvalAcc(), np.load(path)
    for k in z.files:
        kind, name = k.split('::', 1)
        (acc.h if kind == 'h' else acc.aux)[name] = z[k]
    return acc


# У режимі без навчання отримуємо готову модель, якщо власного checkpoint немає.
if not RUN_TRAIN and DOWNLOAD_SHARED_WEIGHTS and not BEST_PT.exists():
    _download_shared_file(BEST_PT, *SHARED_FILES['best.pt'])
    if not THR_JSON.exists():
        _download_shared_file(THR_JSON, *SHARED_FILES['threshold.json'])

best_model, THRESHOLD, acc_val = None, 0.5, None
if BEST_PT.exists():
    best_model, best_ck = load_trained_model(BEST_PT)
    if best_ck.get('manifest_sha') != MANIFEST_SHA:
        raise RuntimeError('best.pt навчено на інших даних — оцінка заблокована (змініть RUN_NAME або дані).')
    cfg_now = dict(epoch=int(best_ck['epoch']), stride=STRIDE, tta=TTA)
    saved = json.loads(THR_JSON.read_text()) if THR_JSON.exists() else None
    if saved and {k: saved.get(k) for k in cfg_now} == cfg_now and VAL_HIST.exists():
        THRESHOLD, acc_val = float(saved['threshold']), load_acc(VAL_HIST)
        print('Поріг відновлено з threshold.json')
    else:
        with Timer('Validation для вибору порога'):
            acc_val = evaluate(best_model, val_rows, stride=STRIDE, tta=TTA, desc='val (поріг)')
        THRESHOLD, best_sc, scores = best_threshold(acc_val)
        save_acc(acc_val, VAL_HIST)
        THR_JSON.write_text(json.dumps(dict(threshold=THRESHOLD, val_score=best_sc, **cfg_now), indent=1))
    sc_at, parts_at = val_score(acc_val, THRESHOLD)
    print(f'Checkpoint epoch {best_ck["epoch"]} | ОБРАНИЙ ПОРІГ = {THRESHOLD:.2f} | val score = {sc_at:.4f}')
    print('Validation при обраному порозі:')
    display(summarize_binary(acc_val, THRESHOLD).round(4))
    display(summarize_classes(acc_val, THRESHOLD).round(4))

    # --- F1(поріг) та PR-криві
    fig, ax = plt.subplots(1, 3, figsize=(19, 5))
    for s in [s for s in SUBSETS_ORDER if s in acc_val.h]:
        f1 = [binary_metrics(acc_val.h[s], t)['F1'] for t in THR_GRID]
        ax[0].plot(THR_GRID, f1, label=s)
        pr = [binary_metrics(acc_val.h[s], t) for t in THR_GRID]
        ax[2].plot([m['R'] for m in pr], [m['P'] for m in pr], label=s)
    ax[0].plot(THR_GRID, [val_score(acc_val, t)[0] for t in THR_GRID], 'k--', lw=2, label='скор валідації')
    pooled = acc_val.pooled(VAL_BIN_SUBSETS)
    for c in (1, 2, 3, 4):
        if pooled[c].sum():
            ax[1].plot(THR_GRID, [class_metrics(pooled, t, c)['F1'] for t in THR_GRID], color=np.array(CLASS_COLORS[c]) / 255,
                       label=CLASS_NAMES[c])
    for a, t in zip(ax[:2], ('Бінарний F1 від порога (val)', 'F1 класів від порога (val, усі джерела)')):
        a.axvline(THRESHOLD, color='r', ls=':', label=f'обраний поріг {THRESHOLD:.2f}')
        a.axhline(TARGET_F1, color='grey', ls='--', lw=0.8)
        a.set_title(t); a.set_xlabel('поріг'); a.set_ylabel('F1'); a.legend(fontsize=8); a.grid(alpha=0.3)
    ax[2].set_title('PR-криві (val, бінарні)'); ax[2].set_xlabel('Recall'); ax[2].set_ylabel('Precision')
    ax[2].legend(fontsize=8); ax[2].grid(alpha=0.3)
    plt.tight_layout(); save_fig(fig, 'threshold_selection'); plt.show()
else:
    print('best.pt ще немає — спочатку навчання (розділ 9).')

## 11. Фінальний TEST (один раз)

Поріг і checkpoint зафіксовані на validation. Результат пишеться у `final_test.json`; повторний запуск лише показує збережене (новий прогін — тільки з `FORCE_RETEST=True`, що треба відобразити у звіті).

Таблиці:
1. **Бінарні піксельні метрики за джерелами** (+ «REAL» — FOTBCD та xBD разом). Синтетика — окремими рядками, xBD-Ukraine — out-of-distribution.
2. **Покласові піксельні метрики** (зруйнований об'єкт / вирва / фортифікація / нова забудова).
3. **Об'єктні метрики** (покриття ≥ 30%).
4. **Shortcut-тест синтетики** — хибні спрацювання на «старих» об'єктах, присутніх на обох знімках.
5. Матриці помилок (реальні дані / синтетика).

In [ ]:
# ==============================================================================
# 11.1 Одноразовий TEST
# ==============================================================================
TEST_JSON, TEST_HIST, TEST_IMG_CSV = RUN_DIR / 'final_test.json', RUN_DIR / 'test_hist.npz', RUN_DIR / 'test_per_image.csv'
test_rows = manifest[manifest['split'] == 'test'].reset_index(drop=True)


def _pass(x):
    return 'PASS' if x >= TARGET_F1 else 'FAIL'


def run_test(model):
    acc = EvalAcc()
    obj = defaultdict(lambda: np.zeros(4, np.int64))
    distr = defaultdict(lambda: np.zeros(2, np.int64))
    per_img = []
    for r, pre, post, lab, aux, prob, typ in iterate_predictions(model, test_rows, STRIDE, TTA, desc='TEST'):
        acc.add(r['subset'], prob, typ, lab, aux)
        pc = pred_class_map(prob, typ, THRESHOLD)
        for c, v in object_counts(pc, lab, float(r['gsd'])).items():
            obj[(r['subset'], c)] += np.array(v)
        if aux is not None:
            distr[r['subset']] += np.array(distractor_hits(prob, aux, lab, THRESHOLD))
        g = GT_LUT[lab]
        chg, valid, pred = g >= 1, lab != LBL_IGNORE, prob >= THRESHOLD
        tp, fp, fn = int((pred & chg).sum()), int((pred & ~chg & valid).sum()), int((~pred & chg).sum())
        per_img.append(dict(uid=r['uid'], subset=r['subset'], tp=tp, fp=fp, fn=fn, gt_px=int(chg.sum()),
                            f1=2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 1.0))
    return acc, obj, distr, pd.DataFrame(per_img)


def test_tables(acc, obj, distr):
    bt = summarize_binary(acc, THRESHOLD)
    bt['F1≥0.75'] = bt['F1'].map(_pass)
    ct = summarize_classes(acc, THRESHOLD)
    if len(ct):
        ct['F1≥0.75'] = ct['F1'].map(_pass)
    orows = []
    for (s, c), (tp_g, n_g, tp_p, n_p) in sorted(obj.items(), key=lambda kv: (SUBSETS_ORDER.index(kv[0][0]), kv[0][1])):
        p = tp_p / n_p if n_p else 0.0; rr = tp_g / n_g if n_g else 0.0
        orows.append(dict(subset=s, cls=c, name=CLASS_NAMES[c], gt_objects=int(n_g), pred_objects=int(n_p),
                          obj_P=p, obj_R=rr, obj_F1=2 * p * rr / (p + rr) if p + rr else 0.0))
    ot = pd.DataFrame(orows)
    srows = []
    for s, (hits, n) in distr.items():
        hist_aux = acc.aux[s]
        px_fp = hist_aux[_tb(THRESHOLD):].sum() / max(hist_aux.sum(), 1)
        srows.append(dict(subset=s, old_objects=int(n), falsely_detected=int(hits),
                          object_false_alarm_rate=hits / max(n, 1), pixel_false_alarm_rate=px_fp))
    return bt, ct, ot, pd.DataFrame(srows)


acc_test = None
if RUN_FINAL_TEST and best_model is not None:
    if TEST_JSON.exists() and not FORCE_RETEST:
        print('⚠ TEST уже виконано раніше — показуємо збережений результат (новий прогін лише з FORCE_RETEST=True).')
        saved = json.loads(TEST_JSON.read_text())
        if saved.get('checkpoint_epoch') != int(best_ck['epoch']) or abs(saved.get('threshold', -1) - THRESHOLD) > 1e-9:
            print(f"⚠⚠ Збережений test зроблено для epoch {saved.get('checkpoint_epoch')} / поріг {saved.get('threshold')}, "
                  f"а поточний best — epoch {best_ck['epoch']} / поріг {THRESHOLD:.2f}. Нижче — СТАРІ результати; "
                  f"для нового checkpoint потрібен FORCE_RETEST=True (зазначте це у звіті).")
        acc_test = load_acc(TEST_HIST)
        bin_tbl, cls_tbl = pd.DataFrame(saved['binary']), pd.DataFrame(saved['classes'])
        obj_tbl, sc_tbl = pd.DataFrame(saved['objects']), pd.DataFrame(saved['shortcut'])
        per_img_df = pd.read_csv(TEST_IMG_CSV)
    else:
        if FORCE_RETEST and TEST_JSON.exists():
            print('⚠ FORCE_RETEST=True: test запускається повторно — зазначте це у звіті.')
        with Timer('TEST'):
            acc_test, obj_acc, distr_acc, per_img_df = run_test(best_model)
        bin_tbl, cls_tbl, obj_tbl, sc_tbl = test_tables(acc_test, obj_acc, distr_acc)
        save_acc(acc_test, TEST_HIST)
        per_img_df.to_csv(TEST_IMG_CSV, index=False)
        TEST_JSON.write_text(json.dumps(dict(
            threshold=THRESHOLD, checkpoint_epoch=int(best_ck['epoch']), stride=STRIDE, tta=TTA, manifest_sha=MANIFEST_SHA,
            forced_retest=bool(FORCE_RETEST and TEST_JSON.exists()), n_pairs=test_rows['subset'].value_counts().to_dict(),
            binary=bin_tbl.to_dict('records'), classes=cls_tbl.to_dict('records'), objects=obj_tbl.to_dict('records'),
            shortcut=sc_tbl.to_dict('records')), ensure_ascii=False, indent=1, default=float))
    print(f'\nПОРІГ = {THRESHOLD:.2f} (обрано на validation) | пар у test:', test_rows['subset'].value_counts().to_dict())
    print('\n1) Бінарні піксельні метрики (є/немає зміни):'); display(bin_tbl.round(4))
    print('\n2) Покласові піксельні метрики:'); display(cls_tbl.round(4))
    print('\n3) Об\'єктні метрики (покриття ≥ 30%):'); display(obj_tbl.round(4))
    print('\n4) Shortcut-тест синтетики («старі» об\'єкти на обох знімках, мітка 0):'); display(sc_tbl.round(4))
    for t, n in ((bin_tbl, 'test_binary'), (cls_tbl, 'test_classes'), (obj_tbl, 'test_objects'), (sc_tbl, 'test_shortcut')):
        t.to_csv(RUN_DIR / f'{n}.csv', index=False)
elif best_model is None:
    print('Немає навченої моделі (best.pt).')
else:
    print('Фінальний test вимкнено.')

In [ ]:
# ==============================================================================
# 11.2 Візуалізація test: матриці помилок, приклади (вдалі та найгірші), збільшення по класах
# ==============================================================================
def plot_confusions(acc):
    groups = {'Реальні дані (FOTBCD + xBD)': [s for s in ('FOTBCD-Instances', 'FOTBCD-Binary', 'xBD-Global') if s in acc.h],
              'Синтетика (ID + Shift)': [s for s in ('SYN-ID', 'SYN-SHIFT') if s in acc.h],
              'xBD-Ukraine (OOD)': [s for s in ('xBD-Ukraine',) if s in acc.h]}
    groups = {k: v for k, v in groups.items() if v}
    fig, axes = plt.subplots(1, len(groups), figsize=(6.5 * len(groups), 5.5), squeeze=False)
    names = ['без змін'] + [CLASS_SHORT[c] for c in (1, 2, 3, 4)]
    for ax, (title, subs) in zip(axes[0], groups.items()):
        cm = confusion(acc.pooled(subs), THRESHOLD)[:5]
        keep = cm.sum(1) > 0
        cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
        ax.imshow(cmn[keep], cmap='Blues', vmin=0, vmax=1)
        ax.set_xticks(range(5)); ax.set_xticklabels(names, rotation=30, fontsize=8)
        ax.set_yticks(range(int(keep.sum()))); ax.set_yticklabels([n for n, k in zip(names, keep) if k], fontsize=8)
        for i, row in enumerate(cmn[keep]):
            for j, v in enumerate(row):
                ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8, color='w' if v > 0.5 else 'k')
        ax.set_title(title, fontsize=10); ax.set_xlabel('прогноз'); ax.set_ylabel('GT')
    plt.tight_layout(); save_fig(fig, 'test_confusion'); plt.show()


def show_predictions(model, rows, title, name, zoom=None):
    if not len(rows):
        return
    fig, axes = plt.subplots(len(rows), 5, figsize=(24, 4.8 * len(rows)), squeeze=False)
    for i, r in enumerate(rows.to_dict('records')):
        pre, post, lab, aux = load_row_arrays(r)
        prob, typ = predict_full(model, pre, post, stride=STRIDE, tta=TTA)
        pc = pred_class_map(prob, typ, THRESHOLD)
        sl = (slice(None), slice(None))
        if zoom:
            m = np.isin(lab, zoom)
            ys, xs = np.where(m) if m.any() else (np.array([0]), np.array([0]))
            k = len(ys) // 2; s = min(192, lab.shape[0], lab.shape[1])
            y0 = int(np.clip(ys[k] - s // 2, 0, lab.shape[0] - s)); x0 = int(np.clip(xs[k] - s // 2, 0, lab.shape[1] - s))
            sl = (slice(y0, y0 + s), slice(x0, x0 + s))
        f1 = per_img_df.set_index('uid')['f1'].get(r['uid'], np.nan) if 'per_img_df' in globals() else np.nan
        panels = [(pre[sl], f'{r["subset"]} | До\n{r["uid"]}'), (post[sl], 'Після'),
                  (label_overlay(post, lab)[sl], 'GT'), (label_overlay(post, pc)[sl], f'Прогноз (поріг {THRESHOLD:.2f}) | F1 кадру {f1:.2f}')]
        for j, (im, t) in enumerate(panels):
            axes[i, j].imshow(im, interpolation='nearest' if zoom else None); axes[i, j].set_title(t, fontsize=8)
        axes[i, 4].imshow(prob[sl], cmap='magma', vmin=0, vmax=1); axes[i, 4].set_title('Ймовірність зміни', fontsize=8)
        for a in axes[i]:
            a.axis('off')
    fig.legend(handles=legend_handles(True, True), loc='lower center', ncol=3, fontsize=9)
    fig.suptitle(title, fontsize=13); plt.tight_layout(rect=(0, 0.03, 1, 0.97)); save_fig(fig, name); plt.show()


if acc_test is not None and best_model is not None:
    plot_confusions(acc_test)
    pi = per_img_df.merge(test_rows[['uid', 'subset']], on=['uid', 'subset'])
    for s in SUBSETS_ORDER:
        sub = pi[(pi['subset'] == s) & (pi['gt_px'] > 0)]
        if not len(sub):
            continue
        good = sub.sample(min(2, len(sub)), random_state=SEED)
        worst = sub.nsmallest(1, 'f1')
        pick = test_rows.set_index('uid').loc[list(good['uid']) + list(worst['uid'])].reset_index()
        show_predictions(best_model, pick, f'TEST · {s}: два випадкові приклади + найгірший кадр', f'test_examples_{s}')
    # збільшення: вирви та фортифікації (синтетика), зруйновані об'єкти (xBD / xBD-Ukraine)
    for cls, subs in ((LBL_CRATER, ('SYN-ID', 'SYN-SHIFT')), (LBL_FORT, ('SYN-ID', 'SYN-SHIFT')), (LBL_DESTROYED, ('xBD-Global', 'xBD-Ukraine'))):
        cand = test_rows[test_rows['subset'].isin(subs)]
        cand = cand[cand['uid'].isin(pi.loc[pi['gt_px'] > 0, 'uid'])]
        if 'px_crater' in cand and cls in (LBL_CRATER, LBL_FORT):
            cand = cand[cand['px_crater' if cls == LBL_CRATER else 'px_fort'].fillna(0) > 0]
        if len(cand):
            show_predictions(best_model, cand.sample(min(3, len(cand)), random_state=SEED + cls),
                             f'TEST · збільшено: {CLASS_NAMES[cls]}', f'test_zoom_{CLASS_SHORT[cls]}', zoom=[cls])

## 12. Швидкодія

Вимірюємо **повний конвеєр** для пари 1024×1024, що вже в пам'яті: перенесення на GPU → 25 вікон BIT-CD (AMP fp16) → злиття → поріг → карта класів → компоненти зв'язності та список об'єктів → повернення на CPU. Прогрів 3 прогони, синхронізація CUDA перед/після кожного заміру. Окремо — режим із TTA×4 (запас по часу) і читання PNG з диска.

In [ ]:
# ==============================================================================
# 12. Швидкодія: пара 1024×1024, повний конвеєр (вимога ≤ 2 с)
# ==============================================================================
DEFAULT_GSD_UNKNOWN = 0.3          # для фільтра мінімальної площі, коли GSD знімка невідомий


def extract_objects(cls_map, prob, conf, gsd=None):
    g = gsd or DEFAULT_GSD_UNKNOWN
    objs = []
    for c in (1, 2, 3, 4):
        m = cls_map == c
        if not m.any():
            continue
        n, comp, stats, cent = cv2.connectedComponentsWithStats(m.astype(np.uint8), connectivity=8)
        for k in range(1, n):
            x, y, w, h, area = (int(v) for v in stats[k])
            if area < MIN_OBJ_AREA_M2[c] / g ** 2:
                continue
            mk = comp[y:y + h, x:x + w] == k
            score = float(prob[y:y + h, x:x + w][mk].mean() * conf[y:y + h, x:x + w][mk].mean())
            objs.append(dict(cls=c, name=CLASS_NAMES[c], target=c in TARGET_CLASSES, area_px=area,
                             area_m2=round(area * gsd ** 2, 2) if gsd else None, bbox_xywh=[x, y, w, h],
                             centroid_xy=[round(float(cent[k][0]), 1), round(float(cent[k][1]), 1)], score=round(score, 3)))
    return sorted(objs, key=lambda o: -o['score'])


def detect_changes_arrays(model, pre, post, thr=None, gsd=None, stride=STRIDE, tta=TTA):
    """Повний конвеєр для пари в пам'яті: інференс → поріг → карта класів → список об'єктів."""
    prob, typ, conf = predict_full(model, pre, post, stride=stride, tta=tta, return_type_prob=True)
    cls_map = pred_class_map(prob, typ, THRESHOLD if thr is None else thr)
    return dict(class_map=cls_map, change_prob=prob, type_conf=conf, objects=extract_objects(cls_map, prob, conf, gsd))


LAT_JSON = RUN_DIR / 'latency.json'
latency = None
if best_model is not None:
    cand = test_rows[(test_rows['h'] == 1024) & (test_rows['w'] == 1024)]
    note = 'реальні тестові пари 1024×1024'
    if len(cand):
        pick = cand.sample(min(20, len(cand)), random_state=SEED)
        t_io = []
        pairs = []
        for r in pick.itertuples():
            t0 = time.perf_counter(); pairs.append((read_rgb(r.pre), read_rgb(r.post))); t_io.append(time.perf_counter() - t0)
    else:
        rng_l = np.random.default_rng(SEED)
        pairs = [(rng_l.integers(0, 255, (1024, 1024, 3), dtype=np.uint8),) * 2 for _ in range(10)]
        t_io, note = [], 'випадкові масиви 1024×1024 (у test немає кадрів 1024)'
    latency = {}
    for name, tta in (('робочий режим', TTA), ('з TTA×4 (запас)', True)):
        for p in pairs[:3]:
            detect_changes_arrays(best_model, *p, tta=tta)                 # прогрів
        times = []
        for p in pairs:
            cuda_sync(); t0 = time.perf_counter()
            detect_changes_arrays(best_model, *p, tta=tta)
            cuda_sync(); times.append(time.perf_counter() - t0)
        times = np.array(times)
        latency[name] = dict(mean=float(times.mean()), median=float(np.median(times)), p95=float(np.percentile(times, 95)),
                             max=float(times.max()), n=len(times), tta=bool(tta))
    latency['meta'] = dict(device=torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU', tile=TILE, stride=STRIDE,
                           windows_per_frame=len(window_origins(1024, TILE, STRIDE)) ** 2, data=note,
                           png_read_pair_s=float(np.mean(t_io)) if t_io else None, torch=torch.__version__)
    LAT_JSON.write_text(json.dumps(latency, ensure_ascii=False, indent=1))
    lt = pd.DataFrame({k: v for k, v in latency.items() if k != 'meta'}).T
    lt['≤ 2 с'] = np.where(lt['max'] <= TARGET_LATENCY_S, 'PASS', 'FAIL')
    print(f"Пристрій: {latency['meta']['device']} | вікон на кадр: {latency['meta']['windows_per_frame']} | дані: {note}")
    print('Час від масивів «до/після» в пам\'яті до карти класів і списку об\'єктів (включно з GPU↔CPU), секунди:')
    display(lt.round(4))
    if t_io:
        print(f'Окремо: читання двох PNG з диска ≈ {np.mean(t_io):.3f} с на пару (залежить від диска, не від моделі).')
else:
    print('Немає навченої моделі.')

## 13. Демонстрація: `detect_changes(pre_path, post_path)`

Функція приймає два вирівняні знімки, повертає карту класів, ймовірність зміни, список об'єктів (клас, площа в px та м² за відомого GSD, рамка, центр, впевненість) і час обробки; за `save_dir` зберігає PNG (класи, оверлей, рисунок) і JSON. Нижче — приклади з test: руйнування (xBD, xBD-Ukraine), синтетичні вирви та фортифікації, знесення/нова забудова (FOTBCD). Свою пару — через `USER_PRE` / `USER_POST`.

In [ ]:
# ==============================================================================
# 13. Демо: detect_changes(pre, post) — те, що показуємо на фінальній демонстрації
# ==============================================================================
USER_PRE, USER_POST, USER_GSD = None, None, None     # ← шляхи до власної пари (і GSD у м/px, якщо відомий)
DEMO_DIR = RUN_DIR / 'demo'


def render_result(pre, post, res, title='', gt=None, path=None):
    n = 5 if gt is not None else 4
    fig, ax = plt.subplots(1, n, figsize=(5.2 * n, 5.6))
    ax[0].imshow(pre); ax[0].set_title('До')
    ax[1].imshow(post); ax[1].set_title('Після')
    ax[2].imshow(label_overlay(post, res['class_map'])); ax[2].set_title(f'Виявлені зміни: {len(res["objects"])} об\'єктів')
    for o in res['objects'][:60]:
        x, y, w, h = o['bbox_xywh']
        ax[2].add_patch(plt.Rectangle((x, y), w, h, fill=False, lw=1.2, ec=np.array(CLASS_COLORS[o['cls']]) / 255))
    ax[3].imshow(res['change_prob'], cmap='magma', vmin=0, vmax=1); ax[3].set_title('Ймовірність зміни')
    if gt is not None:
        ax[4].imshow(label_overlay(post, gt)); ax[4].set_title('GT (для порівняння)')
    for a in ax:
        a.axis('off')
    cnt = Counter(o['name'] for o in res['objects'])
    t = res.get('timing', {})
    fig.suptitle(f"{title}\n" + ', '.join(f'{k}: {v}' for k, v in cnt.items()) +
                 (f" | інференс {t['inference_s']:.3f} с (усього {t['total_s']:.3f} с)" if t else ''), fontsize=11)
    fig.legend(handles=legend_handles(), loc='lower center', ncol=4, fontsize=9)
    plt.tight_layout(rect=(0, 0.04, 1, 0.95))
    if path:
        fig.savefig(path, dpi=100, bbox_inches='tight')
    plt.show()


def detect_changes(pre_path, post_path, gsd=None, save_dir=None, show=True, gt_path=None, title=None):
    """Головна функція прототипу: шляхи до вирівняних знімків «до»/«після» → результат + файли."""
    t0 = time.perf_counter()
    pre, post = read_rgb(pre_path), read_rgb(post_path)
    if pre.shape != post.shape:
        raise ValueError(f'Знімки мають бути вирівняні й однакового розміру: {pre.shape} vs {post.shape}')
    t1 = time.perf_counter(); cuda_sync()
    res = detect_changes_arrays(best_model, pre, post, gsd=gsd)
    cuda_sync(); t2 = time.perf_counter()
    res['timing'] = dict(io_s=t1 - t0, inference_s=t2 - t1, total_s=t2 - t0)
    res['summary'] = {CLASS_NAMES[c]: sum(o['cls'] == c for o in res['objects']) for c in (1, 2, 3, 4)}
    if save_dir:
        save_dir = Path(save_dir); save_dir.mkdir(parents=True, exist_ok=True)
        stem = Path(post_path).stem
        color = np.zeros(pre.shape, np.uint8)
        for c, col in CLASS_COLORS.items():
            color[res['class_map'] == c] = col
        write_rgb(save_dir / f'{stem}_classes.png', color)
        write_rgb(save_dir / f'{stem}_overlay.png', label_overlay(post, res['class_map']))
        (save_dir / f'{stem}_result.json').write_text(json.dumps(dict(
            pre=str(pre_path), post=str(post_path), threshold=THRESHOLD, gsd=gsd, timing=res['timing'],
            summary=res['summary'], objects=res['objects']), ensure_ascii=False, indent=1))
    if show:
        gt = read_gray(gt_path) if gt_path else None
        render_result(pre, post, res, title or Path(post_path).name, gt,
                      (Path(save_dir) / f'{Path(post_path).stem}_figure.png') if save_dir else None)
    return res


if best_model is not None:
    demos = []
    t = test_rows.copy()
    for s, key in (('xBD-Global', 'n_destroyed_px'), ('xBD-Ukraine', 'n_destroyed_px')):
        sub = t[t['subset'] == s]
        if len(sub):
            demos.append((sub.loc[sub[key].idxmax()], f'{s}: зруйновані об\'єкти'))
    syn = t[(t['subset'] == 'SYN-ID')]
    if 'n_crater' in syn and len(syn):
        both = syn[(syn['n_crater'].fillna(0) > 0) & (syn['n_fort'].fillna(0) > 0)]
        both = both[both['w'] == 1024] if (both['w'] == 1024).any() else both
        if len(both):
            demos.append((both.sample(1, random_state=SEED).iloc[0], 'Synthetic: нові вирви + свіжі фортифікації'))
    fot = t[t['source'] == 'FOTBCD']
    if len(fot):
        demos.append((fot.loc[fot['chg_px'].idxmax()], 'FOTBCD: знесення / нова забудова'))
    for r, title in demos:
        out = detect_changes(r['pre'], r['post'], gsd=float(r['gsd']), save_dir=DEMO_DIR, gt_path=r['label'], title=title)
        top = pd.DataFrame(out['objects'][:5])
        if len(top):
            display(top[['name', 'area_px', 'area_m2', 'bbox_xywh', 'score']])
    if USER_PRE and USER_POST:
        detect_changes(USER_PRE, USER_POST, gsd=USER_GSD, save_dir=DEMO_DIR, title='Власна пара')
    print('Файли демо (класи, оверлей, JSON з об\'єктами, рисунок):', DEMO_DIR)
else:
    print('Немає навченої моделі.')

## 14. Підсумковий звіт

In [ ]:
# ==============================================================================
# 14.1 Підсумковий звіт: вимоги кейсу → фактичні результати
# ==============================================================================
def _fmt(x):
    return '—' if x is None or (isinstance(x, float) and not np.isfinite(x)) else f'{x:.3f}'


lines = ['# Підсумок: Change Detection (кейс 4.2)', '',
         f'- Модель: BIT-CD (LEVIR init) + голова типу змін; checkpoint epoch {best_ck["epoch"] if best_model is not None else "—"}',
         f'- Поріг (обрано на validation): {THRESHOLD:.2f}; інференс: вікно {TILE}, крок {STRIDE}, TTA={TTA}',
         f'- Дані: FOTBCD ({globals().get("FOTBCD_ACTIVE_MODE", "?")}), xBD Global (спліт: {XBD_SPLIT_MODE}), '
         f'синтетика (ID + Shift), xBD-Ukraine (лише test)', f'- Відбиток даних: {MANIFEST_SHA}', '']
if acc_test is not None:
    lines += ['## Точність (TEST, вимога F1 ≥ 0,75)', '', '| Джерело | Тип даних | F1 (піксельний, бінарний) | Статус |', '|---|---|---|---|']
    kind = {'FOTBCD-Instances': 'реальні', 'FOTBCD-Binary': 'реальні', 'xBD-Global': 'реальні', 'xBD-Ukraine': 'реальні, OOD',
            'SYN-ID': 'синтетика', 'SYN-SHIFT': 'синтетика (зсув генератора)', 'REAL (FOTBCD+xBD)': 'реальні, разом'}
    for r in bin_tbl.itertuples():
        lines.append(f'| {r.subset} | {kind.get(r.subset, "")} | {r.F1:.3f} (P {r.P:.3f}, R {r.R:.3f}) | {_pass(r.F1)} |')
    lines += ['', '| Джерело | Клас | F1 піксельний | F1 об\'єктний | Статус (піксельний) |', '|---|---|---|---|---|']
    ob = obj_tbl.set_index(['subset', 'cls']) if len(obj_tbl) else None
    for r in cls_tbl.itertuples():
        of = ob.loc[(r.subset, r.cls), 'obj_F1'] if ob is not None and (r.subset, r.cls) in ob.index else float('nan')
        lines.append(f'| {r.subset} | {r.name} | {r.F1:.3f} | {_fmt(of)} | {_pass(r.F1)} |')
    if len(sc_tbl):
        lines += ['', '**Shortcut-тест синтетики:** ' + '; '.join(
            f'{r.subset}: хибних спрацювань на «старих» об\'єктах {r.object_false_alarm_rate:.1%} об\'єктів / '
            f'{r.pixel_false_alarm_rate:.1%} пікселів' for r in sc_tbl.itertuples())]
if latency:
    w = latency['робочий режим']
    lines += ['', '## Швидкодія (вимога ≤ 2 с на пару 1024×1024)', '',
              f"- {latency['meta']['device']}: середнє {w['mean']:.3f} с, p95 {w['p95']:.3f} с, максимум {w['max']:.3f} с "
              f"→ **{'PASS' if w['max'] <= TARGET_LATENCY_S else 'FAIL'}** ({latency['meta']['data']})",
              f"- з TTA×4: максимум {latency['з TTA×4 (запас)']['max']:.3f} с; читання PNG окремо ≈ "
              f"{_fmt(latency['meta']['png_read_pair_s'])} с"]
if baseline_tbl is not None:
    b05 = baseline_tbl[baseline_tbl['threshold'] == 0.5]
    lines += ['', '## Довідково: zero-shot LEVIR BIT-CD (validation, поріг 0,5)', '',
              ', '.join(f'{r.subset}: F1 {r.F1:.3f}' for r in b05.itertuples())]
report_md = '\n'.join(lines)
(RUN_DIR / 'REPORT.md').write_text(report_md)
from IPython.display import Markdown
display(Markdown(report_md))
print('Звіт збережено:', RUN_DIR / 'REPORT.md', '| усі рисунки:', RUN_DIR / 'figures')

## 14.2 Обраний підхід і обмеження (для пункту 7 демонстрації)

**Підхід.** BIT-CD — легка (≈3,5 M параметрів) і швидка модель зміни; стартує з ваг LEVIR-CD (зміни забудови), донавчається на трьох джерелах одночасно, а нова спрямована голова визначає *тип* зміни. Вікно 256 + зважене злиття дає однакову поведінку на кадрах 512, 1024 та довільних.

**Обмеження — говоримо прямо:**
1. **Вирви та фортифікації навчені лише на синтетиці.** Високий F1 на SYN-ID означає, що модель розв'язує задачу *в межах генератора*; SYN-Shift показує перенос на невидані палітри/освітлення, shortcut-тест — що модель реагує на *зміну*, а не на сам вигляд об'єкта. Але це **не** доказ якості на реальних вирвах/траншеях — для цього потрібна хоча б невелика реальна розмічена вибірка (найкращий наступний крок: 50–100 реальних пар з українських полів для тесту).
2. **«Зруйнований об'єкт» = будівлі.** xBD розмічає лише будівлі (major-damage + destroyed); FOTBCD `DEMOLISHED` — планове знесення, що візуально відрізняється від воєнних руйнувань. Руйнування мостів, техніки, інфраструктури не розмічені. xBD-Ukraine — єдиний тест у цільовому домені і він малий (десятки пар).
3. **Домени та роздільність.** FOTBCD — аерофото 0,2 м/px (Франція), xBD — супутник ≈0,3–0,6 м/px. GSD цільових знімків не задано; на сильно іншій роздільності якість може падати.
4. **Розмітка не ідеальна:** xBD `minor-damage` ігнорується (неоднозначний клас); у xBD не розмічені інші зміни (вода, вигорілі ділянки) — частина «хибних» спрацювань може бути реальними змінами.
5. **F1 залежить від визначення.** Показуємо піксельний бінарний, піксельний покласовий та об'єктний F1 окремо — вони відповідають різним питанням; у кейсі спосіб розмітки змін не визначено, тому критерій варто погодити з ментором.
6. **Швидкодія** виміряна на Colab T4 від масивів у пам'яті до списку об'єктів; читання файлів залежить від диска і показане окремо. На CPU час буде значно більшим.
7. **Обсяг навчання** обмежений квотою Colab; збільшення `EPOCHS`/даних FOTBCD-Binary, як правило, покращує результат.

In [ ]:
DEMO_PAIRS = DRIVE_ROOT / 'demo_pairs'; DEMO_PAIRS.mkdir(parents=True, exist_ok=True)
t = manifest[manifest['split'] == 'test']
pick = pd.concat([t[t.subset == 'xBD-Global'].nlargest(3, 'n_destroyed_px'),
                  t[t.subset == 'xBD-Ukraine'].nlargest(2, 'n_destroyed_px'),
                  t[(t.subset == 'SYN-ID') & (t.n_crater > 0) & (t.n_fort > 0)].head(3),
                  t[t.subset == 'FOTBCD-Instances'].nlargest(2, 'chg_px')])
for r in pick.itertuples():
    for k in ('pre', 'post', 'label'):
        shutil.copy2(getattr(r, k), DEMO_PAIRS / f'{r.uid}_{k}.png')
print(len(pick), 'пар →', DEMO_PAIRS)